# Main-text figures: analysis and visualization

This notebook contains only the analytical models and visualization code required to reproduce the **four main-text figures**. Original notebook figures **2–5** correspond to manuscript figures **1–4**. The original notebook's Figure 1, supplementary analyses, exploratory designs, and standalone SI tests are excluded.

**Scope.** Starts from analysis-ready BigQuery tables and intermediate author-level CSV inputs, not raw PubMed Central or OpenAlex ingestion. The original analytical code is retained where used in the final figure, with outputs and historical execution state removed. Running the notebook requires the indicated BigQuery project permissions, source tables, and CSV inputs; some upstream tables may have been deleted after the original analysis.

**Execution.** Run cells in order. The figures' modeling cells produce local CSV results consumed by their plotting cells. `google-cloud-bigquery`, `pandas`, `numpy`, `scipy`, `statsmodels`, `matplotlib`, and `duckdb` are needed. No previously computed output is presented as a new result.

In [ ]:
import os
import numpy as np
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project="aiwriting-488923")


## Manuscript Figure 1 — Team-size redistribution

Final exact team-size response and empirically defined team-size regimes, estimated using the same within-author FWL sample. The exact-size estimator was originally called a bin-design analysis but is required to reproduce panel a of the final figure.

### Exact team-size response (panel a)

In [ ]:
# =============================================================================
# FIGURE 2 — EXACT TEAM-SIZE RESPONSE FUNCTION
#
# DISCOVERY / BIN-DESIGN ANALYSIS
#
# Categories:
#     1, 2, 3, ..., 50, 51+
#
# Specification:
#     author FE
#     accepted-month FE
#     subfield FE
#
# Exposure:
#     ai_score
#
# Effect:
#     percentage-point change in P(team size = k)
#     per 1-SD increase in FWL-residualized AI
#
# Purpose:
#     Discover the full distributional response BEFORE deciding the
#     publication bins for Figure 2.
#
# ANALYSIS ONLY — no visualization.
# =============================================================================

import numpy as np
import pandas as pd
import statsmodels.api as sm


# =============================================================================
# 1. LOAD FIGURE 1 FWL SAMPLE
# =============================================================================

df = client.query("""
    SELECT
        openalex_id,
        author_id,
        accepted_month_fe,
        subfield_id,
        ai_score,
        authors_count
    FROM `aiwriting-488923.redo.fig1_fwl`
""").to_dataframe()


# =============================================================================
# 2. CLEAN
# =============================================================================

df["ai_score"] = pd.to_numeric(
    df["ai_score"],
    errors="coerce"
)

df["authors_count"] = pd.to_numeric(
    df["authors_count"],
    errors="coerce"
)


df = df[
    df["openalex_id"].notna()
    & df["author_id"].notna()
    & df["accepted_month_fe"].notna()
    & df["subfield_id"].notna()
    & df["ai_score"].notna()
    & df["authors_count"].notna()
].copy()


df = df[
    np.isfinite(df["ai_score"])
    & np.isfinite(df["authors_count"])
    & (df["authors_count"] >= 1)
].copy()


print(
    f"Observations: {len(df):,}"
)

print(
    f"Authors: {df['author_id'].nunique():,}"
)


# =============================================================================
# 3. INTEGER-CODE FIXED EFFECTS
# =============================================================================

author_code, _ = pd.factorize(
    df["author_id"],
    sort=False
)

month_code, _ = pd.factorize(
    df["accepted_month_fe"],
    sort=False
)

subfield_code, _ = pd.factorize(
    df["subfield_id"],
    sort=False
)


# =============================================================================
# 4. FAST MATRIX DEMEANING
# =============================================================================

def group_demean_matrix(
    X,
    groups,
    n_groups
):

    X = np.asarray(
        X,
        dtype=np.float64
    )

    out = np.empty_like(X)

    counts = np.bincount(
        groups,
        minlength=n_groups
    ).astype(np.float64)

    counts[counts == 0] = 1.0

    for j in range(X.shape[1]):

        sums = np.bincount(
            groups,
            weights=X[:, j],
            minlength=n_groups
        )

        means = sums / counts

        out[:, j] = (
            X[:, j]
            - means[groups]
        )

    return out


# =============================================================================
# 5. ITERATIVE FWL
# =============================================================================

def fwl_residualize(
    X,
    group_arrays,
    group_sizes,
    tol=1e-8,
    max_iter=100
):

    R = np.asarray(
        X,
        dtype=np.float64
    ).copy()

    for iteration in range(
        1,
        max_iter + 1
    ):

        old = R.copy()

        for groups, n_groups in zip(
            group_arrays,
            group_sizes
        ):

            R = group_demean_matrix(
                R,
                groups,
                n_groups
            )

        change = np.max(
            np.abs(
                R - old
            )
        )

        if change < tol:

            return (
                R,
                iteration,
                change
            )

    return (
        R,
        max_iter,
        change
    )


# =============================================================================
# 6. EXACT TEAM-SIZE CATEGORIES
#
# 1 ... 50
# 51+
# =============================================================================

exact_sizes = list(
    range(1, 51)
)

category_labels = (
    [str(k) for k in exact_sizes]
    + ["51+"]
)


# Exact-size indicator matrix
team_matrix = np.column_stack(

    [
        (
            df["authors_count"].to_numpy()
            == k
        ).astype(np.float64)

        for k in exact_sizes
    ]

    +

    [
        (
            df["authors_count"].to_numpy()
            >= 51
        ).astype(np.float64)
    ]

)


# =============================================================================
# 7. BASELINE SHARES AND COUNTS
# =============================================================================

authors_array = (
    df["authors_count"]
    .to_numpy()
)


base_counts = np.array(

    [
        np.sum(
            authors_array == k
        )

        for k in exact_sizes
    ]

    +

    [
        np.sum(
            authors_array >= 51
        )
    ],

    dtype=np.int64
)


base_shares = (
    base_counts
    / len(df)
)


# =============================================================================
# 8. BUILD ONE MATRIX
#
# Column 0:
#     AI
#
# Columns 1-51:
#     exact team-size indicators
# =============================================================================

X = np.column_stack(
    [
        df["ai_score"].to_numpy(
            dtype=np.float64
        ),
        team_matrix
    ]
)


# =============================================================================
# 9. RESIDUALIZE EVERYTHING AT ONCE
# =============================================================================

R, iterations, final_change = fwl_residualize(

    X,

    group_arrays=[
        author_code,
        month_code,
        subfield_code
    ],

    group_sizes=[
        author_code.max() + 1,
        month_code.max() + 1,
        subfield_code.max() + 1
    ],

    tol=1e-8,

    max_iter=100
)


ai_fwl = R[:, 0]

team_fwl = R[:, 1:]


ai_sd = np.std(
    ai_fwl,
    ddof=0
)


print(
    f"FWL iterations: {iterations}"
)

print(
    f"Final convergence change: {final_change:.3e}"
)

print(
    f"Residualized AI SD: {ai_sd:.6f}"
)


# =============================================================================
# 10. CLUSTERED ESTIMATION
# =============================================================================

author_ids = (
    df["author_id"]
    .astype(str)
    .to_numpy()
)


def estimate_category(
    y,
    x,
    authors,
    ai_sd
):

    valid = (
        np.isfinite(y)
        &
        np.isfinite(x)
    )

    Y = y[valid]

    Xreg = sm.add_constant(
        x[valid],
        has_constant="add"
    )

    model = sm.OLS(
        Y,
        Xreg
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                authors[valid]
        }
    )

    beta = model.params[1]

    se = model.bse[1]

    p = model.pvalues[1]


    # ---------------------------------------------------------
    # Convert to probability change per 1-SD AI
    # ---------------------------------------------------------

    effect = (
        beta
        * ai_sd
    )

    effect_se = (
        se
        * ai_sd
    )


    return {

        "beta_raw": beta,

        "se_raw": se,

        "p": p,

        "effect_probability_1sd":
            effect,

        "se_probability_1sd":
            effect_se,

        "ci_low_probability":
            effect
            - 1.96 * effect_se,

        "ci_high_probability":
            effect
            + 1.96 * effect_se

    }


# =============================================================================
# 11. ESTIMATE ALL EXACT TEAM SIZES
# =============================================================================

rows = []


for j, label in enumerate(
    category_labels
):

    r = estimate_category(

        y=team_fwl[:, j],

        x=ai_fwl,

        authors=author_ids,

        ai_sd=ai_sd
    )


    baseline_share = (
        base_shares[j]
    )


    # ---------------------------------------------------------
    # Probability effect in percentage points
    # ---------------------------------------------------------

    effect_pp = (
        100
        * r["effect_probability_1sd"]
    )

    se_pp = (
        100
        * r["se_probability_1sd"]
    )

    ci_low_pp = (
        100
        * r["ci_low_probability"]
    )

    ci_high_pp = (
        100
        * r["ci_high_probability"]
    )


    # ---------------------------------------------------------
    # Relative change versus baseline prevalence
    #
    # Example:
    #     baseline = 10%
    #     effect   = +0.2 pp
    #
    #     relative change = +2%
    # ---------------------------------------------------------

    if baseline_share > 0:

        relative_change_pct = (
            100
            * r["effect_probability_1sd"]
            / baseline_share
        )

    else:

        relative_change_pct = np.nan


    rows.append({

        "team_category":
            label,

        "category_order":
            j + 1,

        "baseline_n":
            int(base_counts[j]),

        "baseline_share":
            baseline_share,

        "baseline_pct":
            100 * baseline_share,

        "beta_raw":
            r["beta_raw"],

        "se_raw":
            r["se_raw"],

        "p":
            r["p"],

        "effect_pp_1sd":
            effect_pp,

        "se_pp_1sd":
            se_pp,

        "ci_low_pp":
            ci_low_pp,

        "ci_high_pp":
            ci_high_pp,

        "relative_change_pct":
            relative_change_pct
    })


exact_results = pd.DataFrame(
    rows
)


# =============================================================================
# 12. COMPOSITIONAL CHECK
# =============================================================================

print(
    "\nCompositional check"
)

print(
    "-------------------"
)

print(
    "Sum of probability effects:",
    exact_results[
        "effect_pp_1sd"
    ].sum()
)


# =============================================================================
# 13. DISPLAY EXACT RESULTS
# =============================================================================

display(
    exact_results
)


# =============================================================================
# 14. SAVE
# =============================================================================

exact_results.to_csv(
    "fig2_exact_team_size_fwl_results.csv",
    index=False
)


print(
    "\nSaved:"
)

print(
    "fig2_exact_team_size_fwl_results.csv"
)

### Five team-size regimes (panels b–c)

In [ ]:
# =============================================================================
# FIGURE 2 — FINAL TEAM-SIZE REGIME ANALYSIS
#
# Candidate regimes:
#
#     1-5
#     6-7
#     8-12
#     13-19
#     20+
#
# Primary specification:
#
#     Author FE
#     + accepted-month FE
#     + subfield FE
#
# Exposure:
#     AI-associated writing intensity
#
# Effect:
#     Change in probability of belonging to each team-size regime
#     per 1-SD increase in FWL-residualized AI intensity.
#
# SE:
#     Clustered by author.
#
# ANALYSIS ONLY.
# No visualization.
# =============================================================================


import numpy as np
import pandas as pd
import statsmodels.api as sm


# =============================================================================
# 1. LOAD SAME FIGURE 1 IDENTIFICATION SAMPLE
# =============================================================================

df = client.query("""
    SELECT
        openalex_id,
        author_id,
        accepted_month_fe,
        subfield_id,
        ai_score,
        authors_count
    FROM `aiwriting-488923.redo.fig1_fwl`
""").to_dataframe()


# =============================================================================
# 2. CLEAN
# =============================================================================

df["ai_score"] = pd.to_numeric(
    df["ai_score"],
    errors="coerce"
)

df["authors_count"] = pd.to_numeric(
    df["authors_count"],
    errors="coerce"
)


df = df[
    df["openalex_id"].notna()
    & df["author_id"].notna()
    & df["accepted_month_fe"].notna()
    & df["subfield_id"].notna()
    & df["ai_score"].notna()
    & df["authors_count"].notna()
].copy()


df = df[
    np.isfinite(df["ai_score"])
    & np.isfinite(df["authors_count"])
    & (df["authors_count"] >= 1)
].copy()


print(
    f"Observations: {len(df):,}"
)

print(
    f"Authors: {df['author_id'].nunique():,}"
)


# =============================================================================
# 3. DEFINE FINAL CANDIDATE REGIMES
# =============================================================================

n_authors = (
    df["authors_count"]
    .to_numpy(dtype=float)
)


regime_names = [
    "1-5",
    "6-7",
    "8-12",
    "13-19",
    "20+"
]


regime_matrix = np.column_stack([

    (
        (n_authors >= 1)
        & (n_authors <= 5)
    ).astype(float),

    (
        (n_authors >= 6)
        & (n_authors <= 7)
    ).astype(float),

    (
        (n_authors >= 8)
        & (n_authors <= 12)
    ).astype(float),

    (
        (n_authors >= 13)
        & (n_authors <= 19)
    ).astype(float),

    (
        n_authors >= 20
    ).astype(float)

])


# =============================================================================
# 4. CHECK THAT REGIMES EXHAUST THE SAMPLE
# =============================================================================

row_sums = regime_matrix.sum(
    axis=1
)


print(
    "\nRegime assignment QC"
)

print(
    "--------------------"
)

print(
    "Rows assigned to exactly one regime:",
    int(
        np.sum(
            row_sums == 1
        )
    )
)

print(
    "Rows assigned to zero/multiple regimes:",
    int(
        np.sum(
            row_sums != 1
        )
    )
)


# =============================================================================
# 5. BASELINE REGIME PREVALENCE
# =============================================================================

baseline_n = regime_matrix.sum(
    axis=0
)

baseline_share = regime_matrix.mean(
    axis=0
)


baseline = pd.DataFrame({

    "team_regime":
        regime_names,

    "baseline_n":
        baseline_n.astype(int),

    "baseline_pct":
        100 * baseline_share

})


print(
    "\nBaseline distribution"
)

display(
    baseline
)


# =============================================================================
# 6. INTEGER-CODE FIXED EFFECTS
# =============================================================================

author_code, _ = pd.factorize(
    df["author_id"],
    sort=False
)

month_code, _ = pd.factorize(
    df["accepted_month_fe"],
    sort=False
)

subfield_code, _ = pd.factorize(
    df["subfield_id"],
    sort=False
)


# =============================================================================
# 7. FAST GROUP-DEMEAN FUNCTION
# =============================================================================

def group_demean_matrix(
    X,
    groups,
    n_groups
):

    X = np.asarray(
        X,
        dtype=np.float64
    )

    out = np.empty_like(
        X
    )


    counts = np.bincount(
        groups,
        minlength=n_groups
    ).astype(np.float64)


    counts[
        counts == 0
    ] = 1.0


    for j in range(
        X.shape[1]
    ):

        sums = np.bincount(
            groups,
            weights=X[:, j],
            minlength=n_groups
        )


        means = (
            sums
            / counts
        )


        out[:, j] = (
            X[:, j]
            - means[groups]
        )


    return out


# =============================================================================
# 8. ITERATIVE FWL
# =============================================================================

def fwl_residualize(
    X,
    group_arrays,
    group_sizes,
    tol=1e-8,
    max_iter=100
):

    R = np.asarray(
        X,
        dtype=np.float64
    ).copy()


    for iteration in range(
        1,
        max_iter + 1
    ):

        old = R.copy()


        for groups, n_groups in zip(
            group_arrays,
            group_sizes
        ):

            R = group_demean_matrix(
                R,
                groups,
                n_groups
            )


        change = np.max(
            np.abs(
                R - old
            )
        )


        if change < tol:

            return (
                R,
                iteration,
                change
            )


    return (
        R,
        max_iter,
        change
    )


# =============================================================================
# 9. AI + ALL FIVE REGIME OUTCOMES
# =============================================================================

X = np.column_stack([

    df["ai_score"]
    .to_numpy(
        dtype=np.float64
    ),

    regime_matrix

])


# =============================================================================
# 10. FWL RESIDUALIZATION
# =============================================================================

R, iterations, final_change = fwl_residualize(

    X,

    group_arrays=[
        author_code,
        month_code,
        subfield_code
    ],

    group_sizes=[
        author_code.max() + 1,
        month_code.max() + 1,
        subfield_code.max() + 1
    ],

    tol=1e-8,

    max_iter=100
)


ai_fwl = R[:, 0]

regime_fwl = R[:, 1:]


# =============================================================================
# 11. AI SCALE
# =============================================================================

ai_sd = np.std(
    ai_fwl,
    ddof=0
)


print(
    "\nFWL diagnostics"
)

print(
    "---------------"
)

print(
    f"Iterations: {iterations}"
)

print(
    f"Final change: {final_change:.3e}"
)

print(
    f"Residualized AI SD: {ai_sd:.6f}"
)


# =============================================================================
# 12. CLUSTERED REGRESSION
# =============================================================================

author_ids = (
    df["author_id"]
    .astype(str)
    .to_numpy()
)


results = []


for j, regime in enumerate(
    regime_names
):

    y = regime_fwl[:, j]


    valid = (
        np.isfinite(y)
        &
        np.isfinite(ai_fwl)
    )


    Y = y[valid]


    Xreg = sm.add_constant(
        ai_fwl[valid],
        has_constant="add"
    )


    model = sm.OLS(
        Y,
        Xreg
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                author_ids[valid]
        }
    )


    beta = model.params[1]

    se = model.bse[1]


    # -------------------------------------------------------------------------
    # Probability change per one SD of residualized AI
    # -------------------------------------------------------------------------

    effect_probability = (
        beta
        * ai_sd
    )


    effect_probability_se = (
        se
        * ai_sd
    )


    # -------------------------------------------------------------------------
    # Convert to percentage points
    # -------------------------------------------------------------------------

    effect_pp = (
        100
        * effect_probability
    )


    se_pp = (
        100
        * effect_probability_se
    )


    ci_low = (
        effect_pp
        - 1.96 * se_pp
    )


    ci_high = (
        effect_pp
        + 1.96 * se_pp
    )


    # -------------------------------------------------------------------------
    # Relative change compared with baseline prevalence
    # -------------------------------------------------------------------------

    relative_change = (

        100
        * effect_probability
        / baseline_share[j]

    )


    results.append({

        "team_regime":
            regime,

        "baseline_n":
            int(
                baseline_n[j]
            ),

        "baseline_pct":
            100
            * baseline_share[j],

        "beta_raw":
            beta,

        "se_raw":
            se,

        "effect_pp_1sd":
            effect_pp,

        "se_pp_1sd":
            se_pp,

        "ci_low_pp":
            ci_low,

        "ci_high_pp":
            ci_high,

        "relative_change_pct":
            relative_change,

        "p":
            model.pvalues[1],

        "n_obs":
            int(
                valid.sum()
            ),

        "n_authors":
            pd.Series(
                author_ids[valid]
            ).nunique()

    })


# =============================================================================
# 13. RESULTS TABLE
# =============================================================================

regime_results = pd.DataFrame(
    results
)


display(
    regime_results
)


# =============================================================================
# 14. COMPOSITIONAL CHECK
#
# Because the five regimes are mutually exclusive and exhaustive,
# their probability changes must sum to approximately zero.
# =============================================================================

print(
    "\nCompositional check"
)

print(
    "-------------------"
)

print(
    "Sum of percentage-point effects:",
    regime_results[
        "effect_pp_1sd"
    ].sum()
)


# =============================================================================
# 15. POSITIVE / NEGATIVE MASS DECOMPOSITION
#
# This is descriptive and helps interpret where probability mass moves.
# =============================================================================

positive_mass = (
    regime_results.loc[
        regime_results[
            "effect_pp_1sd"
        ] > 0,
        "effect_pp_1sd"
    ]
    .sum()
)


negative_mass = (
    regime_results.loc[
        regime_results[
            "effect_pp_1sd"
        ] < 0,
        "effect_pp_1sd"
    ]
    .sum()
)


print(
    "\nProbability-mass redistribution"
)

print(
    "-------------------------------"
)

print(
    f"Positive mass: "
    f"{positive_mass:.3f} pp"
)

print(
    f"Negative mass: "
    f"{negative_mass:.3f} pp"
)


# =============================================================================
# 16. SAVE
# =============================================================================

regime_results.to_csv(
    "fig2_final_regimes_fwl_results.csv",
    index=False
)


print(
    "\nSaved:"
)

print(
    "fig2_final_regimes_fwl_results.csv"
)

### Main figure 1 visualization

In [ ]:
# =============================================================================
# FIGURE 2 — TEAM-SIZE REDISTRIBUTION
#
# Publication-style visualization using the same visual grammar as Figure 1.
#
# PANEL A
#     Exact team-size response function, 1-30 authors.
#
# PANEL B
#     Probability redistribution across the five empirical regimes:
#         1-5
#         6-7
#         8-12
#         13-19
#         20+
#
# PANEL C
#     Relative change in prevalence by regime.
#
# INPUTS
#     fig2_exact_team_size_fwl_results.csv
#     fig2_final_regimes_fwl_results.csv
#
# NO ANALYSIS / NO FWL ESTIMATION
# =============================================================================


# =============================================================================
# 0. IMPORTS
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.ticker import MultipleLocator


# =============================================================================
# 1. PUBLICATION PARAMETERS
# =============================================================================

plt.rcParams.update({

    "font.family": "DejaVu Sans",

    "font.size": 8.0,

    "axes.titlesize": 8.8,

    "axes.labelsize": 8.0,

    "xtick.labelsize": 7.2,

    "ytick.labelsize": 7.2,

    "legend.fontsize": 7.0,

    "axes.linewidth": 0.6,

    "xtick.major.width": 0.5,

    "ytick.major.width": 0.5,

    "xtick.major.size": 2.5,

    "ytick.major.size": 2.5,

    "savefig.dpi": 600,

    "savefig.bbox": "tight",

    "savefig.pad_inches": 0.04,
})


# =============================================================================
# 2. COLORS
# =============================================================================

AI = "#245A85"
POSITIVE = "#245A85"
NEGATIVE = "#6B6B6B"
NEUTRAL = "#B0B0B0"
REFERENCE = "#9A9A9A"
GRID = "#D0D0D0"
TEXT = "#333333"


# =============================================================================
# 3. LOAD RESULTS
# =============================================================================

exact = pd.read_csv(
    "fig2_exact_team_size_fwl_results.csv"
)

regime = pd.read_csv(
    "fig2_final_regimes_fwl_results.csv"
)


# =============================================================================
# 4. CLEAN / SORT EXACT-SIZE RESULTS
# =============================================================================

def parse_team_size(x):

    x = str(x).strip()

    if x == "51+":
        return 51

    return int(x)


exact["team_size"] = (
    exact["team_category"]
    .apply(parse_team_size)
)


exact = (
    exact
    .sort_values("team_size")
    .reset_index(drop=True)
)


# =============================================================================
# 5. MAIN EXACT-SIZE RANGE
#
# Show 1-30 in the main figure.
# The 31-50 and 51+ estimates remain available for SI.
# =============================================================================

exact_main = exact[
    exact["team_size"] <= 30
].copy()


# =============================================================================
# 6. FIGURE CANVAS
# =============================================================================

fig = plt.figure(
    figsize=(7.25, 6.15)
)


gs = fig.add_gridspec(
    2,
    2,

    left=0.085,

    right=0.985,

    bottom=0.10,

    top=0.865,

    wspace=0.29,

    hspace=0.40
)


ax_a = fig.add_subplot(
    gs[0, :]
)


ax_b = fig.add_subplot(
    gs[1, 0]
)


ax_c = fig.add_subplot(
    gs[1, 1]
)


# =============================================================================
# 7. OVERALL TITLE
# =============================================================================

fig.suptitle(
    "AI intensity redistributes scientific work toward smaller teams",
    fontsize=12,
    fontweight="bold",
    x=0.085,
    y=0.965,
    ha="left"
)


# =============================================================================
# 8. COMMON AXIS CLEANUP
# =============================================================================

def clean_axis(ax):

    ax.spines["top"].set_visible(False)

    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_color("#555555")

    ax.spines["bottom"].set_color("#555555")

    ax.tick_params(
        direction="out",
        color="#555555"
    )

    ax.grid(
        axis="y",
        color=GRID,
        linewidth=0.35,
        alpha=0.45
    )


for ax in [
    ax_a,
    ax_b,
    ax_c
]:
    clean_axis(ax)


# =============================================================================
# PANEL A
# EXACT TEAM-SIZE RESPONSE FUNCTION
# =============================================================================
#
# Outcome:
#     Change in probability of exact team size
#
# Exposure:
#     1 SD increase in FWL-residualized AI intensity
#
# The confidence intervals are the clustered intervals from the original
# exact-size FWL estimation.
#
# =============================================================================

# =============================================================================
# PANEL A — EXACT TEAM-SIZE RESPONSE FUNCTION
#
# Shows exact team sizes 1–30.
#
# Y-axis:
#   Change in probability (percentage points) per 1-SD increase in
#   within-author AI intensity.
#
# Points:
#   FWL estimates
#
# Error bars:
#   95% CIs
#
# Background bands:
#   Empirically derived regimes
#
# =============================================================================


# -----------------------------------------------------------------------------
# Prepare data
# -----------------------------------------------------------------------------

exact_main = (
    exact[
        exact["team_size"] <= 30
    ]
    .sort_values("team_size")
    .copy()
)


x = exact_main[
    "team_size"
].to_numpy(
    dtype=float
)


y = exact_main[
    "effect_pp_1sd"
].to_numpy(
    dtype=float
)


low = exact_main[
    "ci_low_pp"
].to_numpy(
    dtype=float
)


high = exact_main[
    "ci_high_pp"
].to_numpy(
    dtype=float
)


# -----------------------------------------------------------------------------
# Plot confidence intervals manually
# -----------------------------------------------------------------------------

ax_a.errorbar(

    x,

    y,

    yerr=[
        y - low,
        high - y
    ],

    fmt="o",

    color=AI,

    ecolor=AI,

    elinewidth=0.9,

    capsize=1.8,

    capthick=0.8,

    markersize=3.4,

    markeredgewidth=0,

    zorder=4
)


# -----------------------------------------------------------------------------
# Zero reference
# -----------------------------------------------------------------------------

ax_a.axhline(
    0,
    color=REFERENCE,
    linewidth=0.8,
    linestyle=":",
    zorder=1
)


# -----------------------------------------------------------------------------
# Set limits BEFORE placing regime labels
# -----------------------------------------------------------------------------

ax_a.set_xlim(
    0.5,
    30.5
)


y_min = min(
    low.min(),
    -0.75
)

y_max = max(
    high.max(),
    0.65
)


ax_a.set_ylim(
    y_min,
    y_max
)


# -----------------------------------------------------------------------------
# Background regime bands
#
# Very subtle: estimates remain visually dominant.
# -----------------------------------------------------------------------------

ax_a.axvspan(
    0.5,
    5.5,
    color=AI,
    alpha=0.035,
    zorder=0
)


ax_a.axvspan(
    5.5,
    7.5,
    color=NEUTRAL,
    alpha=0.035,
    zorder=0
)


ax_a.axvspan(
    7.5,
    12.5,
    color=NEGATIVE,
    alpha=0.025,
    zorder=0
)


ax_a.axvspan(
    12.5,
    19.5,
    color=NEGATIVE,
    alpha=0.035,
    zorder=0
)


ax_a.axvspan(
    19.5,
    30.5,
    color=NEGATIVE,
    alpha=0.045,
    zorder=0
)


# -----------------------------------------------------------------------------
# Regime labels
# -----------------------------------------------------------------------------

regime_label_y = (
    y_max
    - 0.055 * (y_max - y_min)
)


ax_a.text(
    3,
    regime_label_y,
    "1–5",
    fontsize=6.5,
    color=TEXT,
    ha="center",
    va="top"
)


ax_a.text(
    6.5,
    regime_label_y,
    "6–7",
    fontsize=6.5,
    color=TEXT,
    ha="center",
    va="top"
)


ax_a.text(
    10,
    regime_label_y,
    "8–12",
    fontsize=6.5,
    color=TEXT,
    ha="center",
    va="top"
)


ax_a.text(
    16,
    regime_label_y,
    "13–19",
    fontsize=6.5,
    color=TEXT,
    ha="center",
    va="top"
)


ax_a.text(
    25,
    regime_label_y,
    "20+",
    fontsize=6.5,
    color=TEXT,
    ha="center",
    va="top"
)


# -----------------------------------------------------------------------------
# Labels
# -----------------------------------------------------------------------------

ax_a.set_title(
    "Team-size response to AI intensity",
    loc="left",
    pad=8
)


ax_a.set_xlabel(
    "Number of authors",
    labelpad=5
)


ax_a.set_ylabel(
    "Change in probability (percentage points)\n"
    "per 1 SD increase in AI intensity",
    labelpad=6
)


# -----------------------------------------------------------------------------
# X-axis ticks
#
# Every 2 authors keeps the panel readable.
# -----------------------------------------------------------------------------

ax_a.set_xticks(
    np.arange(
        2,
        31,
        2
    )
)


# -----------------------------------------------------------------------------
# Annotation
# -----------------------------------------------------------------------------

ax_a.text(
    0.99,
    0.045,
    "Points: estimates; whiskers: 95% CI",
    transform=ax_a.transAxes,
    fontsize=6.3,
    color=SECONDARY,
    ha="right",
    va="bottom"
)


# -----------------------------------------------------------------------------
# Grid
# -----------------------------------------------------------------------------

ax_a.grid(
    axis="y",
    color=GRID,
    linewidth=0.35,
    alpha=0.45
)

ax_a.grid(
    axis="x",
    visible=False
)


# -----------------------------------------------------------------------------
# Clean tick spacing
# -----------------------------------------------------------------------------

ax_a.tick_params(
    axis="x",
    pad=3
)

ax_a.tick_params(
    axis="y",
    pad=3
)

# =============================================================================
# PANEL B
# ABSOLUTE PROBABILITY REDISTRIBUTION
# =============================================================================

regime_order = [
    "1-5",
    "6-7",
    "8-12",
    "13-19",
    "20+"
]


regime = (
    regime
    .set_index("team_regime")
    .loc[regime_order]
    .reset_index()
)


rb = regime[
    "effect_pp_1sd"
].to_numpy(dtype=float)

rlo = regime[
    "ci_low_pp"
].to_numpy(dtype=float)

rhi = regime[
    "ci_high_pp"
].to_numpy(dtype=float)


yb = np.arange(
    len(regime)
)[::-1]


# -------------------------------------------------------------------------
# Confidence intervals + point estimates
# -------------------------------------------------------------------------

for i in range(
    len(regime)
):

    value = rb[i]

    low = rlo[i]

    high = rhi[i]


    if value > 0:

        c = POSITIVE

    elif value < 0:

        c = NEGATIVE

    else:

        c = NEUTRAL


    # CI
    ax_b.hlines(
        y=yb[i],
        xmin=low,
        xmax=high,
        color=c,
        linewidth=1.5
    )


    # CI caps
    ax_b.vlines(
        x=low,
        ymin=yb[i] - 0.055,
        ymax=yb[i] + 0.055,
        color=c,
        linewidth=1.0
    )

    ax_b.vlines(
        x=high,
        ymin=yb[i] - 0.055,
        ymax=yb[i] + 0.055,
        color=c,
        linewidth=1.0
    )


    # Point estimate
    ax_b.plot(
        value,
        yb[i],
        marker="o",
        markersize=5.3,
        color=c,
        markeredgewidth=0
    )


# -------------------------------------------------------------------------
# Zero line
# -------------------------------------------------------------------------

ax_b.axvline(
    0,
    color=REFERENCE,
    linewidth=0.8,
    linestyle=":"
)


# -------------------------------------------------------------------------
# Grid
# -------------------------------------------------------------------------

ax_b.grid(
    axis="x",
    color=GRID,
    linewidth=0.35,
    alpha=0.45
)

ax_b.grid(
    axis="y",
    visible=False
)


# -------------------------------------------------------------------------
# Y labels
# -------------------------------------------------------------------------

ax_b.set_yticks(
    yb
)

ax_b.set_yticklabels(
    regime_order
)


# -------------------------------------------------------------------------
# Titles and labels
# -------------------------------------------------------------------------

ax_b.set_title(
    "Probability mass shifts toward 1–5-person teams",
    loc="left",
    pad=8
)

ax_b.set_xlabel(
    "Change in probability (percentage points)\n"
    "per 1 SD increase in AI intensity",
    labelpad=6
)


# -------------------------------------------------------------------------
# X-axis
# -------------------------------------------------------------------------

ax_b.set_xlim(
    -0.75,
    1.25
)

ax_b.xaxis.set_major_locator(
    MultipleLocator(0.5)
)


# -------------------------------------------------------------------------
# Direct estimates
# -------------------------------------------------------------------------

for yy, value in zip(
    yb,
    rb
):

    offset = (
        0.1
        if value >= 0
        else 0.4
    )

    ha = (
        "left"
        if value >= 0
        else "right"
    )

    text_color = (
        POSITIVE
        if value > 0
        else NEGATIVE
    )


    ax_b.text(
        value + offset,
        yy,
        f"{value:+.2f}",
        fontsize=8,
        fontweight="bold",
        color=text_color,
        ha=ha,
        va="center"
    )


# -------------------------------------------------------------------------
# Baseline prevalence
# -------------------------------------------------------------------------

baseline_text = (
    "Baseline prevalence: "
    +
    "  ".join(
        [
            f"{r}: {p:.1f}%"
            for r, p in zip(
                regime_order,
                regime["baseline_pct"]
            )
        ]
    )
)


ax_b.text(
    0.02,
    -0.28,
    baseline_text,
    transform=ax_b.transAxes,
    fontsize=6.2,
    color=SECONDARY,
    ha="left",
    va="top"
)


# =============================================================================
# PANEL C
# RELATIVE CHANGE IN REGIME PREVALENCE
# =============================================================================

relative = regime[
    "relative_change_pct"
].to_numpy()


yc = np.arange(
    len(regime)
)[::-1]


# -------------------------------------------------------------------------
# Bars
#
# Horizontal bars are used here because the regime labels are categorical.
# -------------------------------------------------------------------------

for i in range(
    len(regime)
):

    value = relative[i]


    if value > 0:
        c = POSITIVE

    elif value < 0:
        c = NEGATIVE

    else:
        c = NEUTRAL


    ax_c.barh(
        yc[i],
        value,
        height=0.54,
        color=c
    )


# -------------------------------------------------------------------------
# Zero line
# -------------------------------------------------------------------------

ax_c.axvline(
    0,
    color=REFERENCE,
    linewidth=0.8,
    linestyle=":"
)


# -------------------------------------------------------------------------
# Labels
# -------------------------------------------------------------------------

ax_c.set_yticks(
    yc
)

ax_c.set_yticklabels(
    regime_order
)


ax_c.set_xlabel(
    "Relative change in regime prevalence (%)",
    labelpad=6
)

ax_c.set_title(
    "Compression is proportionally stronger at larger scales",
    loc="left",
    pad=8
)


# -------------------------------------------------------------------------
# Direct percentage values
# -------------------------------------------------------------------------

for yy, value in zip(
    yc,
    relative
):

    if value >= 0:

        ax_c.text(
            value + 0.35,
            yy,
            f"{value:+.1f}%",
            fontsize=8,
            fontweight="bold",
            color=POSITIVE,
            ha="left",
            va="center"
        )

    else:

        ax_c.text(
            value - 0.1,
            yy,
            f"{value:+.1f}%",
            fontsize=8,
            fontweight="bold",
            color=NEGATIVE,
            ha="right",
            va="center"
        )


ax_c.grid(
    axis="x",
    color=GRID,
    linewidth=0.35,
    alpha=0.45
)

ax_c.grid(
    axis="y",
    visible=False
)


ax_c.set_xlim(
    min(relative) - 1.0,
    max(relative) + 1.0
)


# =============================================================================
# 9. PANEL LETTERS
# =============================================================================

for ax, letter in zip(
    [
        ax_a,
        ax_b,
        ax_c
    ],
    [
        "a",
        "b",
        "c"
    ]
):

    ax.text(
        -0.055,
        1.075,
        letter,
        transform=ax.transAxes,
        fontsize=10.5,
        fontweight="bold",
        color="#222222",
        ha="left",
        va="top"
    )


# =============================================================================
# 10. AXIS SPACING
# =============================================================================

for ax in [
    ax_a,
    ax_b,
    ax_c
]:

    ax.tick_params(
        axis="x",
        pad=3
    )

    ax.tick_params(
        axis="y",
        pad=3
    )


# =============================================================================
# 11. FIGURE NOTE
# =============================================================================

fig.text(
    0.085,
    -0.02,
    (
        "Estimates compare papers by the same first author after removing "
        "accepted-month and subfield fixed effects."
    ),
    fontsize=6.2,
    color="#858585",
    ha="left",
    va="bottom"
)


# =============================================================================
# 12. SAVE
# =============================================================================

fig.savefig(
    "Figure2_AI_team_size_redistribution.png"
)

fig.savefig(
    "Figure2_AI_team_size_redistribution.pdf"
)

fig.savefig(
    "Figure2_AI_team_size_redistribution.svg"
)


# =============================================================================
# 13. DISPLAY
# =============================================================================

plt.show()

## Manuscript Figure 2 — Downstream transmission of prior knowledge

Final author-panel retention across follow-up windows, direct-reference-reuse estimates, and compact/rich-control comparisons used in the six-panel main-text figure.

### Reference-control sensitivity (main figure 2 panel c)

In [ ]:
# =============================================================================
# M4 — 24-MONTH FWL ROBUSTNESS SET
#
# Four specifications:
#
#   A. Unconditional retention — compact controls
#   B. Conditional retention   — compact controls
#   C. External retention      — compact controls
#   D. External retention      — rich reference-composition controls
#
# FE:
#   Author
#   Publication month
#   Subfield
#
# SE:
#   Clustered by author
#
# Effect reported:
#   Change in retention (percentage points) for a 1-SD increase in
#   residualized AI.
#
# Analysis only — no visualization.
# =============================================================================


import gc
import numpy as np
import pandas as pd
import statsmodels.api as sm


# =============================================================================
# 1. LOAD ONLY VARIABLES NEEDED
# =============================================================================

query = """
SELECT
    openalex_id,
    focal_first_author_id,
    accepted_year,
    accepted_month,
    subfield_id,

    ai_score,

    reference_count_raw,

    mean_reference_age_years,
    mean_log_prior_citations,
    reference_field_entropy,
    share_refs_same_field,

    share_refs_age_lt3,
    share_refs_age_3_5,
    share_refs_age_6_10,
    share_refs_age_11plus,

    share_refs_uncited_before_focal_year,

    reference_topic_entropy,
    reference_subfield_entropy,

    share_refs_same_topic,
    share_refs_same_subfield,
    share_refs_same_domain,
    share_refs_outside_field,
    share_refs_outside_domain,

    approximate_career_age,
    openalex_prior_article_review_count,
    baseline_mean_team_size,

    eligible_24m,
    cited_within_24m,
    externally_cited_within_24m,

    retention_breadth_24m_all_unconditional,
    retention_breadth_24m_all_conditional,
    retention_breadth_24m_external_conditional

FROM `aiwriting-488923.redo.m4_retention_summary`
"""

df = client.query(query).to_dataframe()


# =============================================================================
# 2. TYPE CLEANING
# =============================================================================

numeric_cols = [
    "accepted_year",
    "accepted_month",
    "ai_score",
    "reference_count_raw",

    "mean_reference_age_years",
    "mean_log_prior_citations",
    "reference_field_entropy",
    "share_refs_same_field",

    "share_refs_age_lt3",
    "share_refs_age_3_5",
    "share_refs_age_6_10",
    "share_refs_age_11plus",

    "share_refs_uncited_before_focal_year",

    "reference_topic_entropy",
    "reference_subfield_entropy",

    "share_refs_same_topic",
    "share_refs_same_subfield",
    "share_refs_same_domain",
    "share_refs_outside_field",
    "share_refs_outside_domain",

    "approximate_career_age",
    "openalex_prior_article_review_count",
    "baseline_mean_team_size",

    "retention_breadth_24m_all_unconditional",
    "retention_breadth_24m_all_conditional",
    "retention_breadth_24m_external_conditional",
]

for col in numeric_cols:

    if col in df.columns:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )


# =============================================================================
# 3. TRANSFORM SKEWED PRE-TREATMENT VARIABLES
# =============================================================================

df["log_reference_count"] = np.log1p(
    df["reference_count_raw"]
)

df["log_prior_productivity"] = np.log1p(
    df["openalex_prior_article_review_count"]
)


# =============================================================================
# 4. FE KEYS
# =============================================================================

df["author_code"] = pd.factorize(
    df["focal_first_author_id"],
    sort=False
)[0].astype(np.int64)


df["month_key"] = (
    df["accepted_year"]
    .fillna(-1)
    .astype(int)
    * 100
    +
    df["accepted_month"]
    .fillna(-1)
    .astype(int)
)


df["month_code"] = pd.factorize(
    df["month_key"],
    sort=False
)[0].astype(np.int64)


df["subfield_code"] = pd.factorize(
    df["subfield_id"],
    sort=False
)[0].astype(np.int64)


# =============================================================================
# 5. CONTROL SETS
#
# Compact:
#   reference quantity
#   reference age
#   predecessor prominence
#   field entropy
#   same-field share
#   career age
#   prior productivity
#   baseline team size
#
# Rich:
#   compact + detailed age composition
#   uncited-reference share
#   topic/subfield entropy
#   detailed disciplinary composition
# =============================================================================

compact_controls = [

    "log_reference_count",

    "mean_reference_age_years",

    "mean_log_prior_citations",

    "reference_field_entropy",

    "share_refs_same_field",

    "approximate_career_age",

    "log_prior_productivity",

    "baseline_mean_team_size",

]


rich_controls = [

    "log_reference_count",

    "mean_reference_age_years",

    "mean_log_prior_citations",

    "reference_field_entropy",

    "share_refs_same_field",

    "approximate_career_age",

    "log_prior_productivity",

    "baseline_mean_team_size",

    "share_refs_age_lt3",

    "share_refs_age_3_5",

    "share_refs_age_6_10",

    "share_refs_age_11plus",

    "share_refs_uncited_before_focal_year",

    "reference_topic_entropy",

    "reference_subfield_entropy",

    "share_refs_same_topic",

    "share_refs_same_subfield",

    "share_refs_same_domain",

    "share_refs_outside_field",

    "share_refs_outside_domain",

]


# Remove duplicated variable names while preserving order
compact_controls = list(
    dict.fromkeys(compact_controls)
)

rich_controls = list(
    dict.fromkeys(rich_controls)
)


# =============================================================================
# 6. MEMORY-EFFICIENT FWL
# =============================================================================

def fwl_matrix(
    X,
    groups,
    max_iter=150,
    tol=1e-8
):
    """
    Joint iterative within-transformation.

    X:
        N x K NumPy matrix

    groups:
        integer FE arrays

    The transformation cycles through all FE dimensions until convergence.
    """

    X = np.asarray(
        X,
        dtype=np.float64
    ).copy()


    previous = np.empty_like(
        X
    )


    for iteration in range(
        1,
        max_iter + 1
    ):

        previous[:] = X


        for g in groups:

            n_groups = int(
                np.max(g)
            ) + 1


            counts = np.bincount(
                g,
                minlength=n_groups
            ).astype(np.float64)


            sums = np.empty(
                (
                    n_groups,
                    X.shape[1]
                ),
                dtype=np.float64
            )


            for j in range(
                X.shape[1]
            ):

                sums[:, j] = np.bincount(
                    g,
                    weights=X[:, j],
                    minlength=n_groups
                )


            valid_counts = (
                counts > 0
            )


            means = np.zeros_like(
                sums
            )


            means[
                valid_counts,
                :
            ] = (
                sums[
                    valid_counts,
                    :
                ]
                /
                counts[
                    valid_counts,
                    None
                ]
            )


            X -= means[g]


        max_change = np.max(
            np.abs(
                X - previous
            )
        )


        if max_change < tol:

            return (
                X,
                iteration,
                max_change
            )


    return (
        X,
        max_iter,
        max_change
    )


# =============================================================================
# 7. MODEL FUNCTION
# =============================================================================

def run_fwl_model(
    data,
    outcome,
    sample_mask,
    controls,
    model_label
):

    # -------------------------------------------------------------------------
    # Select sample
    # -------------------------------------------------------------------------

    d = data.loc[
        sample_mask
    ].copy()


    # -------------------------------------------------------------------------
    # Required variables
    # -------------------------------------------------------------------------

    required = [
        "ai_score",
        outcome
    ] + controls


    required = [
        x for x in required
        if x in d.columns
    ]


    # -------------------------------------------------------------------------
    # Complete-case sample
    # -------------------------------------------------------------------------

    d = d.dropna(
        subset=required
    ).copy()


    if len(d) == 0:

        return None


    # -------------------------------------------------------------------------
    # Within-author AI variation
    # -------------------------------------------------------------------------

    author_ai_n = (
        d.groupby(
            "focal_first_author_id",
            sort=False
        )["ai_score"]
        .transform("nunique")
    )


    d = d.loc[
        author_ai_n >= 2
    ].copy()


    if len(d) == 0:

        return None


    # -------------------------------------------------------------------------
    # Matrix:
    #
    #   [ outcome | AI | controls ]
    # -------------------------------------------------------------------------

    matrix_cols = [
        outcome,
        "ai_score"
    ] + controls


    M = d[
        matrix_cols
    ].to_numpy(
        dtype=np.float64,
        copy=True
    )


    outcome_pos = (
        matrix_cols.index(
            outcome
        )
    )


    ai_pos = (
        matrix_cols.index(
            "ai_score"
        )
    )


    # -------------------------------------------------------------------------
    # FE arrays
    # -------------------------------------------------------------------------

    author = d[
        "author_code"
    ].to_numpy(
        dtype=np.int64
    )


    month = d[
        "month_code"
    ].to_numpy(
        dtype=np.int64
    )


    subfield = d[
        "subfield_code"
    ].to_numpy(
        dtype=np.int64
    )


    # -------------------------------------------------------------------------
    # FWL residualization
    # -------------------------------------------------------------------------

    M_resid, iterations, final_change = (
        fwl_matrix(
            M,
            groups=[
                author,
                month,
                subfield
            ],
            max_iter=150,
            tol=1e-8
        )
    )


    # -------------------------------------------------------------------------
    # Outcome
    # -------------------------------------------------------------------------

    Y = M_resid[
        :,
        outcome_pos
    ]


    # -------------------------------------------------------------------------
    # Predictors
    # -------------------------------------------------------------------------

    predictor_positions = [

        j

        for j in range(
            M_resid.shape[1]
        )

        if j != outcome_pos

    ]


    predictor_names = [

        matrix_cols[j]

        for j in predictor_positions

    ]


    X = M_resid[
        :,
        predictor_positions
    ]


    # -------------------------------------------------------------------------
    # Remove near-zero FE-residual variation
    # -------------------------------------------------------------------------

    keep = []


    for j in range(
        X.shape[1]
    ):

        sd = np.std(
            X[:, j],
            ddof=1
        )


        if (
            np.isfinite(sd)
            and sd > 1e-10
        ):

            keep.append(j)


    X = X[
        :,
        keep
    ]


    predictor_names = [

        predictor_names[j]

        for j in keep

    ]


    # -------------------------------------------------------------------------
    # Final finite mask
    # -------------------------------------------------------------------------

    valid = np.isfinite(
        Y
    )


    for j in range(
        X.shape[1]
    ):

        valid &= np.isfinite(
            X[:, j]
        )


    Y = Y[
        valid
    ]


    X = X[
        valid
    ]


    cluster = d.loc[
        valid,
        "author_code"
    ].to_numpy(
        dtype=np.int64
    )


    # -------------------------------------------------------------------------
    # Add constant
    # -------------------------------------------------------------------------

    X_model = sm.add_constant(
        X,
        has_constant="add"
    )


    # -------------------------------------------------------------------------
    # OLS
    # -------------------------------------------------------------------------

    model = sm.OLS(
        Y,
        X_model
    ).fit(
        cov_type="cluster",
        cov_kwds={
            "groups": cluster
        }
    )


    # -------------------------------------------------------------------------
    # Identify AI coefficient
    # -------------------------------------------------------------------------

    fitted_names = [
        "const"
    ] + predictor_names


    ai_position = fitted_names.index(
        "ai_score"
    )


    beta = float(
        model.params[
            ai_position
        ]
    )


    se = float(
        model.bse[
            ai_position
        ]
    )


    p = float(
        model.pvalues[
            ai_position
        ]
    )


    # -------------------------------------------------------------------------
    # Residualized AI SD
    # -------------------------------------------------------------------------

    ai_col = predictor_names.index(
        "ai_score"
    )


    ai_resid = X[
        :,
        ai_col
    ]


    ai_sd = float(
        np.std(
            ai_resid,
            ddof=1
        )
    )


    # -------------------------------------------------------------------------
    # 1-SD effect
    # -------------------------------------------------------------------------

    beta_1sd = (
        beta * ai_sd
    )


    se_1sd = (
        se * ai_sd
    )


    ci_low = (
        beta_1sd
        -
        1.96 * se_1sd
    )


    ci_high = (
        beta_1sd
        +
        1.96 * se_1sd
    )


    # -------------------------------------------------------------------------
    # Convert retention proportion to percentage points
    # -------------------------------------------------------------------------

    effect_pp = (
        100 * beta_1sd
    )


    se_pp = (
        100 * se_1sd
    )


    ci_low_pp = (
        100 * ci_low
    )


    ci_high_pp = (
        100 * ci_high
    )


    # -------------------------------------------------------------------------
    # Mean outcome in estimation sample
    # -------------------------------------------------------------------------

    original_y = d.loc[
        valid,
        outcome
    ]


    y_mean = float(
        original_y.mean()
    )


    pct_of_mean = np.nan


    if (
        np.isfinite(y_mean)
        and y_mean != 0
    ):

        pct_of_mean = (
            beta_1sd
            /
            y_mean
            *
            100
        )


    # -------------------------------------------------------------------------
    # Result
    # -------------------------------------------------------------------------

    result = {

        "model":
            model_label,

        "outcome":
            outcome,

        "controls":
            (
                "rich"
                if len(controls) >
                   len(compact_controls)
                else "compact"
            ),

        "beta_raw_ai":
            beta,

        "se_raw_ai":
            se,

        "p_raw_ai":
            p,

        "ai_resid_sd":
            ai_sd,

        "effect_pp_1sd":
            effect_pp,

        "se_pp_1sd":
            se_pp,

        "ci_low_pp_1sd":
            ci_low_pp,

        "ci_high_pp_1sd":
            ci_high_pp,

        "pct_of_mean_1sd":
            pct_of_mean,

        "n_obs":
            int(
                len(Y)
            ),

        "n_authors":
            int(
                d.loc[
                    valid,
                    "focal_first_author_id"
                ].nunique()
            ),

        "n_controls":
            len(
                predictor_names
            ) - 1,

        "fwl_iterations":
            iterations,

        "fwl_final_change":
            final_change,

    }


    # -------------------------------------------------------------------------
    # Cleanup
    # -------------------------------------------------------------------------

    del M
    del M_resid
    del X
    del X_model
    del Y
    d

    gc.collect()


    return result


# =============================================================================
# 8. SAMPLE DEFINITIONS
# =============================================================================

base_24m = (

    (df["eligible_24m"] == True)

    &

    (df["reference_count_raw"] >= 3)

)


sample_conditional = (

    base_24m

    &

    (df["cited_within_24m"] == True)

)


sample_external = (

    base_24m

    &

    (df["externally_cited_within_24m"] == True)

)


# =============================================================================
# 9. FOUR SPECIFICATIONS
# =============================================================================

model_specs = [

    (
        "A_24m_unconditional_compact",

        "retention_breadth_24m_all_unconditional",

        base_24m,

        compact_controls
    ),


    (
        "B_24m_conditional_compact",

        "retention_breadth_24m_all_conditional",

        sample_conditional,

        compact_controls
    ),


    (
        "C_24m_external_compact",

        "retention_breadth_24m_external_conditional",

        sample_external,

        compact_controls
    ),


    (
        "D_24m_external_rich",

        "retention_breadth_24m_external_conditional",

        sample_external,

        rich_controls
    ),

]


# =============================================================================
# 10. RUN
# =============================================================================

results = []


for model_label, outcome, sample, controls in model_specs:

    print(
        f"Running {model_label} ..."
    )


    result = run_fwl_model(
        data=df,
        outcome=outcome,
        sample_mask=sample,
        controls=controls,
        model_label=model_label
    )


    if result is not None:

        results.append(
            result
        )


    gc.collect()


# =============================================================================
# 11. RESULTS TABLE
# =============================================================================

results_df = pd.DataFrame(
    results
)


display(
    results_df[
        [
            "model",
            "outcome",
            "controls",
            "beta_raw_ai",
            "se_raw_ai",
            "p_raw_ai",
            "ai_resid_sd",
            "effect_pp_1sd",
            "se_pp_1sd",
            "ci_low_pp_1sd",
            "ci_high_pp_1sd",
            "pct_of_mean_1sd",
            "n_obs",
            "n_authors",
            "n_controls",
            "fwl_iterations",
            "fwl_final_change"
        ]
    ]
)


# =============================================================================
# 12. SAVE
# =============================================================================

results_df.to_csv(
    "m4_24m_fwl_robustness_results.csv",
    index=False
)


print(
    "\nSaved:"
)

print(
    "  m4_24m_fwl_robustness_results.csv"
)


### Direct reference reuse (main figure 2 panels b–c)

In [ ]:
# =============================================================================
# M4 — DIRECT REFERENCE REUSE
# WEIGHTED + UNWEIGHTED FWL
#
# UNIT:
#   focal paper
#
# OUTCOME:
#   mean share of the focal paper's references reproduced by downstream papers
#   that cite the focal paper within 24 months.
#
# PRIMARY MODELS:
#
#   A. Pair-weighted, all descendants
#   B. Pair-weighted, external descendants
#   C. Unweighted, all descendants
#   D. Pair-weighted, external descendants + rich controls
#
# FIXED EFFECTS:
#   focal author
#   focal publication month
#   focal subfield
#
# STANDARD ERRORS:
#   clustered by focal author
#
# IMPORTANT:
#   All focal-paper covariates are pulled from m4_retention_summary.
#   The direct-reuse table only supplies the descendant-level outcome summaries.
#
#   We do NOT load the 30.2M descendant-pair table into Python.
#   The estimation table is ~3.2M focal-paper rows.
# =============================================================================


import gc
import numpy as np
import pandas as pd
import statsmodels.api as sm


# =============================================================================
# 1. LOAD COMPACT FOCAL-PAPER DATA
#
# IMPORTANT:
#   The join is performed in BigQuery.
#   This prevents a large pandas-side merge and avoids referencing variables
#   that are not actually present in m4_direct_reuse_author_24m.
# =============================================================================

query = """
SELECT

    /*==========================================================================
      DIRECT-REUSE OUTCOMES
      ==========================================================================*/

    d.focal_openalex_id,

    d.focal_accepted_year,

    d.focal_ai_score,

    d.focal_subfield_id,

    d.focal_field_id,

    d.focal_domain_id,

    d.focal_source_id,

    d.focal_reference_count,

    d.n_descendant_pairs,

    d.n_external_descendant_pairs,

    d.mean_pair_reuse_rate,

    d.mean_external_pair_reuse_rate,


    /*==========================================================================
      FOCAL AUTHOR / TIME
      ==========================================================================*/

    f.focal_first_author_id,

    f.accepted_month,


    /*==========================================================================
      CAREER / PRIOR PRODUCTIVITY
      ==========================================================================*/

    f.approximate_career_age,

    f.openalex_prior_article_review_count,

    f.baseline_mean_team_size,


    /*==========================================================================
      REFERENCE COMPOSITION
      ==========================================================================*/

    f.mean_reference_age_years,

    f.mean_log_prior_citations,

    f.reference_field_entropy,

    f.reference_topic_entropy,

    f.reference_subfield_entropy,

    f.share_refs_same_field,

    f.share_refs_same_topic,

    f.share_refs_same_subfield,

    f.share_refs_same_domain,

    f.share_refs_outside_field,

    f.share_refs_outside_domain,

    f.share_refs_age_lt3,

    f.share_refs_age_3_5,

    f.share_refs_age_6_10,

    f.share_refs_age_11plus,

    f.share_refs_uncited_before_focal_year


FROM `aiwriting-488923.redo.m4_direct_reuse_author_24m` AS d

INNER JOIN `aiwriting-488923.redo.m4_retention_summary` AS f

    ON d.focal_openalex_id =
       f.openalex_id

WHERE

    d.n_descendant_pairs > 0
"""

df = client.query(
    query
).to_dataframe()


print(
    "Loaded focal-paper rows:",
    f"{len(df):,}"
)

print(
    "Focal papers:",
    f"{df['focal_openalex_id'].nunique():,}"
)

print(
    "Authors:",
    f"{df['focal_first_author_id'].nunique():,}"
)


# =============================================================================
# 2. NUMERIC CLEANING
# =============================================================================

numeric_cols = [

    "focal_accepted_year",

    "accepted_month",

    "focal_ai_score",

    "focal_reference_count",

    "n_descendant_pairs",

    "n_external_descendant_pairs",

    "mean_pair_reuse_rate",

    "mean_external_pair_reuse_rate",

    "approximate_career_age",

    "openalex_prior_article_review_count",

    "baseline_mean_team_size",

    "mean_reference_age_years",

    "mean_log_prior_citations",

    "reference_field_entropy",

    "reference_topic_entropy",

    "reference_subfield_entropy",

    "share_refs_same_field",

    "share_refs_same_topic",

    "share_refs_same_subfield",

    "share_refs_same_domain",

    "share_refs_outside_field",

    "share_refs_outside_domain",

    "share_refs_age_lt3",

    "share_refs_age_3_5",

    "share_refs_age_6_10",

    "share_refs_age_11plus",

    "share_refs_uncited_before_focal_year",

]


for col in numeric_cols:

    if col in df.columns:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )


# =============================================================================
# 3. CREATE TRANSFORMED CONTROLS
# =============================================================================

df["log_reference_count"] = np.log1p(
    df["focal_reference_count"]
)

df["log_prior_productivity"] = np.log1p(
    df["openalex_prior_article_review_count"]
)


# =============================================================================
# 4. CONSTRUCT FIXED-EFFECT KEYS
# =============================================================================

# -------------------------------------------------------------------------
# Author FE
# -------------------------------------------------------------------------

df["author_code"] = pd.factorize(
    df["focal_first_author_id"],
    sort=False
)[0].astype(
    np.int64
)


# -------------------------------------------------------------------------
# Publication-month FE
# -------------------------------------------------------------------------

df["month_key"] = (

    df["focal_accepted_year"]
    .fillna(-1)
    .astype(int)
    * 100

    +

    df["accepted_month"]
    .fillna(-1)
    .astype(int)

)


df["month_code"] = pd.factorize(
    df["month_key"],
    sort=False
)[0].astype(
    np.int64
)


# -------------------------------------------------------------------------
# Subfield FE
# -------------------------------------------------------------------------

df["subfield_code"] = pd.factorize(
    df["focal_subfield_id"],
    sort=False
)[0].astype(
    np.int64
)


# =============================================================================
# 5. CONTROL SETS
# =============================================================================

compact_controls = [

    "log_reference_count",

    "mean_reference_age_years",

    "mean_log_prior_citations",

    "reference_field_entropy",

    "share_refs_same_field",

    "approximate_career_age",

    "log_prior_productivity",

    "baseline_mean_team_size",

]


rich_controls = [

    # Compact controls
    "log_reference_count",

    "mean_reference_age_years",

    "mean_log_prior_citations",

    "reference_field_entropy",

    "share_refs_same_field",

    "approximate_career_age",

    "log_prior_productivity",

    "baseline_mean_team_size",

    # Detailed reference-age composition
    "share_refs_age_lt3",

    "share_refs_age_3_5",

    "share_refs_age_6_10",

    "share_refs_age_11plus",

    # Prior prominence
    "share_refs_uncited_before_focal_year",

    # Knowledge-base diversity
    "reference_topic_entropy",

    "reference_subfield_entropy",

    # Disciplinary composition
    "share_refs_same_topic",

    "share_refs_same_subfield",

    "share_refs_same_domain",

    "share_refs_outside_field",

    "share_refs_outside_domain",

]


# Remove duplicates
compact_controls = list(
    dict.fromkeys(
        compact_controls
    )
)

rich_controls = list(
    dict.fromkeys(
        rich_controls
    )
)


# =============================================================================
# 6. MEMORY-EFFICIENT FWL
# =============================================================================

def weighted_fwl_matrix(
    X,
    groups,
    weights,
    max_iter=150,
    tol=1e-8
):
    """
    Joint weighted within-transformation.

    For each FE dimension:

        x_i - weighted_group_mean(x)

    with:

        weighted_group_mean(x)
        =
        sum(w_i x_i) / sum(w_i)

    X contains:
        outcome
        AI
        controls

    All variables are residualized simultaneously.
    """

    X = np.asarray(
        X,
        dtype=np.float64
    ).copy()


    weights = np.asarray(
        weights,
        dtype=np.float64
    )


    previous = np.empty_like(
        X
    )


    for iteration in range(
        1,
        max_iter + 1
    ):

        previous[:] = X


        for g in groups:

            n_groups = int(
                np.max(g)
            ) + 1


            weighted_counts = np.bincount(
                g,
                weights=weights,
                minlength=n_groups
            )


            means = np.zeros(
                (
                    n_groups,
                    X.shape[1]
                ),
                dtype=np.float64
            )


            valid_groups = (
                weighted_counts > 0
            )


            for j in range(
                X.shape[1]
            ):

                weighted_sums = np.bincount(
                    g,
                    weights=(
                        weights
                        *
                        X[:, j]
                    ),
                    minlength=n_groups
                )


                means[
                    valid_groups,
                    j
                ] = (

                    weighted_sums[
                        valid_groups
                    ]

                    /

                    weighted_counts[
                        valid_groups
                    ]

                )


            X -= means[g]


        max_change = np.max(
            np.abs(
                X - previous
            )
        )


        if max_change < tol:

            return (
                X,
                iteration,
                max_change
            )


    return (
        X,
        max_iter,
        max_change
    )


# =============================================================================
# 7. DIRECT-REUSE MODEL
# =============================================================================

def run_direct_reuse_model(
    data,
    outcome,
    weight_var,
    sample_mask,
    controls,
    model_label,
    weighted=True
):

    # -------------------------------------------------------------------------
    # Select sample
    # -------------------------------------------------------------------------

    d = data.loc[
        sample_mask
    ].copy()


    # -------------------------------------------------------------------------
    # Required variables
    # -------------------------------------------------------------------------

    required = [

        "focal_ai_score",

        outcome,

        weight_var,

    ] + controls


    required = list(
        dict.fromkeys(
            required
        )
    )


    d = d.dropna(
        subset=required
    ).copy()


    if len(d) == 0:

        return None


    # -------------------------------------------------------------------------
    # Positive weights
    # -------------------------------------------------------------------------

    d = d[
        d[weight_var] > 0
    ].copy()


    if len(d) == 0:

        return None


    # -------------------------------------------------------------------------
    # Require within-author variation in AI
    # -------------------------------------------------------------------------

    author_ai_n = (

        d.groupby(
            "focal_first_author_id",
            sort=False
        )["focal_ai_score"]

        .transform(
            "nunique"
        )

    )


    d = d.loc[
        author_ai_n >= 2
    ].copy()


    if len(d) == 0:

        return None


    # -------------------------------------------------------------------------
    # Matrix
    #
    # [ outcome | AI | controls ]
    # -------------------------------------------------------------------------

    matrix_cols = [

        outcome,

        "focal_ai_score"

    ] + controls


    M = d[
        matrix_cols
    ].to_numpy(
        dtype=np.float64,
        copy=True
    )


    outcome_pos = matrix_cols.index(
        outcome
    )


    # -------------------------------------------------------------------------
    # FE arrays
    # -------------------------------------------------------------------------

    author = d[
        "author_code"
    ].to_numpy(
        dtype=np.int64
    )


    month = d[
        "month_code"
    ].to_numpy(
        dtype=np.int64
    )


    subfield = d[
        "subfield_code"
    ].to_numpy(
        dtype=np.int64
    )


    # -------------------------------------------------------------------------
    # Regression weights
    # -------------------------------------------------------------------------

    weights = d[
        weight_var
    ].to_numpy(
        dtype=np.float64
    )


    # -------------------------------------------------------------------------
    # FWL
    # -------------------------------------------------------------------------

    M_resid, iterations, final_change = (
        weighted_fwl_matrix(

            X=M,

            groups=[
                author,
                month,
                subfield
            ],

            weights=(
                weights
                if weighted
                else np.ones(
                    len(d),
                    dtype=np.float64
                )
            ),

            max_iter=150,

            tol=1e-8
        )
    )


    # -------------------------------------------------------------------------
    # Residualized outcome
    # -------------------------------------------------------------------------

    Y = M_resid[
        :,
        outcome_pos
    ]


    # -------------------------------------------------------------------------
    # Residualized predictors
    # -------------------------------------------------------------------------

    predictor_positions = [

        j

        for j in range(
            M_resid.shape[1]
        )

        if j != outcome_pos

    ]


    predictor_names = [

        matrix_cols[j]

        for j in predictor_positions

    ]


    X = M_resid[
        :,
        predictor_positions
    ]


    # -------------------------------------------------------------------------
    # Remove zero-variation predictors
    # -------------------------------------------------------------------------

    keep = []


    for j in range(
        X.shape[1]
    ):

        sd = np.std(
            X[:, j],
            ddof=1
        )


        if (
            np.isfinite(sd)
            and sd > 1e-10
        ):

            keep.append(
                j
            )


    X = X[
        :,
        keep
    ]


    predictor_names = [

        predictor_names[j]

        for j in keep

    ]


    # -------------------------------------------------------------------------
    # Final finite mask
    # -------------------------------------------------------------------------

    valid = np.isfinite(
        Y
    )


    for j in range(
        X.shape[1]
    ):

        valid &= np.isfinite(
            X[:, j]
        )


    Y = Y[
        valid
    ]


    X = X[
        valid
    ]


    weights = weights[
        valid
    ]


    cluster = d.loc[
        valid,
        "author_code"
    ].to_numpy(
        dtype=np.int64
    )


    # -------------------------------------------------------------------------
    # Add constant
    # -------------------------------------------------------------------------

    X_model = sm.add_constant(
        X,
        has_constant="add"
    )


    # -------------------------------------------------------------------------
    # Estimate
    # -------------------------------------------------------------------------

    if weighted:

        model = sm.WLS(

            Y,

            X_model,

            weights=weights

        ).fit(

            cov_type="cluster",

            cov_kwds={
                "groups":
                    cluster
            }

        )

    else:

        model = sm.OLS(

            Y,

            X_model

        ).fit(

            cov_type="cluster",

            cov_kwds={
                "groups":
                    cluster
            }

        )


    # -------------------------------------------------------------------------
    # Locate AI coefficient
    # -------------------------------------------------------------------------

    fitted_names = [
        "const"
    ] + predictor_names


    ai_position = fitted_names.index(
        "focal_ai_score"
    )


    beta = float(
        model.params[
            ai_position
        ]
    )


    se = float(
        model.bse[
            ai_position
        ]
    )


    p = float(
        model.pvalues[
            ai_position
        ]
    )


    # -------------------------------------------------------------------------
    # Residualized AI SD
    # -------------------------------------------------------------------------

    ai_col = predictor_names.index(
        "focal_ai_score"
    )


    ai_resid = X[
        :,
        ai_col
    ]


    ai_sd = float(
        np.std(
            ai_resid,
            ddof=1
        )
    )


    # -------------------------------------------------------------------------
    # One-SD effect
    # -------------------------------------------------------------------------

    beta_1sd = (
        beta
        *
        ai_sd
    )


    se_1sd = (
        se
        *
        ai_sd
    )


    ci_low = (
        beta_1sd
        -
        1.96 * se_1sd
    )


    ci_high = (
        beta_1sd
        +
        1.96 * se_1sd
    )


    # -------------------------------------------------------------------------
    # Percentage points
    # -------------------------------------------------------------------------

    effect_pp = (
        beta_1sd
        *
        100
    )


    se_pp = (
        se_1sd
        *
        100
    )


    ci_low_pp = (
        ci_low
        *
        100
    )


    ci_high_pp = (
        ci_high
        *
        100
    )


    # -------------------------------------------------------------------------
    # Mean outcome
    # -------------------------------------------------------------------------

    y_original = d.loc[
        valid,
        outcome
    ].to_numpy(
        dtype=np.float64
    )


    y_mean = float(
        np.mean(
            y_original
        )
    )


    # -------------------------------------------------------------------------
    # Weighted outcome mean
    # -------------------------------------------------------------------------

    weighted_y_mean = float(

        np.sum(
            weights
            *
            y_original
        )

        /

        np.sum(
            weights
        )

    )


    # -------------------------------------------------------------------------
    # Relative effect
    # -------------------------------------------------------------------------

    pct_of_mean = np.nan


    if (
        np.isfinite(
            y_mean
        )
        and
        y_mean != 0
    ):

        pct_of_mean = (

            beta_1sd
            /
            y_mean
            *
            100

        )


    # -------------------------------------------------------------------------
    # Number of clusters
    # -------------------------------------------------------------------------

    n_authors = int(
        len(
            np.unique(
                cluster
            )
        )
    )


    # -------------------------------------------------------------------------
    # Return
    # -------------------------------------------------------------------------

    result = {

        "model":
            model_label,

        "outcome":
            outcome,

        "weighted":
            weighted,

        "weight_variable":
            (
                weight_var
                if weighted
                else "none"
            ),

        "controls":
            (
                "rich"
                if len(controls)
                   > len(compact_controls)
                else "compact"
            ),

        "beta_raw_ai":
            beta,

        "se_raw_ai":
            se,

        "p_raw_ai":
            p,

        "ai_resid_sd":
            ai_sd,

        "effect_pp_1sd":
            effect_pp,

        "se_pp_1sd":
            se_pp,

        "ci_low_pp_1sd":
            ci_low_pp,

        "ci_high_pp_1sd":
            ci_high_pp,

        "pct_of_mean_1sd":
            pct_of_mean,

        "mean_outcome":
            y_mean,

        "weighted_mean_outcome":
            weighted_y_mean,

        "n_focal_papers":
            int(
                len(Y)
            ),

        "n_authors":
            n_authors,

        "total_weight":
            float(
                np.sum(
                    weights
                )
            ),

        "n_controls":
            len(
                predictor_names
            ) - 1,

        "fwl_iterations":
            iterations,

        "fwl_final_change":
            final_change,

    }


    # -------------------------------------------------------------------------
    # Free memory
    # -------------------------------------------------------------------------

    del M
    del M_resid
    del X
    del X_model
    del Y
    d

    gc.collect()


    return result


# =============================================================================
# 8. SAMPLE DEFINITIONS
# =============================================================================

base_sample = (

    df["focal_reference_count"]
    >= 3

)


all_descendant_sample = (

    base_sample

    &

    (
        df["n_descendant_pairs"]
        > 0
    )

)


external_sample = (

    base_sample

    &

    (
        df["n_external_descendant_pairs"]
        > 0
    )

)


# =============================================================================
# 9. MODEL DEFINITIONS
# =============================================================================

model_specs = [

    (
        "A_pair_weighted_all",

        "mean_pair_reuse_rate",

        "n_descendant_pairs",

        all_descendant_sample,

        compact_controls,

        True
    ),


    (
        "B_pair_weighted_external",

        "mean_external_pair_reuse_rate",

        "n_external_descendant_pairs",

        external_sample,

        compact_controls,

        True
    ),


    (
        "C_unweighted_all",

        "mean_pair_reuse_rate",

        "n_descendant_pairs",

        all_descendant_sample,

        compact_controls,

        False
    ),


    (
        "D_pair_weighted_external_rich",

        "mean_external_pair_reuse_rate",

        "n_external_descendant_pairs",

        external_sample,

        rich_controls,

        True
    ),

]


# =============================================================================
# 10. RUN MODELS
# =============================================================================

results = []


for (

    model_label,

    outcome,

    weight_var,

    sample,

    controls,

    weighted

) in model_specs:

    print(
        f"Running {model_label} ..."
    )


    result = run_direct_reuse_model(

        data=df,

        outcome=outcome,

        weight_var=weight_var,

        sample_mask=sample,

        controls=controls,

        model_label=model_label,

        weighted=weighted

    )


    if result is not None:

        results.append(
            result
        )


    gc.collect()


# =============================================================================
# 11. RESULTS TABLE
# =============================================================================

results_df = pd.DataFrame(
    results
)


if len(results_df) > 0:

    display(

        results_df[

            [
                "model",

                "outcome",

                "weighted",

                "controls",

                "beta_raw_ai",

                "se_raw_ai",

                "p_raw_ai",

                "ai_resid_sd",

                "effect_pp_1sd",

                "se_pp_1sd",

                "ci_low_pp_1sd",

                "ci_high_pp_1sd",

                "pct_of_mean_1sd",

                "mean_outcome",

                "weighted_mean_outcome",

                "n_focal_papers",

                "n_authors",

                "n_controls",

                "fwl_iterations",

                "fwl_final_change",

            ]

        ]

    )


# =============================================================================
# 12. SAVE
# =============================================================================

results_df.to_csv(
    "m4_direct_reuse_fwl_results.csv",
    index=False
)


print(
    "\nSaved:"
)

print(
    "  m4_direct_reuse_fwl_results.csv"
)

### Common author-panel retention (main figure 2 panel a)

In [ ]:
# =============================================================================
# M4 — COMMON LONGITUDINAL AUTHOR PANEL + COMPLETE 4 x 3 HORIZON GRID
#
# AUTHOR-PANEL DEFINITION
# -----------------------
# Match Figure 1's logic:
#
#   1. use the full focal-paper universe to define the author panel
#   2. keep authors with >=2 focal papers
#   3. require within-author variation in AI score
#   4. freeze that author cohort
#   5. apply horizon-specific outcome/sample restrictions afterward
#
# This means the AUTHOR COHORT is common across all M4 horizons.
#
#                           12m          24m          36m
#
# All eligible papers       compact      compact      compact
# Papers cited downstream   compact      compact      compact
# No-overlap authorship     compact      compact      compact
# No-overlap + rich         rich         rich         rich
#
# Fixed effects:
#   focal first-author FE
#   accepted-month FE
#   subfield FE
#
# SE:
#   clustered by focal first author
#
# =============================================================================


import gc
import numpy as np
import pandas as pd
import statsmodels.api as sm


# =============================================================================
# 1. LOAD DATA
# =============================================================================

query = """
SELECT

    /*----------------------------------------------------------------------
      IDENTIFIERS
    ----------------------------------------------------------------------*/

    openalex_id,

    focal_first_author_id,

    accepted_year,

    accepted_month,

    subfield_id,


    /*----------------------------------------------------------------------
      TREATMENT
    ----------------------------------------------------------------------*/

    ai_score,


    /*----------------------------------------------------------------------
      CONTROLS
    ----------------------------------------------------------------------*/

    reference_count_raw,

    reference_count_resolved,

    reference_count_temporally_valid,

    share_references_resolved,

    share_references_temporally_valid,

    mean_reference_age_years,

    mean_log_prior_citations,

    reference_field_entropy,

    share_refs_same_field,

    share_refs_same_subfield,

    share_refs_outside_field,

    share_refs_uncited_before_focal_year,

    n_reference_topics,

    reference_topic_entropy,

    n_reference_subfields,

    reference_subfield_entropy,

    n_reference_fields,

    approximate_career_age,

    openalex_prior_article_review_count,

    baseline_mean_team_size,


    /*----------------------------------------------------------------------
      HORIZON ELIGIBILITY
    ----------------------------------------------------------------------*/

    eligible_12m,

    eligible_24m,

    eligible_36m,


    /*----------------------------------------------------------------------
      DOWNSTREAM CITATION STATUS
    ----------------------------------------------------------------------*/

    cited_within_12m,

    cited_within_24m,

    cited_within_36m,

    externally_cited_within_12m,

    externally_cited_within_24m,

    externally_cited_within_36m,


    /*----------------------------------------------------------------------
      OUTCOMES — ALL ELIGIBLE / UNCONDITIONAL
    ----------------------------------------------------------------------*/

    retention_breadth_12m_all_unconditional,

    retention_breadth_24m_all_unconditional,

    retention_breadth_36m_all_unconditional,


    /*----------------------------------------------------------------------
      OUTCOMES — CITED DOWNSTREAM / CONDITIONAL
    ----------------------------------------------------------------------*/

    retention_breadth_12m_all_conditional,

    retention_breadth_24m_all_conditional,

    retention_breadth_36m_all_conditional,


    /*----------------------------------------------------------------------
      OUTCOMES — NO-OVERLAPPING AUTHORSHIP
    ----------------------------------------------------------------------*/

    retention_breadth_12m_external_conditional,

    retention_breadth_24m_external_conditional,

    retention_breadth_36m_external_conditional

FROM
    `aiwriting-488923.redo.m4_retention_summary`
"""


df = client.query(
    query
).to_dataframe()


print(
    "Loaded rows:",
    f"{len(df):,}"
)

print(
    "Unique focal papers:",
    f"{df['openalex_id'].nunique():,}"
)

print(
    "Unique focal first authors:",
    f"{df['focal_first_author_id'].nunique():,}"
)


# =============================================================================
# 2. TYPE CLEANING
# =============================================================================

numeric_cols = [

    "accepted_year",

    "accepted_month",

    "ai_score",

    "reference_count_raw",

    "reference_count_resolved",

    "reference_count_temporally_valid",

    "share_references_resolved",

    "share_references_temporally_valid",

    "mean_reference_age_years",

    "mean_log_prior_citations",

    "reference_field_entropy",

    "share_refs_same_field",

    "share_refs_same_subfield",

    "share_refs_outside_field",

    "share_refs_uncited_before_focal_year",

    "n_reference_topics",

    "reference_topic_entropy",

    "n_reference_subfields",

    "reference_subfield_entropy",

    "n_reference_fields",

    "approximate_career_age",

    "openalex_prior_article_review_count",

    "baseline_mean_team_size",

    "retention_breadth_12m_all_unconditional",

    "retention_breadth_24m_all_unconditional",

    "retention_breadth_36m_all_unconditional",

    "retention_breadth_12m_all_conditional",

    "retention_breadth_24m_all_conditional",

    "retention_breadth_36m_all_conditional",

    "retention_breadth_12m_external_conditional",

    "retention_breadth_24m_external_conditional",

    "retention_breadth_36m_external_conditional",

]


for col in numeric_cols:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )


# =============================================================================
# 3. CLEAN CORE IDENTIFIERS
# =============================================================================

df = df.loc[

    df["focal_first_author_id"].notna()

    &

    df["accepted_year"].notna()

    &

    df["accepted_month"].notna()

    &

    df["subfield_id"].notna()

    &

    df["ai_score"].notna()

].copy()


df = df.loc[
    np.isfinite(
        df["ai_score"]
    )
].copy()


# =============================================================================
# 4. DEFINE THE COMMON AUTHOR COHORT
#
# EXACTLY THE PRINCIPLE USED IN FIGURE 1:
#
#   authors with >=2 papers
#   AND positive within-author AI variation
#
# IMPORTANT:
# This is computed BEFORE any 12m / 24m / 36m restriction.
# =============================================================================

author_summary = (

    df

    .groupby(
        "focal_first_author_id",
        observed=True
    )

    .agg(

        n_focal_papers=(
            "openalex_id",
            "size"
        ),

        ai_sd=(
            "ai_score",
            "std"
        )

    )

)


valid_authors = author_summary.index[
    (
        author_summary["n_focal_papers"]
        >= 2
    )

    &

    (
        author_summary["ai_sd"]
        > 0
    )
]


print(
    "\nCOMMON AUTHOR COHORT"
)

print(
    "Authors before cohort restriction:",
    f"{df['focal_first_author_id'].nunique():,}"
)

print(
    "Authors in common cohort:",
    f"{len(valid_authors):,}"
)


# Freeze the author cohort
df = df.loc[
    df["focal_first_author_id"].isin(
        valid_authors
    )
].copy()


print(
    "Focal papers after common-author restriction:",
    f"{len(df):,}"
)


# =============================================================================
# 5. INTEGER-ENCODE FIXED EFFECTS
# =============================================================================

author_code, _ = pd.factorize(
    df["focal_first_author_id"],
    sort=False
)

month_key = (

    df["accepted_year"].astype(np.int64)
    * 100

    +

    df["accepted_month"].astype(np.int64)

)


month_code, _ = pd.factorize(
    month_key,
    sort=False
)


subfield_code, _ = pd.factorize(
    df["subfield_id"],
    sort=False
)


author_code = author_code.astype(
    np.int64
)

month_code = month_code.astype(
    np.int64
)

subfield_code = subfield_code.astype(
    np.int64
)


# =============================================================================
# 6. CONSTRUCT TRANSFORMED CONTROLS
# =============================================================================

df["log_reference_count"] = np.log1p(
    df["reference_count_raw"]
)


df["log_prior_productivity"] = np.log1p(
    df["openalex_prior_article_review_count"]
)


# =============================================================================
# 7. CONTROL SETS
#
# The lists are fixed across all horizons.
# =============================================================================

compact_controls = [

    "log_reference_count",

    "mean_reference_age_years",

    "mean_log_prior_citations",

    "reference_field_entropy",

    "share_refs_same_field",

    "approximate_career_age",

    "log_prior_productivity",

    "baseline_mean_team_size",

]


rich_controls = [

    "log_reference_count",

    "mean_reference_age_years",

    "mean_log_prior_citations",

    "reference_field_entropy",

    "share_refs_same_field",

    "approximate_career_age",

    "log_prior_productivity",

    "baseline_mean_team_size",

    "share_refs_uncited_before_focal_year",

    "n_reference_topics",

    "reference_topic_entropy",

    "n_reference_subfields",

    "reference_subfield_entropy",

    "n_reference_fields",

    "share_refs_same_subfield",

    "share_refs_outside_field",

    "reference_count_resolved",

    "reference_count_temporally_valid",

    "share_references_temporally_valid",

]


assert len(compact_controls) == 8

assert len(rich_controls) == 19


print(
    "\nCompact controls:",
    len(compact_controls)
)

print(
    "Rich controls:",
    len(rich_controls)
)


# =============================================================================
# 8. FAST GROUP-DEMEAN
# =============================================================================

def group_demean_matrix(
    X,
    groups,
    n_groups
):

    X = np.asarray(
        X,
        dtype=np.float64
    )


    out = np.empty_like(
        X
    )


    counts = np.bincount(
        groups,
        minlength=n_groups
    ).astype(
        np.float64
    )


    counts[counts == 0] = 1.0


    for j in range(
        X.shape[1]
    ):

        sums = np.bincount(

            groups,

            weights=X[:, j],

            minlength=n_groups

        )


        means = (
            sums
            /
            counts
        )


        out[:, j] = (

            X[:, j]

            -

            means[groups]

        )


    return out


# =============================================================================
# 9. ITERATIVE FWL
# =============================================================================

def fwl_residualize(
    X,
    group_arrays,
    group_sizes,
    tol=1e-8,
    max_iter=100
):

    R = np.asarray(
        X,
        dtype=np.float64
    ).copy()


    for iteration in range(
        1,
        max_iter + 1
    ):

        old = R.copy()


        for groups, n_groups in zip(

            group_arrays,

            group_sizes

        ):

            R = group_demean_matrix(

                R,

                groups,

                n_groups

            )


        change = np.max(

            np.abs(
                R - old
            )

        )


        if change < tol:

            return (

                R,

                iteration,

                change

            )


    return (

        R,

        max_iter,

        change

    )


# =============================================================================
# 10. MODEL FUNCTION
# =============================================================================

def run_model(

    data,

    outcome,

    sample_mask,

    label,

    controls

):

    print(
        f"\nRunning {label} ..."
    )


    # -------------------------------------------------------------------------
    # Apply horizon/sample restriction AFTER the common author cohort is fixed
    # -------------------------------------------------------------------------

    d = data.loc[
        sample_mask
    ].copy()


    print(
        "  sample:",
        f"{len(d):,}"
    )


    # -------------------------------------------------------------------------
    # Complete cases
    # -------------------------------------------------------------------------

    required = [

        outcome,

        "ai_score"

    ] + controls


    d = d.dropna(
        subset=required
    ).copy()


    print(
        "  complete cases:",
        f"{len(d):,}"
    )


    if len(d) == 0:

        return None


    # -------------------------------------------------------------------------
    # Integer-coded FE
    #
    # Re-factorize only the retained rows so memory stays compact.
    # The AUTHOR COHORT itself remains fixed; only the category codes change.
    # -------------------------------------------------------------------------

    author, _ = pd.factorize(
        d["focal_first_author_id"],
        sort=False
    )

    month_key_d = (

        d["accepted_year"].astype(np.int64)
        * 100

        +

        d["accepted_month"].astype(np.int64)

    )

    month, _ = pd.factorize(
        month_key_d,
        sort=False
    )

    subfield, _ = pd.factorize(
        d["subfield_id"],
        sort=False
    )


    author = author.astype(
        np.int64
    )

    month = month.astype(
        np.int64
    )

    subfield = subfield.astype(
        np.int64
    )


    # -------------------------------------------------------------------------
    # Build matrix
    # -------------------------------------------------------------------------

    matrix_cols = [

        outcome,

        "ai_score"

    ] + controls


    M = d[
        matrix_cols
    ].to_numpy(

        dtype=np.float64,

        copy=True

    )


    outcome_pos = 0


    # -------------------------------------------------------------------------
    # FWL
    # -------------------------------------------------------------------------

    M_resid, iterations, final_change = (

        fwl_residualize(

            M,

            group_arrays=[

                author,

                month,

                subfield

            ],

            group_sizes=[

                int(
                    author.max()
                ) + 1,

                int(
                    month.max()
                ) + 1,

                int(
                    subfield.max()
                ) + 1,

            ],

            tol=1e-8,

            max_iter=100

        )

    )


    # -------------------------------------------------------------------------
    # Residualized variables
    # -------------------------------------------------------------------------

    Y = M_resid[
        :,
        outcome_pos
    ]


    X = M_resid[
        :,
        1:
    ]


    predictor_names = [

        "ai_score"

    ] + controls


    # -------------------------------------------------------------------------
    # AI is first predictor
    # -------------------------------------------------------------------------

    ai_resid = X[
        :,
        0
    ]


    # -------------------------------------------------------------------------
    # Finite observations
    # -------------------------------------------------------------------------

    valid = (

        np.isfinite(Y)

        &

        np.all(
            np.isfinite(X),
            axis=1
        )

    )


    Y_valid = Y[
        valid
    ]


    X_valid = X[
        valid
    ]


    cluster = author[
        valid
    ]


    # -------------------------------------------------------------------------
    # OLS
    # -------------------------------------------------------------------------

    X_model = sm.add_constant(

        X_valid,

        has_constant="add"

    )


    model = sm.OLS(

        Y_valid,

        X_model

    ).fit(

        cov_type="cluster",

        cov_kwds={

            "groups":
                cluster

        }

    )


    # -------------------------------------------------------------------------
    # AI coefficient
    # -------------------------------------------------------------------------

    beta = float(
        model.params[1]
    )


    se = float(
        model.bse[1]
    )


    p = float(
        model.pvalues[1]
    )


    # -------------------------------------------------------------------------
    # Residualized AI SD
    # -------------------------------------------------------------------------

    ai_sd = float(

        np.std(

            X_valid[:, 0],

            ddof=0

        )

    )


    beta_1sd = (
        beta
        *
        ai_sd
    )


    se_1sd = (
        se
        *
        ai_sd
    )


    ci_low = (

        beta_1sd

        -

        1.96 * se_1sd

    )


    ci_high = (

        beta_1sd

        +

        1.96 * se_1sd

    )


    # -------------------------------------------------------------------------
    # Percentage points
    # -------------------------------------------------------------------------

    effect_pp = (
        beta_1sd
        *
        100
    )


    se_pp = (
        se_1sd
        *
        100
    )


    ci_low_pp = (
        ci_low
        *
        100
    )


    ci_high_pp = (
        ci_high
        *
        100
    )


    # -------------------------------------------------------------------------
    # Outcome mean
    # -------------------------------------------------------------------------

    y_original = d.loc[
        valid,
        outcome
    ].to_numpy(
        dtype=np.float64
    )


    mean_outcome = float(
        np.mean(
            y_original
        )
    )


    pct_of_mean = np.nan


    if (

        np.isfinite(
            mean_outcome
        )

        and

        mean_outcome != 0

    ):

        pct_of_mean = (

            beta_1sd

            /

            mean_outcome

            *

            100

        )


    # -------------------------------------------------------------------------
    # Final result
    # -------------------------------------------------------------------------

    result = {

        "model":
            label,

        "control_set":
            (
                "compact"
                if controls == compact_controls
                else "rich"
            ),

        "outcome":
            outcome,

        "beta_raw_ai":
            beta,

        "se_raw_ai":
            se,

        "p_raw_ai":
            p,

        "ai_resid_sd":
            ai_sd,

        "effect_pp_1sd":
            effect_pp,

        "se_pp_1sd":
            se_pp,

        "ci_low_pp_1sd":
            ci_low_pp,

        "ci_high_pp_1sd":
            ci_high_pp,

        "pct_of_mean_1sd":
            pct_of_mean,

        "mean_outcome":
            mean_outcome,

        "n_obs":
            int(
                len(Y_valid)
            ),

        "n_authors":
            int(
                np.unique(
                    cluster
                ).size
            ),

        "n_controls":
            len(controls),

        "fwl_iterations":
            iterations,

        "fwl_final_change":
            final_change,

    }


    # -------------------------------------------------------------------------
    # Cleanup
    # -------------------------------------------------------------------------

    del M

    del M_resid

    del X_valid

    del X_model

    del Y_valid

    del d

    gc.collect()


    return result


# =============================================================================
# 11. DEFINE THE 12 SPECIFICATIONS
#
# IMPORTANT:
# The common author cohort was already frozen above.
# =============================================================================

specifications = [

    # -------------------------------------------------------------------------
    # ALL ELIGIBLE PAPERS — COMPACT
    # -------------------------------------------------------------------------

    (
        "12m_all_eligible_compact",

        "retention_breadth_12m_all_unconditional",

        df["eligible_12m"],

        compact_controls
    ),

    (
        "24m_all_eligible_compact",

        "retention_breadth_24m_all_unconditional",

        df["eligible_24m"],

        compact_controls
    ),

    (
        "36m_all_eligible_compact",

        "retention_breadth_36m_all_unconditional",

        df["eligible_36m"],

        compact_controls
    ),


    # -------------------------------------------------------------------------
    # PAPERS CITED DOWNSTREAM — COMPACT
    # -------------------------------------------------------------------------

    (
        "12m_cited_downstream_compact",

        "retention_breadth_12m_all_conditional",

        (
            df["eligible_12m"]
            &
            df["cited_within_12m"]
        ),

        compact_controls
    ),

    (
        "24m_cited_downstream_compact",

        "retention_breadth_24m_all_conditional",

        (
            df["eligible_24m"]
            &
            df["cited_within_24m"]
        ),

        compact_controls
    ),

    (
        "36m_cited_downstream_compact",

        "retention_breadth_36m_all_conditional",

        (
            df["eligible_36m"]
            &
            df["cited_within_36m"]
        ),

        compact_controls
    ),


    # -------------------------------------------------------------------------
    # NO-OVERLAPPING AUTHORSHIP — COMPACT
    # -------------------------------------------------------------------------

    (
        "12m_no_overlap_compact",

        "retention_breadth_12m_external_conditional",

        (
            df["eligible_12m"]
            &
            df["externally_cited_within_12m"]
        ),

        compact_controls
    ),

    (
        "24m_no_overlap_compact",

        "retention_breadth_24m_external_conditional",

        (
            df["eligible_24m"]
            &
            df["externally_cited_within_24m"]
        ),

        compact_controls
    ),

    (
        "36m_no_overlap_compact",

        "retention_breadth_36m_external_conditional",

        (
            df["eligible_36m"]
            &
            df["externally_cited_within_36m"]
        ),

        compact_controls
    ),


    # -------------------------------------------------------------------------
    # NO-OVERLAPPING AUTHORSHIP + RICH CONTROLS
    # -------------------------------------------------------------------------

    (
        "12m_no_overlap_rich",

        "retention_breadth_12m_external_conditional",

        (
            df["eligible_12m"]
            &
            df["externally_cited_within_12m"]
        ),

        rich_controls
    ),

    (
        "24m_no_overlap_rich",

        "retention_breadth_24m_external_conditional",

        (
            df["eligible_24m"]
            &
            df["externally_cited_within_24m"]
        ),

        rich_controls
    ),

    (
        "36m_no_overlap_rich",

        "retention_breadth_36m_external_conditional",

        (
            df["eligible_36m"]
            &
            df["externally_cited_within_36m"]
        ),

        rich_controls
    ),

]


# =============================================================================
# 12. RUN ALL 12
# =============================================================================

results = []


for label, outcome, sample_mask, controls in specifications:

    result = run_model(

        data=df,

        outcome=outcome,

        sample_mask=sample_mask,

        label=label,

        controls=controls

    )


    if result is not None:

        results.append(
            result
        )


    gc.collect()


# =============================================================================
# 13. ORGANIZE RESULTS
# =============================================================================

results_df = pd.DataFrame(
    results
)


model_order = [

    "12m_all_eligible_compact",
    "24m_all_eligible_compact",
    "36m_all_eligible_compact",

    "12m_cited_downstream_compact",
    "24m_cited_downstream_compact",
    "36m_cited_downstream_compact",

    "12m_no_overlap_compact",
    "24m_no_overlap_compact",
    "36m_no_overlap_compact",

    "12m_no_overlap_rich",
    "24m_no_overlap_rich",
    "36m_no_overlap_rich",

]


results_df["model_order"] = (

    results_df["model"]

    .map(
        {
            model: i
            for i, model in enumerate(
                model_order
            )
        }
    )

)


results_df = (

    results_df

    .sort_values(
        "model_order"
    )

    .drop(
        columns="model_order"
    )

    .reset_index(
        drop=True
    )

)


# =============================================================================
# 14. DISPLAY
# =============================================================================

display(

    results_df[

        [

            "model",

            "control_set",

            "outcome",

            "beta_raw_ai",

            "se_raw_ai",

            "p_raw_ai",

            "ai_resid_sd",

            "effect_pp_1sd",

            "se_pp_1sd",

            "ci_low_pp_1sd",

            "ci_high_pp_1sd",

            "pct_of_mean_1sd",

            "mean_outcome",

            "n_obs",

            "n_authors",

            "n_controls",

            "fwl_iterations",

            "fwl_final_change",

        ]

    ]

)


# =============================================================================
# 15. SAVE
# =============================================================================

results_df.to_csv(

    "m4_retention_4x3_common_author_panel_results.csv",

    index=False

)


print(
    "\nSaved:"
)

print(
    "  m4_retention_4x3_common_author_panel_results.csv"
)


# =============================================================================
# 16. AUTHOR-COHORT QC
# =============================================================================

print(
    "\nCOMMON AUTHOR PANEL QC"
)

print(
    "Authors in common cohort:",
    f"{len(valid_authors):,}"
)


print(
    "Papers in common cohort:",
    f"{len(df):,}"
)


print(
    "\nAuthor counts by model:"
)


display(

    results_df[

        [

            "model",

            "n_obs",

            "n_authors"

        ]

    ]

)

### Main figure 2 visualization

In [ ]:
# =============================================================================
# FIGURE 3 — DOWNSTREAM TRANSMISSION OF PRIOR KNOWLEDGE
#
# FINAL SIX-PANEL VERSION
#
# PANEL A
#   4 measurement categories × 3 citation windows
#
#       All eligible papers
#           12m
#           24m
#           36m
#
#       Papers cited downstream
#           12m
#           24m
#           36m
#
#       No-overlapping authorship
#           12m
#           24m
#           36m
#
#       No-overlap + rich controls
#           12m
#           24m
#           36m
#
# PANEL B
#   Direct reference reuse: regression robustness
#
# PANEL C
#   Sensitivity to rich reference-base controls
#
# PANEL D
#   Direct reference reuse across AI-intensity bins
#
# PANEL E
#   Extensive margin: whether any focal reference is reused
#
# PANEL F
#   Zero-AI versus high-AI contrast
#
# =============================================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# =============================================================================
# 1. LOAD CSV RESULTS
# =============================================================================

horizon = pd.read_csv(
    "m4_retention_4x3_common_author_panel_results.csv"
)

direct = pd.read_csv(
    "m4_direct_reuse_fwl_results.csv"
)

ret24 = pd.read_csv(
    "m4_24m_fwl_robustness_results.csv"
)


# =============================================================================
# 2. CLEAN MODEL NAMES
# =============================================================================

for data in [
    horizon,
    direct,
    ret24,
]:

    data["model"] = (
        data["model"]
        .astype(str)
    )


# =============================================================================
# 3. HELPER
# =============================================================================

def get_row(
    data,
    model
):

    x = data.loc[
        data["model"].eq(model)
    ]

    if x.empty:

        raise ValueError(
            f"Model not found: {model}\n"
            f"Available models:\n{data['model'].tolist()}"
        )

    return x.iloc[0]


# =============================================================================
# 4. VERIFY THE COMPLETE 4 × 3 GRID
# =============================================================================

expected_models = [

    "12m_all_eligible_compact",
    "24m_all_eligible_compact",
    "36m_all_eligible_compact",

    "12m_cited_downstream_compact",
    "24m_cited_downstream_compact",
    "36m_cited_downstream_compact",

    "12m_no_overlap_compact",
    "24m_no_overlap_compact",
    "36m_no_overlap_compact",

    "12m_no_overlap_rich",
    "24m_no_overlap_rich",
    "36m_no_overlap_rich",

]


missing_models = [

    m

    for m in expected_models

    if m not in horizon["model"].tolist()

]


if missing_models:

    raise ValueError(
        "The horizon CSV is missing models:\n"
        + "\n".join(
            missing_models
        )
    )


# =============================================================================
# 5. EXTRACT DIRECT-REUSE ROBUSTNESS RESULTS
# =============================================================================

direct_all = get_row(
    direct,
    "A_pair_weighted_all"
)

direct_external = get_row(
    direct,
    "B_pair_weighted_external"
)

direct_unweighted = get_row(
    direct,
    "C_unweighted_all"
)

direct_external_rich = get_row(
    direct,
    "D_pair_weighted_external_rich"
)


ret24_external = get_row(
    ret24,
    "C_24m_external_compact"
)

ret24_external_rich = get_row(
    ret24,
    "D_24m_external_rich"
)


# =============================================================================
# 6. DESCRIPTIVE AI-GRADIENT DATA
# =============================================================================

dose_query = """
WITH binned AS (

    SELECT

        CASE

            WHEN focal_ai_score = 0
                THEN '0'

            WHEN focal_ai_score < 0.10
                THEN '0.01-0.09'

            WHEN focal_ai_score < 0.20
                THEN '0.10-0.19'

            WHEN focal_ai_score < 0.40
                THEN '0.20-0.39'

            WHEN focal_ai_score < 0.60
                THEN '0.40-0.59'

            WHEN focal_ai_score < 0.80
                THEN '0.60-0.79'

            ELSE '0.80-1.00'

        END AS ai_bin,

        focal_ai_score,

        direct_reference_reuse_rate,

        n_focal_references_reused,

        any_shared_author

    FROM
        `aiwriting-488923.redo.m4_direct_reuse_pair_24m`
)


SELECT

    ai_bin,

    MIN(
        focal_ai_score
    ) AS bin_min_ai,

    AVG(
        focal_ai_score
    ) AS mean_ai,

    COUNT(*) AS n_pairs,


    AVG(
        direct_reference_reuse_rate
    ) AS mean_reuse,


    AVG(
        IF(
            NOT any_shared_author,
            direct_reference_reuse_rate,
            NULL
        )
    ) AS mean_no_overlap_reuse,


    SAFE_DIVIDE(

        COUNTIF(
            n_focal_references_reused > 0
        ),

        COUNT(*)

    ) AS any_reuse,


    SAFE_DIVIDE(

        COUNTIF(

            NOT any_shared_author

            AND

            n_focal_references_reused > 0

        ),

        COUNTIF(
            NOT any_shared_author
        )

    ) AS no_overlap_any_reuse

FROM binned

GROUP BY
    ai_bin

ORDER BY
    bin_min_ai
"""


dose = client.query(
    dose_query
).to_dataframe()


# Convert proportions to percentages
for col in [

    "mean_reuse",

    "mean_no_overlap_reuse",

    "any_reuse",

    "no_overlap_any_reuse",

]:

    dose[col] = (
        dose[col]
        *
        100
    )


# =============================================================================
# 7. BIN LABELS
# =============================================================================

bin_labels = [

    "0",

    "0.01–0.09",

    "0.10–0.19",

    "0.20–0.39",

    "0.40–0.59",

    "0.60–0.79",

    "0.80–1.00",

]


# =============================================================================
# 8. COLOR PALETTE
#
# Category is encoded by color.
#
# Horizon is encoded by vertical position + marker fill.
#
# 24m = filled marker
# 12m / 36m = open marker
#
# Use the exact same four category colors as Figures 1–2 here if your
# existing plotting code has named palette objects.
# =============================================================================

COLOR_ALL = "#4C78A8"

COLOR_CITED = "#F58518"

COLOR_NO_OVERLAP = "#54A24B"

COLOR_RICH = "#B279A2"


# =============================================================================
# 9. GLOBAL STYLE
# =============================================================================

plt.rcParams.update({

    "font.family": "sans-serif",

    "font.size": 9.2,

    "axes.titlesize": 12.1,

    "axes.labelsize": 9.7,

    "xtick.labelsize": 8.3,

    "ytick.labelsize": 8.5,

    "axes.spines.top": False,

    "axes.spines.right": False,

    "axes.linewidth": 0.8,

    "xtick.major.width": 0.7,

    "ytick.major.width": 0.7,

})


# =============================================================================
# 10. FIGURE
# =============================================================================

fig, axes = plt.subplots(

    3,

    2,

    figsize=(15.2, 16.4)

)


# =============================================================================
# PANEL A
# MAIN KNOWLEDGE RETENTION — 4 × 3 HORIZON-CLUSTERED FOREST PLOT
# =============================================================================

ax = axes[0, 0]


# -----------------------------------------------------------------------------
# CATEGORY DEFINITIONS
# -----------------------------------------------------------------------------

categories = [

    (
        "All eligible papers",
        COLOR_ALL,
        [
            "12m_all_eligible_compact",
            "24m_all_eligible_compact",
            "36m_all_eligible_compact",
        ]
    ),

    (
        "Papers cited downstream",
        COLOR_CITED,
        [
            "12m_cited_downstream_compact",
            "24m_cited_downstream_compact",
            "36m_cited_downstream_compact",
        ]
    ),

    (
        "No-overlapping authorship",
        COLOR_NO_OVERLAP,
        [
            "12m_no_overlap_compact",
            "24m_no_overlap_compact",
            "36m_no_overlap_compact",
        ]
    ),

    (
        "No-overlapping authorship\n+ rich controls",
        COLOR_RICH,
        [
            "12m_no_overlap_rich",
            "24m_no_overlap_rich",
            "36m_no_overlap_rich",
        ]
    ),

]


# -----------------------------------------------------------------------------
# CLUSTER POSITIONS
# -----------------------------------------------------------------------------

cluster_centers = np.array([
    0.0,
    1.55,
    3.10,
    4.65,
])


horizon_offset = 0.26


y12 = cluster_centers - horizon_offset
y24 = cluster_centers
y36 = cluster_centers + horizon_offset


# -----------------------------------------------------------------------------
# PLOT CATEGORIES
# -----------------------------------------------------------------------------

for cat_idx, (
    category,
    color,
    model_names
) in enumerate(categories):

    rows = [

        get_row(
            horizon,
            model_name
        )

        for model_name in model_names

    ]


    ys = [

        y12[cat_idx],
        y24[cat_idx],
        y36[cat_idx],

    ]


    # -------------------------------------------------------------------------
    # CONNECT THE THREE HORIZONS
    # -------------------------------------------------------------------------

    ax.plot(

        [
            row["effect_pp_1sd"]
            for row in rows
        ],

        ys,

        color=color,

        linewidth=1.35,

        alpha=0.55,

        zorder=1

    )


    # -------------------------------------------------------------------------
    # ESTIMATES + 95% CI
    # -------------------------------------------------------------------------

    for horizon_idx, (
        row,
        y0
    ) in enumerate(
        zip(
            rows,
            ys
        )
    ):

        effect = row["effect_pp_1sd"]

        ci_low = row["ci_low_pp_1sd"]

        ci_high = row["ci_high_pp_1sd"]


        is_primary = (
            horizon_idx == 1
        )


        # CI line
        ax.hlines(

            y0,

            ci_low,

            ci_high,

            color=color,

            linewidth=1.7,

            zorder=2

        )


        # CI caps
        ax.vlines(

            ci_low,

            y0 - 0.055,

            y0 + 0.055,

            color=color,

            linewidth=1.0,

            zorder=2

        )


        ax.vlines(

            ci_high,

            y0 - 0.055,

            y0 + 0.055,

            color=color,

            linewidth=1.0,

            zorder=2

        )


        # Estimate
        ax.scatter(

            effect,

            y0,

            s=66 if is_primary else 50,

            marker="o",

            facecolor=(
                color
                if is_primary
                else "white"
            ),

            edgecolor=color,

            linewidth=1.35,

            zorder=4

        )


        # Value
        ax.annotate(

            f"{effect:.3f}",

            xy=(effect, y0),

            xytext=(

                0,

                7

            ),

            textcoords="offset points",

            ha="center",

            va="bottom",

            fontsize=7.3,

            color=color,

            fontweight=(
                "bold"
                if is_primary
                else "normal"
            )

        )


# -----------------------------------------------------------------------------
# HORIZON LABELS
# -----------------------------------------------------------------------------

for i in range(
    len(cluster_centers)
):

    ax.text(

        -0.57,

        y12[i],

        "12m",

        ha="right",

        va="center",

        fontsize=7.5

    )


    ax.text(

        -0.57,

        y24[i],

        "24m",

        ha="right",

        va="center",

        fontsize=7.7,

        fontweight="bold"

    )


    ax.text(

        -0.57,

        y36[i],

        "36m",

        ha="right",

        va="center",

        fontsize=7.5

    )


# -----------------------------------------------------------------------------
# CATEGORY LABELS
# -----------------------------------------------------------------------------

category_x = -0.72


for i, (
    category,
    color,
    _
) in enumerate(categories):

    ax.text(

        category_x,

        cluster_centers[i],

        category,

        ha="right",

        va="center",

        fontsize=8.0,

        color=color,

        fontweight="bold",

        linespacing=1.15

    )


# -----------------------------------------------------------------------------
# SEPARATORS
# -----------------------------------------------------------------------------

for i in range(
    len(cluster_centers) - 1
):

    separator = (

        cluster_centers[i]
        +
        cluster_centers[i + 1]

    ) / 2


    ax.axhline(

        separator,

        color="0.88",

        linewidth=0.7,

        zorder=0

    )


# -----------------------------------------------------------------------------
# ZERO LINE
# -----------------------------------------------------------------------------

ax.axvline(

    0,

    color="black",

    linestyle=":",

    linewidth=1.0,

    alpha=0.65,

    zorder=0

 )


# -----------------------------------------------------------------------------
# AXES
# -----------------------------------------------------------------------------

ax.set_yticks([])

ax.set_xlim(
    -0.56,
    0.04
)

ax.set_ylim(
    -0.62,
    5.27
)

ax.invert_yaxis()


ax.set_xlabel(

    "Change in retained references per 1 SD AI\n"
    "(percentage points)"

)


# -----------------------------------------------------------------------------
# TITLE
# -----------------------------------------------------------------------------

ax.set_title(

    "The negative association persists across\n"
    "citation windows and measurement definitions",

    loc="left",

    pad=13

)


ax.text(

    -0.095,

    1.035,

    "a",

    transform=ax.transAxes,

    fontsize=17,

    fontweight="bold"

)


# -----------------------------------------------------------------------------
# LEGENDS — UPPER RIGHT
#
# Category legend is compact and placed above the estimates.
# Horizon key is immediately underneath.
# -----------------------------------------------------------------------------

category_handles = [

    Line2D(
        [0],
        [0],
        marker="o",
        color=COLOR_ALL,
        markerfacecolor=COLOR_ALL,
        markersize=5,
        linewidth=0,
        label="All eligible"
    ),

    Line2D(
        [0],
        [0],
        marker="o",
        color=COLOR_CITED,
        markerfacecolor=COLOR_CITED,
        markersize=5,
        linewidth=0,
        label="Cited downstream"
    ),

    Line2D(
        [0],
        [0],
        marker="o",
        color=COLOR_NO_OVERLAP,
        markerfacecolor=COLOR_NO_OVERLAP,
        markersize=5,
        linewidth=0,
        label="No-overlapping authorship"
    ),

    Line2D(
        [0],
        [0],
        marker="o",
        color=COLOR_RICH,
        markerfacecolor=COLOR_RICH,
        markersize=5,
        linewidth=0,
        label="No-overlap + rich controls"
    ),

]


horizon_handles = [

    Line2D(
        [0],
        [0],
        marker="o",
        color="black",
        markerfacecolor="white",
        markersize=4.8,
        linewidth=0,
        label="12m / 36m"
    ),

    Line2D(
        [0],
        [0],
        marker="o",
        color="black",
        markerfacecolor="black",
        markersize=5.3,
        linewidth=0,
        label="24m"
    ),

]


legend_categories = ax.legend(

    handles=category_handles,

    frameon=False,

    fontsize=6.7,

    loc="upper right",

    bbox_to_anchor=(1.05, 0.995),

    borderaxespad=0,

    handletextpad=0.35,

    labelspacing=0.25

)


ax.add_artist(
    legend_categories
)


ax.legend(

    handles=horizon_handles,

    frameon=False,

    fontsize=6.8,

    loc="upper right",

    bbox_to_anchor=(1.05, 0.72),

    borderaxespad=0,

    handletextpad=0.35,

    labelspacing=0.25

)


ax.grid(

    axis="x",

    alpha=0.18,

    linewidth=0.7

)


# =============================================================================
# PANEL B
# DIRECT REFERENCE-REUSE REGRESSION RESULTS
# =============================================================================

ax = axes[0, 1]


b_rows = [

    (
        "All citing papers",
        direct_all
    ),

    (
        "No-overlapping\nauthorship",
        direct_external
    ),

    (
        "All citing papers\n(unweighted)",
        direct_unweighted
    ),

    (
        "No-overlapping authorship\n+ rich controls",
        direct_external_rich
    ),

]


yb = np.arange(
    len(b_rows)
)


for i, (
    label,
    row
) in enumerate(b_rows):

    ax.hlines(

        i,

        row["ci_low_pp_1sd"],

        row["ci_high_pp_1sd"],

        linewidth=2.0,

        color="black",

        zorder=2

    )


    ax.scatter(

        row["effect_pp_1sd"],

        i,

        s=57,

        color="black",

        zorder=4

    )


    ax.annotate(

        f"{row['effect_pp_1sd']:.3f}",

        xy=(

            row["effect_pp_1sd"],

            i

        ),

        xytext=(

            0,

            8

        ),

        textcoords="offset points",

        ha="center",

        va="bottom",

        fontsize=7.9

    )


ax.axvline(

    0,

    color="black",

    linestyle=":",

    linewidth=1.0,

    alpha=0.65

)


ax.set_yticks(yb)

ax.set_yticklabels(

    [
        x[0]
        for x in b_rows
    ]

)

ax.invert_yaxis()


ax.set_xlim(
    -0.17,
    0.03
)


ax.set_xlabel(

    "Change in direct reference reuse per 1 SD AI\n"
    "(percentage points)"

)


ax.set_title(

    "Citing papers carry forward less of the\n"
    "focal paper's references",

    loc="left",

    pad=13

)


ax.text(

    -0.095,

    1.035,

    "b",

    transform=ax.transAxes,

    fontsize=17,

    fontweight="bold"

)


# -----------------------------------------------------------------------------
# ANNOTATION — UPPER RIGHT
# -----------------------------------------------------------------------------

ax.text(

    0.985,

    0.965,

    "24-month estimates \n 95% confidence intervals",

    transform=ax.transAxes,

    ha="right",

    va="top",

    fontsize=7.5

)


ax.grid(

    axis="x",

    alpha=0.18,

    linewidth=0.7

)


# =============================================================================
# PANEL C
# PAIRED SENSITIVITY TRAJECTORY
# =============================================================================

ax = axes[1, 0]


outcome_labels = [

    "Knowledge retention",

    "Direct reference reuse",

]


compact = np.array([

    ret24_external["effect_pp_1sd"],

    direct_external["effect_pp_1sd"],

])


rich = np.array([

    ret24_external_rich["effect_pp_1sd"],

    direct_external_rich["effect_pp_1sd"],

])


compact_low = np.array([

    ret24_external["ci_low_pp_1sd"],

    direct_external["ci_low_pp_1sd"],

])


compact_high = np.array([

    ret24_external["ci_high_pp_1sd"],

    direct_external["ci_high_pp_1sd"],

])


rich_low = np.array([

    ret24_external_rich["ci_low_pp_1sd"],

    direct_external_rich["ci_low_pp_1sd"],

])


rich_high = np.array([

    ret24_external_rich["ci_high_pp_1sd"],

    direct_external_rich["ci_high_pp_1sd"],

])


# -----------------------------------------------------------------------------
# ROW POSITIONS
#
# Slightly compressed vertically, with more unused horizontal space for
# annotation and the paired estimates.
# -----------------------------------------------------------------------------

yc = np.array([
    0.30,
    0.72
])


# -----------------------------------------------------------------------------
# TUNING PARAMETERS
# -----------------------------------------------------------------------------

COMPACT_DX = -6
COMPACT_DY = 9

RICH_DX = 6
RICH_DY = 9

ATTEN_DY = -17


# -----------------------------------------------------------------------------
# PLOT
# -----------------------------------------------------------------------------

for i in range(
    len(outcome_labels)
):

    y0 = yc[i]


    # -------------------------------------------------------------------------
    # Compact controls
    # -------------------------------------------------------------------------

    ax.errorbar(

        compact[i],

        y0,

        xerr=np.array([

            [
                compact[i]
                -
                compact_low[i]
            ],

            [
                compact_high[i]
                -
                compact[i]
            ]

        ]),

        fmt="o",

        color=COLOR_CITED,

        markerfacecolor=COLOR_CITED,

        markeredgecolor=COLOR_CITED,

        markersize=6.3,

        capsize=3,

        linewidth=1.5,

        zorder=4

    )


    # -------------------------------------------------------------------------
    # Rich controls
    # -------------------------------------------------------------------------

    ax.errorbar(

        rich[i],

        y0,

        xerr=np.array([

            [
                rich[i]
                -
                rich_low[i]
            ],

            [
                rich_high[i]
                -
                rich[i]
            ]

        ]),

        fmt="s",

        color=COLOR_RICH,

        markerfacecolor="white",

        markeredgecolor=COLOR_RICH,

        markersize=6.0,

        capsize=3,

        linewidth=1.5,

        zorder=4

    )


    # -------------------------------------------------------------------------
    # Connector
    # -------------------------------------------------------------------------

    ax.plot(

        [

            compact[i],

            rich[i]

        ],

        [

            y0,

            y0

        ],

        color="0.45",

        linewidth=1.25,

        alpha=0.65,

        zorder=2

    )


    # -------------------------------------------------------------------------
    # Compact label
    # -------------------------------------------------------------------------

    ax.annotate(

        f"{compact[i]:.3f}",

        xy=(

            compact[i],

            y0

        ),

        xytext=(

            COMPACT_DX,

            COMPACT_DY

        ),

        textcoords="offset points",

        ha="right",

        va="bottom",

        fontsize=7.8,

        color=COLOR_CITED

    )


    # -------------------------------------------------------------------------
    # Rich label
    # -------------------------------------------------------------------------

    ax.annotate(

        f"{rich[i]:.3f}",

        xy=(

            rich[i],

            y0

        ),

        xytext=(

            RICH_DX,

            RICH_DY

        ),

        textcoords="offset points",

        ha="left",

        va="bottom",

        fontsize=7.8,

        color=COLOR_RICH

    )


    # -------------------------------------------------------------------------
    # ATTENUATION
    # -------------------------------------------------------------------------

    attenuation = (

        1

        -

        abs(rich[i])
        /
        abs(compact[i])

    ) * 100


    midpoint = (

        compact[i]
        +
        rich[i]

    ) / 2


    ax.annotate(

        f"{attenuation:.1f}% attenuation",

        xy=(

            midpoint,

            y0

        ),

        xytext=(

            0,

            ATTEN_DY

        ),

        textcoords="offset points",

        ha="center",

        va="top",

        fontsize=7.4,

        color="0.35"

    )


# -----------------------------------------------------------------------------
# ZERO LINE
# -----------------------------------------------------------------------------

ax.axvline(

    0,

    color="black",

    linestyle=":",

    linewidth=1.0,

    alpha=0.65

)


# -----------------------------------------------------------------------------
# Y AXIS
# -----------------------------------------------------------------------------

ax.set_yticks(
    yc
)

ax.set_yticklabels(
    outcome_labels
)

ax.invert_yaxis()


# -----------------------------------------------------------------------------
# IMPORTANT:
# Give the left side more room for the outcome labels but center the
# estimate region visually within the plotting area.
# -----------------------------------------------------------------------------

ax.set_xlim(

    -0.30,

    0.025

)


ax.set_xlabel(

    "Change per 1 SD AI (percentage points)"

)


ax.set_title(

    "Detailed reference controls barely change\nthe estimated associations",

    loc="left",

    pad=13

)


ax.text(

    -0.095,

    1.035,

    "c",

    transform=ax.transAxes,

    fontsize=17,

    fontweight="bold"

)

ax.set_yticks(yc)

ax.set_yticklabels(
    outcome_labels
)

ax.invert_yaxis()

ax.set_ylim(
    0.95,
    0.05
)
# -----------------------------------------------------------------------------
# COMPACT / RICH KEY — UPPER RIGHT
# -----------------------------------------------------------------------------

ax.text(

    0.985,

    0.965,

    "● Compact controls   \n□ Rich reference controls",

    transform=ax.transAxes,

    ha="right",

    va="top",

    fontsize=7.4

)


ax.grid(

    axis="x",

    alpha=0.18,

    linewidth=0.7

)

# =============================================================================
# PANEL D
# DIRECT REFERENCE REUSE ACROSS AI-INTENSITY BINS
# =============================================================================

ax = axes[1, 1]


x = np.arange(
    len(dose)
)


ax.plot(

    x,

    dose["mean_reuse"],

    marker="o",

    color=COLOR_CITED,

    linewidth=2.0,

    markersize=5.5,

    label="All citing papers"

)


ax.plot(

    x,

    dose["mean_no_overlap_reuse"],

    marker="s",

    color=COLOR_NO_OVERLAP,

    markerfacecolor="white",

    markeredgecolor=COLOR_NO_OVERLAP,

    linestyle="--",

    linewidth=1.8,

    markersize=5.3,

    label="No-overlapping authorship"

)


last = dose.iloc[-1]


ax.annotate(

    f"{last['mean_reuse']:.1f}%",

    (
        x[-1],
        last["mean_reuse"]
    ),

    xytext=(7, 4),

    textcoords="offset points",

    fontsize=8.2,

    color=COLOR_CITED

)


ax.annotate(

    f"{last['mean_no_overlap_reuse']:.1f}%",

    (
        x[-1],
        last["mean_no_overlap_reuse"]
    ),

    xytext=(7, -13),

    textcoords="offset points",

    fontsize=8.2,

    color=COLOR_NO_OVERLAP

)


ax.set_xticks(x)

ax.set_xticklabels(
    [
        "0",
        "0.01–\n0.09",
        "0.10–\n0.19",
        "0.20–\n0.39",
        "0.40–\n0.59",
        "0.60–\n0.79",
        "0.80–\n1.00",
    ]
)


ax.set_xlabel(
    "AI-associated writing intensity"
)


ax.set_ylabel(

    "Focal references also cited by\n"
    "the downstream paper (%)"

)


ax.set_title(

    "Direct reference reuse declines steadily\n"
    "across the AI-intensity gradient",

    loc="left",

    pad=13

)


ax.text(

    -0.095,

    1.035,

    "d",

    transform=ax.transAxes,

    fontsize=17,

    fontweight="bold"

)


ax.legend(

    frameon=False,

    fontsize=7.7,

    loc="upper right"

)


ax.grid(

    axis="y",

    alpha=0.18,

    linewidth=0.7

)


# =============================================================================
# PANEL E
# EXTENSIVE MARGIN
# =============================================================================

ax = axes[2, 0]


ax.plot(

    x,

    dose["any_reuse"],

    marker="o",

    color=COLOR_CITED,

    linewidth=2.0,

    markersize=5.5,

    label="All citing papers"

)


ax.plot(

    x,

    dose["no_overlap_any_reuse"],

    marker="s",

    color=COLOR_NO_OVERLAP,

    markerfacecolor="white",

    markeredgecolor=COLOR_NO_OVERLAP,

    linestyle="--",

    linewidth=1.8,

    markersize=5.3,

    label="No-overlapping authorship"

)


last = dose.iloc[-1]


ax.annotate(

    f"{last['any_reuse']:.1f}%",

    (
        x[-1],
        last["any_reuse"]
    ),

    xytext=(7, 4),

    textcoords="offset points",

    fontsize=8.2,

    color=COLOR_CITED

)


ax.annotate(

    f"{last['no_overlap_any_reuse']:.1f}%",

    (
        x[-1],
        last["no_overlap_any_reuse"]
    ),

    xytext=(7, -13),

    textcoords="offset points",

    fontsize=8.2,

    color=COLOR_NO_OVERLAP

)


ax.set_xticks(x)

ax.set_xticklabels(
    [
        "0",
        "0.01–\n0.09",
        "0.10–\n0.19",
        "0.20–\n0.39",
        "0.40–\n0.59",
        "0.60–\n0.79",
        "0.80–\n1.00",
    ]
)


ax.set_xlabel(
    "AI-associated writing intensity"
)


ax.set_ylabel(

    "Citing papers carrying forward\n"
    "at least one focal reference (%)"

)


ax.set_title(

    "The decline is also visible in whether any\n"
    "prior reference is carried forward",

    loc="left",

    pad=13

)


ax.text(

    -0.095,

    1.035,

    "e",

    transform=ax.transAxes,

    fontsize=17,

    fontweight="bold"

)


ax.legend(

    frameon=False,

    fontsize=7.6,

    loc="upper right"

)


ax.grid(

    axis="y",

    alpha=0.18,

    linewidth=0.7

)


# =============================================================================
# PANEL F
# ZERO-AI VS HIGH-AI
# =============================================================================

ax = axes[2, 1]


low = dose.iloc[0]

high = dose.iloc[-1]


categories_f = [

    "Direct\nreference reuse",

    "No-overlapping\nauthorship"

]


low_values = np.array([

    low["mean_reuse"],

    low["mean_no_overlap_reuse"]

])


high_values = np.array([

    high["mean_reuse"],

    high["mean_no_overlap_reuse"]

])


xf = np.arange(
    len(categories_f)
)


width = 0.34


bars_low = ax.bar(

    xf - width / 2,

    low_values,

    width,

    color=COLOR_ALL,

    label="AI = 0"

)


bars_high = ax.bar(

    xf + width / 2,

    high_values,

    width,

    color="white",

    edgecolor=COLOR_CITED,

    linewidth=1.2,

    label="AI = 0.80–1.00"

)


# Values
for bars in [
    bars_low,
    bars_high
]:

    for bar in bars:

        h = bar.get_height()


        ax.text(

            bar.get_x()
            +
            bar.get_width() / 2,

            h + 0.08,

            f"{h:.1f}%",

            ha="center",

            va="bottom",

            fontsize=8.0

        )


# Relative difference
for i in range(
    len(categories_f)
):

    reduction = (

        1

        -

        high_values[i]
        /
        low_values[i]

    ) * 100


    ax.text(

        xf[i],

        (
            low_values[i]
            +
            high_values[i]
        ) / 2,

        f"−{reduction:.0f}%",

        ha="center",

        va="center",

        fontsize=8.4,

        fontweight="bold"

    )


ax.set_xticks(xf)

ax.set_xticklabels(
    categories_f
)


ax.set_ylabel(
    "Focal references also cited (%)"
)


ax.set_title(

    "Downstream reference reuse is substantially lower\n"
    "at high AI intensity",

    loc="left",

    pad=13

)


ax.text(

    -0.095,

    1.035,

    "f",

    transform=ax.transAxes,

    fontsize=17,

    fontweight="bold"

)


ax.legend(

    frameon=False,

    fontsize=7.7,

    loc="upper right"

)


ax.grid(

    axis="y",

    alpha=0.18,

    linewidth=0.7

)


# =============================================================================
# 11. GLOBAL TITLE
# =============================================================================

fig.suptitle(

    "AI-associated writing is linked to weaker downstream transmission of prior knowledge",

    fontsize=16,

    y=0.993

)


# =============================================================================
# 12. FOOTNOTE
# =============================================================================

fig.text(

    0.5,

    0.014,

    "Panels a–c show adjusted estimates per 1 SD increase in residualized "
    "AI-writing intensity with 95% confidence intervals. All adjusted models "
    "include focal-author, publication-month and subfield fixed effects.\n "
    "No-overlapping authorship excludes descendants sharing any focal-paper "
    "author. Panels d–f show unadjusted 24-month descriptive relationships.",

    ha="center",

    va="bottom",

    fontsize=7.8

)


# =============================================================================
# 13. SPACING
# =============================================================================

fig.subplots_adjust(
    left=0.16, right=0.975, top=0.91, bottom=0.095,
    wspace=0.40, hspace=0.50
)


# =============================================================================
# 14. SAVE
# =============================================================================

fig.savefig(

    "figure3_knowledge_retention_final_v11.png",

    dpi=600,

    bbox_inches="tight"

)


fig.savefig(

    "figure3_knowledge_retention_final_v11.pdf",

    bbox_inches="tight"

)


plt.show()

## Manuscript Figure 3 — Scientific collaboration and expertise

Final static collaborator and expertise estimates, pretreatment author-level moment construction, final analysis-sample entropy balancing, change-from-baseline estimates, and main figure.

### Static collaboration (main figure 3 panel a)

In [ ]:
# =============================================================================
# FIGURE 4 — STATIC COLLABORATION / EXPERTISE ANALYSIS
#
# Corrected to use the actual schema of:
#     redo.fig4_collaboration_paper_final
#
# No new OpenAlex construction.
# No visualization.
#
# Primary models:
#   1. Returning collaborators
#   2. New collaborators
#   3. Share returning collaborators
#   4. Cross-subfield collaborators
#   5. Cross-field collaborators
#   6. Mean field-level expertise overlap
#
# FE:
#   focal first-author FE
#   accepted-month FE
#   focal subfield FE
#
# SE:
#   clustered by focal first author
# =============================================================================

import gc
import numpy as np
import pandas as pd
import statsmodels.api as sm

from google.cloud import bigquery


# =============================================================================
# 1. BIGQUERY CLIENT
# =============================================================================

client = bigquery.Client(
    project="aiwriting-488923"
)


# =============================================================================
# 2. LOAD PAPER-LEVEL FIGURE 4 TABLE
# =============================================================================
#
# IMPORTANT:
# The actual focal subfield field is:
#
#     subfield_id
#
# not focal_subfield_id.
#
# =============================================================================

query = """
SELECT

    focal_openalex_id,

    focal_first_author_id,

    accepted_date,

    accepted_year,

    accepted_month,

    ai_score,

    subfield_id,

    field_id,

    domain_id,

    source_id,

    n_coauthors,

    n_returning_collaborators,

    n_new_collaborators,

    n_returning_2plus,

    n_returning_recent_3y,

    n_returning_recent_5y,

    share_returning_collaborators,

    mean_prior_joint_papers,

    max_prior_joint_papers,

    mean_collaborator_topic_breadth,

    mean_collaborator_subfield_breadth,

    mean_collaborator_field_breadth,

    mean_collaborator_prior_papers,

    share_matching_current_subfield,

    share_matching_current_field,

    share_cross_subfield_collaborators,

    share_cross_field_collaborators,

    solo_authorship,

    ln_returning_collaborators,

    ln_new_collaborators,

    ln_coauthors

FROM
    `aiwriting-488923.redo.fig4_collaboration_paper_final`

WHERE

    focal_openalex_id IS NOT NULL

    AND focal_first_author_id IS NOT NULL

    AND ai_score IS NOT NULL

    AND accepted_year IS NOT NULL

    AND accepted_month IS NOT NULL

    AND subfield_id IS NOT NULL
"""


df = client.query(
    query
).to_dataframe()


print(
    "Loaded focal-paper rows:",
    f"{len(df):,}"
)

print(
    "Unique focal papers:",
    f"{df['focal_openalex_id'].nunique():,}"
)

print(
    "Unique focal first authors:",
    f"{df['focal_first_author_id'].nunique():,}"
)


# =============================================================================
# 3. NUMERIC CLEANING
# =============================================================================

numeric_cols = [

    "accepted_year",
    "accepted_month",
    "ai_score",

    "n_coauthors",
    "n_returning_collaborators",
    "n_new_collaborators",

    "n_returning_2plus",
    "n_returning_recent_3y",
    "n_returning_recent_5y",

    "share_returning_collaborators",

    "mean_prior_joint_papers",
    "max_prior_joint_papers",

    "mean_collaborator_topic_breadth",
    "mean_collaborator_subfield_breadth",
    "mean_collaborator_field_breadth",
    "mean_collaborator_prior_papers",

    "share_matching_current_subfield",
    "share_matching_current_field",

    "share_cross_subfield_collaborators",
    "share_cross_field_collaborators",

    "solo_authorship",

    "ln_returning_collaborators",
    "ln_new_collaborators",
    "ln_coauthors",

]


for col in numeric_cols:

    df[col] = pd.to_numeric(

        df[col],

        errors="coerce"

    )


# =============================================================================
# 4. BASIC QC
# =============================================================================

print(
    "\nBASIC COLLABORATION QC"
)

print(
    "Mean coauthors:",
    round(
        df["n_coauthors"].mean(),
        4
    )
)

print(
    "Mean returning collaborators:",
    round(
        df["n_returning_collaborators"].mean(),
        4
    )
)

print(
    "Mean new collaborators:",
    round(
        df["n_new_collaborators"].mean(),
        4
    )
)

print(
    "Mean returning share:",
    round(
        df["share_returning_collaborators"].mean(),
        4
    )
)

print(
    "Mean cross-subfield share:",
    round(
        df["share_cross_subfield_collaborators"].mean(),
        4
    )
)

print(
    "Mean cross-field share:",
    round(
        df["share_cross_field_collaborators"].mean(),
        4
    )
)

print(
    "Solo-authorship rate:",
    round(
        df["solo_authorship"].mean(),
        4
    )
)


# =============================================================================
# 5. VERIFY THE COMMON AUTHOR PANEL
#
# We reconstruct the Figure 1-style cohort from the loaded focal-paper table.
#
# This is only an in-memory check. We are NOT redefining the underlying
# BigQuery table.
# =============================================================================

author_summary = (

    df

    .groupby(

        "focal_first_author_id",

        observed=True

    )

    .agg(

        n_focal_papers=(

            "focal_openalex_id",

            "size"

        ),

        ai_sd=(

            "ai_score",

            "std"

        )

    )

)


valid_authors = author_summary.index[

    (

        author_summary[
            "n_focal_papers"
        ]

        >= 2

    )

    &

    (

        author_summary[
            "ai_sd"
        ]

        > 0

    )

]


df = df.loc[

    df[
        "focal_first_author_id"
    ].isin(
        valid_authors
    )

].copy()


print(
    "\nCOMMON AUTHOR PANEL CHECK"
)

print(
    "Authors:",
    f"{len(valid_authors):,}"
)

print(
    "Focal papers:",
    f"{len(df):,}"
)


# =============================================================================
# 6. FIXED EFFECT CODES
# =============================================================================

author_code, _ = pd.factorize(

    df[
        "focal_first_author_id"
    ],

    sort=False

)


month_key = (

    df[
        "accepted_year"
    ].astype(np.int64)

    * 100

    +

    df[
        "accepted_month"
    ].astype(np.int64)

)


month_code, _ = pd.factorize(

    month_key,

    sort=False

)


subfield_code, _ = pd.factorize(

    df[
        "subfield_id"
    ],

    sort=False

)


author_code = author_code.astype(
    np.int64
)

month_code = month_code.astype(
    np.int64
)

subfield_code = subfield_code.astype(
    np.int64
)


# =============================================================================
# 7. FWL FUNCTIONS
# =============================================================================

def group_demean_matrix(

    X,

    groups,

    n_groups

):

    X = np.asarray(

        X,

        dtype=np.float64

    )


    out = np.empty_like(
        X
    )


    counts = np.bincount(

        groups,

        minlength=n_groups

    ).astype(
        np.float64
    )


    counts[
        counts == 0
    ] = 1.0


    for j in range(
        X.shape[1]
    ):

        sums = np.bincount(

            groups,

            weights=X[:, j],

            minlength=n_groups

        )


        means = (
            sums
            /
            counts
        )


        out[:, j] = (

            X[:, j]

            -

            means[groups]

        )


    return out


def fwl_residualize(

    X,

    group_arrays,

    group_sizes,

    tol=1e-8,

    max_iter=100

):

    R = np.asarray(

        X,

        dtype=np.float64

    ).copy()


    for iteration in range(

        1,

        max_iter + 1

    ):

        old = R.copy()


        for groups, n_groups in zip(

            group_arrays,

            group_sizes

        ):

            R = group_demean_matrix(

                R,

                groups,

                n_groups

            )


        change = np.max(

            np.abs(

                R - old

            )

        )


        if change < tol:

            return (

                R,

                iteration,

                change

            )


    return (

        R,

        max_iter,

        change

    )


# =============================================================================
# 8. BUILD ANALYSIS VARIABLES
# =============================================================================

# -----------------------------------------------------------------------------
# Log collaborator counts
# -----------------------------------------------------------------------------

df["ln_returning"] = np.log1p(

    df[
        "n_returning_collaborators"
    ]

)


df["ln_new"] = np.log1p(

    df[
        "n_new_collaborators"
    ]

)


# -----------------------------------------------------------------------------
# Outcomes
# -----------------------------------------------------------------------------

outcomes = [

    "ln_returning",

    "ln_new",

    "share_returning_collaborators",

    "share_cross_subfield_collaborators",

    "share_cross_field_collaborators",

]


# =============================================================================
# 9. FOCAL-PAPER FIELD JACCARD
#
# Pull only the collaborator-level field overlap required to construct the
# sixth outcome.
# =============================================================================

field_overlap_query = """

SELECT

    focal_openalex_id,

    AVG(
        field_jaccard
    ) AS mean_field_jaccard,

    APPROX_QUANTILES(
        field_jaccard,
        100
    )[OFFSET(50)] AS median_field_jaccard,

    AVG(
        collaborator_only_field_share
    ) AS mean_collaborator_only_field_share

FROM
    `aiwriting-488923.redo.fig4_expertise_overlap`

WHERE

    field_jaccard IS NOT NULL

GROUP BY
    focal_openalex_id

"""


field_overlap = client.query(
    field_overlap_query
).to_dataframe()


print(
    "\nField-overlap rows:",
    f"{len(field_overlap):,}"
)


df = df.merge(

    field_overlap,

    on="focal_openalex_id",

    how="left"

)


outcomes.append(
    "mean_field_jaccard"
)


# =============================================================================
# FIGURE 4 — CORRECTED STATIC FWL ANALYSIS
#
# IMPORTANT:
# Each outcome gets:
#   1. its own complete-case sample
#   2. its own FWL residualization
#   3. its own author/month/subfield FE
#
# This prevents missing values in one outcome from contaminating the
# residualization of all other outcomes.
# =============================================================================


import gc
import numpy as np
import pandas as pd
import statsmodels.api as sm


# =============================================================================
# 1. OUTCOME DEFINITIONS
# =============================================================================

outcome_specs = [

    (
        "ln_returning",
        "n_returning_collaborators",
        "log_count"
    ),

    (
        "ln_new",
        "n_new_collaborators",
        "log_count"
    ),

    (
        "share_returning_collaborators",
        "share_returning_collaborators",
        "percentage_point"
    ),

    (
        "share_cross_subfield_collaborators",
        "share_cross_subfield_collaborators",
        "percentage_point"
    ),

    (
        "share_cross_field_collaborators",
        "share_cross_field_collaborators",
        "percentage_point"
    ),

    (
        "mean_field_jaccard",
        "mean_field_jaccard",
        "percentage_point"
    ),

]


# =============================================================================
# 2. PREPARE LOG VARIABLES
# =============================================================================

df["ln_returning"] = np.log1p(
    df["n_returning_collaborators"]
)


df["ln_new"] = np.log1p(
    df["n_new_collaborators"]
)


# =============================================================================
# 3. FWL HELPERS
# =============================================================================

def group_demean_matrix(

    X,

    groups,

    n_groups

):

    X = np.asarray(
        X,
        dtype=np.float64
    )

    out = np.empty_like(
        X
    )


    counts = np.bincount(
        groups,
        minlength=n_groups
    ).astype(
        np.float64
    )


    # This should never be zero for observed groups
    counts[
        counts == 0
    ] = 1.0


    for j in range(
        X.shape[1]
    ):

        sums = np.bincount(

            groups,

            weights=X[:, j],

            minlength=n_groups

        )


        means = (

            sums

            /

            counts

        )


        out[:, j] = (

            X[:, j]

            -

            means[groups]

        )


    return out


def fwl_residualize(

    X,

    group_arrays,

    group_sizes,

    tol=1e-8,

    max_iter=100

):

    R = np.asarray(

        X,

        dtype=np.float64

    ).copy()


    for iteration in range(

        1,

        max_iter + 1

    ):

        old = R.copy()


        for groups, n_groups in zip(

            group_arrays,

            group_sizes

        ):

            R = group_demean_matrix(

                R,

                groups,

                n_groups

            )


        change = np.max(

            np.abs(
                R - old
            )

        )


        if change < tol:

            return (

                R,

                iteration,

                change

            )


    return (

        R,

        max_iter,

        change

    )


# =============================================================================
# 4. SINGLE-OUTCOME MODEL
#
# FWL is done separately for each outcome.
# =============================================================================

def run_single_outcome(

    df,

    outcome_name,

    source_outcome,

    interpretation_type

):

    print(
        f"\nRunning {outcome_name} ..."
    )


    # -------------------------------------------------------------------------
    # Complete-case sample for THIS outcome only
    # -------------------------------------------------------------------------

    required = [

        "ai_score",

        outcome_name,

        "focal_first_author_id",

        "accepted_year",

        "accepted_month",

        "subfield_id",

    ]


    d = df.dropna(

        subset=required

    ).copy()


    # -------------------------------------------------------------------------
    # Ensure all required numeric values are finite
    # -------------------------------------------------------------------------

    finite_mask = (

        np.isfinite(
            d["ai_score"].to_numpy(
                dtype=np.float64
            )
        )

        &

        np.isfinite(
            d[outcome_name].to_numpy(
                dtype=np.float64
            )
        )

    )


    d = d.loc[
        finite_mask
    ].copy()


    print(
        "  complete-case rows:",
        f"{len(d):,}"
    )


    if len(d) == 0:

        return {

            "outcome":
                source_outcome,

            "status":
                "no_complete_cases",

        }


    # -------------------------------------------------------------------------
    # Fixed effects
    # -------------------------------------------------------------------------

    author_code, _ = pd.factorize(

        d[
            "focal_first_author_id"
        ],

        sort=False

    )


    month_key = (

        d[
            "accepted_year"
        ].astype(np.int64)

        * 100

        +

        d[
            "accepted_month"
        ].astype(np.int64)

    )


    month_code, _ = pd.factorize(

        month_key,

        sort=False

    )


    subfield_code, _ = pd.factorize(

        d[
            "subfield_id"
        ],

        sort=False

    )


    author_code = author_code.astype(
        np.int64
    )

    month_code = month_code.astype(
        np.int64
    )

    subfield_code = subfield_code.astype(
        np.int64
    )


    # -------------------------------------------------------------------------
    # AI and outcome
    # -------------------------------------------------------------------------

    ai = d[
        "ai_score"
    ].to_numpy(
        dtype=np.float64
    )


    y = d[
        outcome_name
    ].to_numpy(
        dtype=np.float64
    )


    # -------------------------------------------------------------------------
    # FWL
    #
    # Residualize only AI + THIS outcome.
    # -------------------------------------------------------------------------

    M = np.column_stack(

        [

            ai,

            y

        ]

    )


    R, iterations, final_change = (

        fwl_residualize(

            M,

            group_arrays=[

                author_code,

                month_code,

                subfield_code

            ],

            group_sizes=[

                int(
                    author_code.max()
                ) + 1,

                int(
                    month_code.max()
                ) + 1,

                int(
                    subfield_code.max()
                ) + 1,

            ],

            tol=1e-8,

            max_iter=300

        )

    )


    ai_resid = R[
        :,
        0
    ]


    y_resid = R[
        :,
        1
    ]


    # -------------------------------------------------------------------------
    # Final finite check
    # -------------------------------------------------------------------------

    valid = (

        np.isfinite(
            ai_resid
        )

        &

        np.isfinite(
            y_resid
        )

    )


    n_valid = int(
        valid.sum()
    )


    print(
        "  FWL-valid rows:",
        f"{n_valid:,}"
    )


    if n_valid == 0:

        return {

            "outcome":
                source_outcome,

            "status":
                "no_fwl_valid_rows",

            "n_obs":
                0,

            "n_authors":
                0,

            "fwl_iterations":
                iterations,

            "fwl_final_change":
                final_change,

        }


    # -------------------------------------------------------------------------
    # Residualized variation diagnostics
    # -------------------------------------------------------------------------

    ai_valid = ai_resid[
        valid
    ]


    y_valid = y_resid[
        valid
    ]


    ai_sd = float(

        np.std(

            ai_valid,

            ddof=0

        )

    )


    y_sd = float(

        np.std(

            y_valid,

            ddof=0

        )

    )


    print(
        "  residualized AI SD:",
        f"{ai_sd:.10f}"
    )


    print(
        "  residualized outcome SD:",
        f"{y_sd:.10f}"
    )


    if (

        not np.isfinite(ai_sd)

        or

        ai_sd <= 1e-12

    ):

        return {

            "outcome":
                source_outcome,

            "status":
                "no_ai_variation",

            "ai_resid_sd":
                ai_sd,

            "outcome_resid_sd":
                y_sd,

            "n_obs":
                n_valid,

            "n_authors":
                np.unique(
                    author_code[valid]
                ).size,

            "fwl_iterations":
                iterations,

            "fwl_final_change":
                final_change,

        }


    if (

        not np.isfinite(y_sd)

        or

        y_sd <= 1e-12

    ):

        return {

            "outcome":
                source_outcome,

            "status":
                "no_outcome_variation",

            "ai_resid_sd":
                ai_sd,

            "outcome_resid_sd":
                y_sd,

            "n_obs":
                n_valid,

            "n_authors":
                np.unique(
                    author_code[valid]
                ).size,

            "fwl_iterations":
                iterations,

            "fwl_final_change":
                final_change,

        }


    # -------------------------------------------------------------------------
    # OLS
    # -------------------------------------------------------------------------

    X = np.column_stack(

        [

            np.ones(
                n_valid
            ),

            ai_valid

        ]

    )


    clusters = author_code[
        valid
    ]


    model = sm.OLS(

        y_valid,

        X

    ).fit(

        cov_type="cluster",

        cov_kwds={

            "groups":
                clusters

        }

    )


    # -------------------------------------------------------------------------
    # AI coefficient
    # -------------------------------------------------------------------------

    beta = float(
        model.params[1]
    )


    se = float(
        model.bse[1]
    )


    p = float(
        model.pvalues[1]
    )


    # -------------------------------------------------------------------------
    # Standardized AI effect
    # -------------------------------------------------------------------------

    effect_1sd = (

        beta

        *

        ai_sd

    )


    se_1sd = (

        se

        *

        ai_sd

    )


    ci_low = (

        effect_1sd

        -

        1.96

        *

        se_1sd

    )


    ci_high = (

        effect_1sd

        +

        1.96

        *

        se_1sd

    )


    # -------------------------------------------------------------------------
    # Original-scale mean
    # -------------------------------------------------------------------------

    mean_outcome = float(

        d.loc[
            valid,
            outcome_name
        ].mean()

    )


    # -------------------------------------------------------------------------
    # Interpretation
    # -------------------------------------------------------------------------

    if interpretation_type == "log_count":

        effect_interpreted = (

            np.exp(
                effect_1sd
            )

            -

            1

        ) * 100


        interpretation_unit = (
            "percent change"
        )

    else:

        effect_interpreted = (

            effect_1sd

            *

            100

        )


        interpretation_unit = (
            "percentage points"
        )


    # -------------------------------------------------------------------------
    # Return
    # -------------------------------------------------------------------------

    return {

        "outcome":
            source_outcome,

        "status":
            "estimated",

        "beta_raw_ai":
            beta,

        "se_raw_ai":
            se,

        "p_raw_ai":
            p,

        "ai_resid_sd":
            ai_sd,

        "outcome_resid_sd":
            y_sd,

        "effect_per_1sd":
            effect_1sd,

        "se_per_1sd":
            se_1sd,

        "ci_low_per_1sd":
            ci_low,

        "ci_high_per_1sd":
            ci_high,

        "effect_interpreted":
            effect_interpreted,

        "interpretation_unit":
            interpretation_unit,

        "mean_outcome":
            mean_outcome,

        "n_obs":
            n_valid,

        "n_authors":
            int(
                np.unique(
                    clusters
                ).size
            ),

        "fwl_iterations":
            iterations,

        "fwl_final_change":
            final_change,

    }


# =============================================================================
# 5. RUN ALL SIX OUTCOMES
# =============================================================================

results = []


for (

    outcome_name,

    source_outcome,

    interpretation_type

) in outcome_specs:

    result = run_single_outcome(

        df=df,

        outcome_name=outcome_name,

        source_outcome=source_outcome,

        interpretation_type=interpretation_type

    )


    results.append(
        result
    )


results_df = pd.DataFrame(
    results
)


# =============================================================================
# 6. DISPLAY
# =============================================================================

display(

    results_df

)


# More compact display
display(

    results_df.loc[

        results_df[
            "status"
        ]
        ==
        "estimated",

        [

            "outcome",

            "status",

            "beta_raw_ai",

            "se_raw_ai",

            "p_raw_ai",

            "ai_resid_sd",

            "effect_per_1sd",

            "se_per_1sd",

            "ci_low_per_1sd",

            "ci_high_per_1sd",

            "effect_interpreted",

            "interpretation_unit",

            "mean_outcome",

            "n_obs",

            "n_authors",

            "fwl_iterations",

            "fwl_final_change",

        ]

    ]

)


# =============================================================================
# 7. SAVE
# =============================================================================

results_df.to_csv(

    "fig4_static_collaboration_results.csv",

    index=False

)


print(
    "\nSaved:"
)

print(
    "  fig4_static_collaboration_results.csv"
)


# =============================================================================
# 8. CLEANUP
# =============================================================================

gc.collect()

### Within-type expertise (main figure 3 panels b–c)

In [ ]:
# =============================================================================
# FIGURE 4 — WITHIN-TYPE EXPERTISE DECOMPOSITION
#
# QUESTION
# -------
# Is the apparent increase in collaborator disciplinary distance with AI
# intensity merely a compositional consequence of fewer returning
# collaborators?
#
# We therefore estimate expertise composition separately for:
#
#   A. RETURNING collaborators
#   B. NEW collaborators
#
# For each type:
#
#   1. field-level Jaccard overlap
#   2. subfield-level Jaccard overlap
#   3. share cross-field
#   4. share cross-subfield
#
# We also keep the extensive-margin counts:
#
#   n_returning
#   n_new
#
# PRIMARY IDENTIFICATION
# ----------------------
# Common longitudinal focal-author panel:
#
#     >=2 focal papers
#     + within-author AI variation
#
# FE:
#
#     focal first-author FE
#     accepted-month FE
#     focal subfield FE
#
# SE:
#
#     clustered by focal first author
#
# IMPORTANT
# ---------
# Expertise outcomes are CONDITIONAL:
#
#     returning expertise outcome
#         is defined only if the focal paper has >=1 returning collaborator
#
#     new expertise outcome
#         is defined only if the focal paper has >=1 new collaborator
#
# We do NOT assign zero expertise distance to papers with no collaborators of
# that type.
#
# INPUT TABLES
# ------------
#   redo.fig4_focal_papers
#   redo.fig4_expertise_overlap
#
# OUTPUTS
# -------
#   fig4_within_type_expertise_results.csv
#   fig4_within_type_expertise_dose_response.csv
# =============================================================================


import gc
import numpy as np
import pandas as pd
import statsmodels.api as sm

from google.cloud import bigquery


# =============================================================================
# 1. CLIENT
# =============================================================================

client = bigquery.Client(
    project="aiwriting-488923"
)


# =============================================================================
# 2. LOAD FOCAL-PAPER INFORMATION
#
# We explicitly obtain subfield_id from fig4_focal_papers.
#
# Do NOT substitute another variable for the subfield FE.
# =============================================================================

focal_query = """
SELECT

    focal_openalex_id,

    focal_first_author_id,

    accepted_date,

    accepted_year,

    accepted_month,

    ai_score,

    subfield_id,

    field_id,

    domain_id,

    source_id

FROM
    `aiwriting-488923.redo.fig4_focal_papers`

WHERE

    focal_openalex_id IS NOT NULL

    AND focal_first_author_id IS NOT NULL

    AND accepted_date IS NOT NULL

    AND accepted_year IS NOT NULL

    AND accepted_month IS NOT NULL

    AND ai_score IS NOT NULL

    AND subfield_id IS NOT NULL
"""


focal = client.query(
    focal_query
).to_dataframe()


print(
    "Focal-paper rows loaded:",
    f"{len(focal):,}"
)

print(
    "Unique focal papers:",
    f"{focal['focal_openalex_id'].nunique():,}"
)

print(
    "Unique focal authors:",
    f"{focal['focal_first_author_id'].nunique():,}"
)


# =============================================================================
# 3. LOAD COLLABORATOR-LEVEL EXPERTISE DATA
#
# IMPORTANT:
# This table already contains:
#
#   returning_collaborator
#   field_jaccard
#   subfield_jaccard
#   collaborator_only_field_share
#   collaborator_only_subfield_share
#
# plus the focal-paper identifiers.
#
# We only pull variables needed for this analysis.
# =============================================================================

expertise_query = """
SELECT

    focal_openalex_id,

    focal_first_author_id,

    collaborator_id,

    returning_collaborator,

    field_jaccard,

    subfield_jaccard,

    topic_jaccard,

    collaborator_only_field_share,

    collaborator_only_subfield_share,

    collaborator_only_topic_share,

    collaborator_has_new_field,

    collaborator_has_new_subfield,

    collaborator_has_new_topic

FROM
    `aiwriting-488923.redo.fig4_expertise_overlap`

WHERE

    focal_openalex_id IS NOT NULL

    AND focal_first_author_id IS NOT NULL

    AND collaborator_id IS NOT NULL
"""


expertise = client.query(
    expertise_query
).to_dataframe()


print(
    "\nCollaborator-level expertise rows:",
    f"{len(expertise):,}"
)

print(
    "Focal papers:",
    f"{expertise['focal_openalex_id'].nunique():,}"
)

print(
    "Collaborators:",
    f"{expertise['collaborator_id'].nunique():,}"
)


# =============================================================================
# 4. TYPE CONVERSION
# =============================================================================

numeric_expertise_cols = [

    "field_jaccard",

    "subfield_jaccard",

    "topic_jaccard",

    "collaborator_only_field_share",

    "collaborator_only_subfield_share",

    "collaborator_only_topic_share",

]


for col in numeric_expertise_cols:

    expertise[col] = pd.to_numeric(

        expertise[col],

        errors="coerce"

    )


focal["accepted_year"] = pd.to_numeric(
    focal["accepted_year"],
    errors="coerce"
)

focal["accepted_month"] = pd.to_numeric(
    focal["accepted_month"],
    errors="coerce"
)

focal["ai_score"] = pd.to_numeric(
    focal["ai_score"],
    errors="coerce"
)


# =============================================================================
# 5. CHECK COLLABORATOR-LEVEL TYPE BALANCE
# =============================================================================

type_qc = (

    expertise

    .groupby(
        "returning_collaborator",
        observed=True
    )

    .agg(

        collaborator_rows=(
            "collaborator_id",
            "size"
        ),

        focal_papers=(
            "focal_openalex_id",
            "nunique"
        ),

        collaborators=(
            "collaborator_id",
            "nunique"
        ),

        mean_field_jaccard=(
            "field_jaccard",
            "mean"
        ),

        mean_subfield_jaccard=(
            "subfield_jaccard",
            "mean"
        ),

        mean_topic_jaccard=(
            "topic_jaccard",
            "mean"
        ),

        mean_only_field_share=(
            "collaborator_only_field_share",
            "mean"
        ),

        mean_only_subfield_share=(
            "collaborator_only_subfield_share",
            "mean"
        ),

        mean_only_topic_share=(
            "collaborator_only_topic_share",
            "mean"
        )

    )

    .reset_index()

)


print(
    "\nRETURNING VS NEW COLLABORATOR QC"
)

display(
    type_qc
)


# =============================================================================
# 6. AGGREGATE EXPERTISE TO FOCAL-PAPER × COLLABORATOR TYPE
#
# Crucially:
#
#   returning outcome = mean among returning collaborators ONLY
#   new outcome       = mean among new collaborators ONLY
#
# If a focal paper has no returning collaborators:
#
#   returning_mean_* = NaN
#
# If a focal paper has no new collaborators:
#
#   new_mean_* = NaN
#
# This preserves the conditional estimand.
# =============================================================================

expertise_paper_query = """
WITH x AS (

    SELECT

        focal_openalex_id,

        focal_first_author_id,


        /*------------------------------------------------------------------
          Returning collaborators
        ------------------------------------------------------------------*/

        COUNTIF(
            returning_collaborator
        ) AS n_returning,

        COUNTIF(
            NOT returning_collaborator
        ) AS n_new,


        AVG(
            IF(
                returning_collaborator,
                field_jaccard,
                NULL
            )
        ) AS returning_field_jaccard,

        AVG(
            IF(
                returning_collaborator,
                subfield_jaccard,
                NULL
            )
        ) AS returning_subfield_jaccard,

        AVG(
            IF(
                returning_collaborator,
                topic_jaccard,
                NULL
            )
        ) AS returning_topic_jaccard,


        AVG(
            IF(
                returning_collaborator,
                collaborator_only_field_share,
                NULL
            )
        ) AS returning_only_field_share,

        AVG(
            IF(
                returning_collaborator,
                collaborator_only_subfield_share,
                NULL
            )
        ) AS returning_only_subfield_share,

        AVG(
            IF(
                returning_collaborator,
                collaborator_only_topic_share,
                NULL
            )
        ) AS returning_only_topic_share,


        AVG(
            IF(
                returning_collaborator,
                CAST(
                    collaborator_has_new_field
                    AS INT64
                ),
                NULL
            )
        ) AS returning_share_new_field,

        AVG(
            IF(
                returning_collaborator,
                CAST(
                    collaborator_has_new_subfield
                    AS INT64
                ),
                NULL
            )
        ) AS returning_share_new_subfield,

        AVG(
            IF(
                returning_collaborator,
                CAST(
                    collaborator_has_new_topic
                    AS INT64
                ),
                NULL
            )
        ) AS returning_share_new_topic,


        /*------------------------------------------------------------------
          New collaborators
        ------------------------------------------------------------------*/

        AVG(
            IF(
                NOT returning_collaborator,
                field_jaccard,
                NULL
            )
        ) AS new_field_jaccard,

        AVG(
            IF(
                NOT returning_collaborator,
                subfield_jaccard,
                NULL
            )
        ) AS new_subfield_jaccard,

        AVG(
            IF(
                NOT returning_collaborator,
                topic_jaccard,
                NULL
            )
        ) AS new_topic_jaccard,


        AVG(
            IF(
                NOT returning_collaborator,
                collaborator_only_field_share,
                NULL
            )
        ) AS new_only_field_share,

        AVG(
            IF(
                NOT returning_collaborator,
                collaborator_only_subfield_share,
                NULL
            )
        ) AS new_only_subfield_share,

        AVG(
            IF(
                NOT returning_collaborator,
                collaborator_only_topic_share,
                NULL
            )
        ) AS new_only_topic_share,


        AVG(
            IF(
                NOT returning_collaborator,
                CAST(
                    collaborator_has_new_field
                    AS INT64
                ),
                NULL
            )
        ) AS new_share_new_field,

        AVG(
            IF(
                NOT returning_collaborator,
                CAST(
                    collaborator_has_new_subfield
                    AS INT64
                ),
                NULL
            )
        ) AS new_share_new_subfield,

        AVG(
            IF(
                NOT returning_collaborator,
                CAST(
                    collaborator_has_new_topic
                    AS INT64
                ),
                NULL
            )
        ) AS new_share_new_topic

    FROM
        `aiwriting-488923.redo.fig4_expertise_overlap`

    GROUP BY

        focal_openalex_id,

        focal_first_author_id

)

SELECT * FROM x
"""


paper_expertise = client.query(
    expertise_paper_query
).to_dataframe()


print(
    "\nFocal-paper expertise rows:",
    f"{len(paper_expertise):,}"
)


# =============================================================================
# 7. MERGE WITH FOCAL-PAPER DATA
# =============================================================================

df = focal.merge(

    paper_expertise,

    on=[

        "focal_openalex_id",

        "focal_first_author_id"

    ],

    how="left"

)


# =============================================================================
# 8. BASIC PAPER-LEVEL QC
# =============================================================================

print(
    "\nPAPER-LEVEL QC"
)

print(
    "Focal papers:",
    f"{len(df):,}"
)

print(
    "Mean returning collaborators:",
    round(
        df["n_returning"].mean(),
        4
    )
)

print(
    "Mean new collaborators:",
    round(
        df["n_new"].mean(),
        4
    )
)


print(
    "Papers with >=1 returning collaborator:",
    f"{(df['n_returning'] > 0).sum():,}"
)


print(
    "Papers with >=1 new collaborator:",
    f"{(df['n_new'] > 0).sum():,}"
)


print(
    "Returning field-Jaccard mean:",
    round(
        df[
            "returning_field_jaccard"
        ].mean(),
        4
    )
)


print(
    "New field-Jaccard mean:",
    round(
        df[
            "new_field_jaccard"
        ].mean(),
        4
    )
)


# =============================================================================
# 9. CHECK COLLABORATOR DECOMPOSITION
# =============================================================================
#
# This must hold at the focal-paper level for papers represented in the
# collaborator table:
#
#     n_returning + n_new = total coauthors
#
# We compare against the already validated focal-paper n_coauthors.
# =============================================================================

if "n_coauthors" in df.columns:

    decomposition = (

        df["n_returning"].fillna(0)

        +

        df["n_new"].fillna(0)

        ==

        df["n_coauthors"].fillna(0)

    )


    print(
        "\nCoauthor decomposition violations:",
        int(
            (~decomposition).sum()
        )
    )


# =============================================================================
# 10. FREEZE THE COMMON AUTHOR COHORT
# =============================================================================
#
# Important:
# This is defined from the complete focal-paper universe, not separately
# within each conditional expertise outcome.
# =============================================================================

author_summary = (

    df

    .groupby(
        "focal_first_author_id",
        observed=True
    )

    .agg(

        n_focal_papers=(
            "focal_openalex_id",
            "size"
        ),

        ai_sd=(
            "ai_score",
            "std"
        )

    )

)


valid_authors = author_summary.index[

    (

        author_summary[
            "n_focal_papers"
        ]

        >= 2

    )

    &

    (

        author_summary[
            "ai_sd"
        ]

        > 0

    )

]


df = df.loc[

    df[
        "focal_first_author_id"
    ].isin(
        valid_authors
    )

].copy()


print(
    "\nCOMMON AUTHOR PANEL"
)

print(
    "Authors:",
    f"{len(valid_authors):,}"
)

print(
    "Focal papers:",
    f"{len(df):,}"
)


# =============================================================================
# 11. FWL HELPER FUNCTIONS
# =============================================================================

def group_demean_matrix(

    X,

    groups,

    n_groups

):

    X = np.asarray(

        X,

        dtype=np.float64

    )


    out = np.empty_like(
        X
    )


    counts = np.bincount(

        groups,

        minlength=n_groups

    ).astype(
        np.float64
    )


    counts[
        counts == 0
    ] = 1.0


    for j in range(
        X.shape[1]
    ):

        sums = np.bincount(

            groups,

            weights=X[:, j],

            minlength=n_groups

        )


        means = (

            sums
            /
            counts

        )


        out[:, j] = (

            X[:, j]

            -

            means[groups]

        )


    return out


def fwl_residualize(

    X,

    group_arrays,

    group_sizes,

    tol=1e-8,

    max_iter=200

):

    R = np.asarray(

        X,

        dtype=np.float64

    ).copy()


    for iteration in range(

        1,

        max_iter + 1

    ):

        old = R.copy()


        for groups, n_groups in zip(

            group_arrays,

            group_sizes

        ):

            R = group_demean_matrix(

                R,

                groups,

                n_groups

            )


        change = np.max(

            np.abs(
                R - old
            )

        )


        if change < tol:

            return (

                R,

                iteration,

                change

            )


    return (

        R,

        max_iter,

        change

    )


# =============================================================================
# 12. FIXED EFFECT CODES
# =============================================================================

author_code, _ = pd.factorize(

    df[
        "focal_first_author_id"
    ],

    sort=False

)


month_key = (

    df[
        "accepted_year"
    ].astype(
        np.int64
    )

    * 100

    +

    df[
        "accepted_month"
    ].astype(
        np.int64
    )

)


month_code, _ = pd.factorize(

    month_key,

    sort=False

)


subfield_code, _ = pd.factorize(

    df[
        "subfield_id"
    ],

    sort=False

)


author_code = author_code.astype(
    np.int64
)

month_code = month_code.astype(
    np.int64
)

subfield_code = subfield_code.astype(
    np.int64
)


# =============================================================================
# 13. OUTCOME DEFINITIONS
#
# We deliberately separate:
#
#   field overlap
#   subfield overlap
#   cross-field share
#   cross-subfield share
#
# for:
#
#   returning collaborators
#   new collaborators.
#
# =============================================================================

outcome_specs = [

    (
        "returning_field_jaccard",

        "Returning collaborators: field expertise overlap",

        "jaccard"

    ),

    (
        "new_field_jaccard",

        "New collaborators: field expertise overlap",

        "jaccard"

    ),

    (
        "returning_subfield_jaccard",

        "Returning collaborators: subfield expertise overlap",

        "jaccard"

    ),

    (
        "new_subfield_jaccard",

        "New collaborators: subfield expertise overlap",

        "jaccard"

    ),

    (
        "returning_only_field_share",

        "Returning collaborators: expertise outside focal field portfolio",

        "percentage_point"

    ),

    (
        "new_only_field_share",

        "New collaborators: expertise outside focal field portfolio",

        "percentage_point"

    ),

    (
        "returning_only_subfield_share",

        "Returning collaborators: expertise outside focal subfield portfolio",

        "percentage_point"

    ),

    (
        "new_only_subfield_share",

        "New collaborators: expertise outside focal subfield portfolio",

        "percentage_point"

    ),

]


# =============================================================================
# 14. MODEL FUNCTION
#
# Complete-case filtering occurs separately for every outcome.
#
# Additionally:
#
#   returning outcomes require n_returning > 0
#   new outcomes require n_new > 0
#
# This is the critical conditional estimand.
# =============================================================================

def run_conditional_expertise_model(

    outcome_name,

    outcome_label,

    interpretation_type

):

    print(
        f"\nRunning {outcome_label} ..."
    )


    # -------------------------------------------------------------------------
    # Determine collaborator type
    # -------------------------------------------------------------------------

    if outcome_name.startswith(
        "returning_"
    ):

        type_mask = (
            df["n_returning"] > 0
        )

        collaborator_type = (
            "returning"
        )

    elif outcome_name.startswith(
        "new_"
    ):

        type_mask = (
            df["n_new"] > 0
        )

        collaborator_type = (
            "new"
        )

    else:

        raise ValueError(
            f"Cannot infer collaborator type: {outcome_name}"
        )


    # -------------------------------------------------------------------------
    # Sample
    # -------------------------------------------------------------------------

    d = df.loc[
        type_mask
    ].copy()


    required = [

        "ai_score",

        outcome_name,

        "focal_first_author_id",

        "accepted_year",

        "accepted_month",

        "subfield_id",

    ]


    d = d.dropna(
        subset=required
    ).copy()


    finite = (

        np.isfinite(
            d["ai_score"].to_numpy(
                dtype=np.float64
            )
        )

        &

        np.isfinite(
            d[outcome_name].to_numpy(
                dtype=np.float64
            )
        )

    )


    d = d.loc[
        finite
    ].copy()


    print(
        "  collaborator type:",
        collaborator_type
    )


    print(
        "  complete-case rows:",
        f"{len(d):,}"
    )


    if len(d) == 0:

        return {

            "outcome":
                outcome_name,

            "outcome_label":
                outcome_label,

            "collaborator_type":
                collaborator_type,

            "status":
                "no_complete_cases"

        }


    # -------------------------------------------------------------------------
    # FE codes
    # -------------------------------------------------------------------------

    a, _ = pd.factorize(

        d[
            "focal_first_author_id"
        ],

        sort=False

    )


    month = (

        d[
            "accepted_year"
        ].astype(
            np.int64
        )

        * 100

        +

        d[
            "accepted_month"
        ].astype(
            np.int64
        )

    )


    m, _ = pd.factorize(

        month,

        sort=False

    )


    s, _ = pd.factorize(

        d[
            "subfield_id"
        ],

        sort=False

    )


    a = a.astype(
        np.int64
    )

    m = m.astype(
        np.int64
    )

    s = s.astype(
        np.int64
    )


    # -------------------------------------------------------------------------
    # AI + outcome matrix
    # -------------------------------------------------------------------------

    ai = d[
        "ai_score"
    ].to_numpy(
        dtype=np.float64
    )


    y = d[
        outcome_name
    ].to_numpy(
        dtype=np.float64
    )


    M = np.column_stack(

        [

            ai,

            y

        ]

    )


    # -------------------------------------------------------------------------
    # FWL
    # -------------------------------------------------------------------------

    R, iterations, final_change = (

        fwl_residualize(

            M,

            group_arrays=[

                a,

                m,

                s

            ],

            group_sizes=[

                int(
                    a.max()
                )
                + 1,

                int(
                    m.max()
                )
                + 1,

                int(
                    s.max()
                )
                + 1,

            ],

            tol=1e-8,

            max_iter=200

        )

    )


    ai_resid = R[
        :,
        0
    ]


    y_resid = R[
        :,
        1
    ]


    # -------------------------------------------------------------------------
    # Final valid sample
    # -------------------------------------------------------------------------

    valid = (

        np.isfinite(
            ai_resid
        )

        &

        np.isfinite(
            y_resid
        )

    )


    n_valid = int(
        valid.sum()
    )


    print(
        "  FWL-valid rows:",
        f"{n_valid:,}"
    )


    if n_valid == 0:

        return {

            "outcome":
                outcome_name,

            "outcome_label":
                outcome_label,

            "collaborator_type":
                collaborator_type,

            "status":
                "no_fwl_valid_rows",

            "n_obs":
                0,

        }


    # -------------------------------------------------------------------------
    # Residualized variation
    # -------------------------------------------------------------------------

    ai_valid = ai_resid[
        valid
    ]

    y_valid = y_resid[
        valid
    ]


    ai_sd = float(

        np.std(

            ai_valid,

            ddof=0

        )

    )


    outcome_sd = float(

        np.std(

            y_valid,

            ddof=0

        )

    )


    print(
        "  residualized AI SD:",
        f"{ai_sd:.10f}"
    )


    print(
        "  residualized outcome SD:",
        f"{outcome_sd:.10f}"
    )


    if (

        not np.isfinite(ai_sd)

        or

        ai_sd <= 1e-12

    ):

        return {

            "outcome":
                outcome_name,

            "outcome_label":
                outcome_label,

            "collaborator_type":
                collaborator_type,

            "status":
                "no_ai_variation",

            "ai_resid_sd":
                ai_sd,

            "outcome_resid_sd":
                outcome_sd,

            "n_obs":
                n_valid,

            "n_authors":
                int(
                    np.unique(
                        a[valid]
                    ).size
                ),

        }


    if (

        not np.isfinite(outcome_sd)

        or

        outcome_sd <= 1e-12

    ):

        return {

            "outcome":
                outcome_name,

            "outcome_label":
                outcome_label,

            "collaborator_type":
                collaborator_type,

            "status":
                "no_outcome_variation",

            "ai_resid_sd":
                ai_sd,

            "outcome_resid_sd":
                outcome_sd,

            "n_obs":
                n_valid,

            "n_authors":
                int(
                    np.unique(
                        a[valid]
                    ).size
                ),

        }


    # -------------------------------------------------------------------------
    # OLS
    # -------------------------------------------------------------------------

    X_model = np.column_stack(

        [

            np.ones(
                n_valid
            ),

            ai_valid

        ]

    )


    clusters = a[
        valid
    ]


    model = sm.OLS(

        y_valid,

        X_model

    ).fit(

        cov_type="cluster",

        cov_kwds={

            "groups":
                clusters

        }

    )


    beta = float(
        model.params[1]
    )


    se = float(
        model.bse[1]
    )


    p = float(
        model.pvalues[1]
    )


    # -------------------------------------------------------------------------
    # One-SD AI effect
    # -------------------------------------------------------------------------

    effect = (

        beta

        *

        ai_sd

    )


    se_effect = (

        se

        *

        ai_sd

    )


    ci_low = (

        effect

        -

        1.96

        *

        se_effect

    )


    ci_high = (

        effect

        +

        1.96

        *

        se_effect

    )


    # -------------------------------------------------------------------------
    # Original-scale mean
    # -------------------------------------------------------------------------

    original_y = d.loc[
        valid,
        outcome_name
    ]


    mean_outcome = float(
        original_y.mean()
    )


    # -------------------------------------------------------------------------
    # Interpretation
    # -------------------------------------------------------------------------

    if interpretation_type == "jaccard":

        effect_interpreted = (

            effect
            *
            100

        )

        interpretation_unit = (
            "percentage points of Jaccard overlap"
        )

    else:

        effect_interpreted = (

            effect
            *
            100

        )

        interpretation_unit = (
            "percentage points"
        )


    # -------------------------------------------------------------------------
    # Return
    # -------------------------------------------------------------------------

    return {

        "outcome":
            outcome_name,

        "outcome_label":
            outcome_label,

        "collaborator_type":
            collaborator_type,

        "status":
            "estimated",

        "beta_raw_ai":
            beta,

        "se_raw_ai":
            se,

        "p_raw_ai":
            p,

        "ai_resid_sd":
            ai_sd,

        "outcome_resid_sd":
            outcome_sd,

        "effect_per_1sd":
            effect,

        "se_per_1sd":
            se_effect,

        "ci_low_per_1sd":
            ci_low,

        "ci_high_per_1sd":
            ci_high,

        "effect_interpreted":
            effect_interpreted,

        "interpretation_unit":
            interpretation_unit,

        "mean_outcome":
            mean_outcome,

        "n_obs":
            n_valid,

        "n_authors":
            int(
                np.unique(
                    clusters
                ).size
            ),

        "fwl_iterations":
            iterations,

        "fwl_final_change":
            final_change,

    }


# =============================================================================
# 15. RUN ALL WITHIN-TYPE MODELS
# =============================================================================

results = []


for (

    outcome_name,

    outcome_label,

    interpretation_type

) in outcome_specs:

    result = run_conditional_expertise_model(

        outcome_name=outcome_name,

        outcome_label=outcome_label,

        interpretation_type=interpretation_type

    )


    results.append(
        result
    )


results_df = pd.DataFrame(
    results
)


# =============================================================================
# 16. DISPLAY RESULTS
# =============================================================================

print(
    "\nFINAL WITHIN-TYPE EXPERTISE RESULTS"
)

display(
    results_df
)


# =============================================================================
# 17. DESCRIPTIVE AI-GRADIENT FOR WITHIN-TYPE OUTCOMES
#
# This is descriptive only.
#
# We keep the conditional nature of each outcome:
#
#   returning expertise:
#       papers with >=1 returning collaborator
#
#   new expertise:
#       papers with >=1 new collaborator
# =============================================================================

df["ai_bin"] = pd.cut(

    df["ai_score"],

    bins=[

        -1e-9,

        0.0,

        0.10,

        0.20,

        0.40,

        0.60,

        0.80,

        1.000001

    ],

    labels=[

        "0",

        "0.01-0.09",

        "0.10-0.19",

        "0.20-0.39",

        "0.40-0.59",

        "0.60-0.79",

        "0.80-1.00"

    ],

    include_lowest=True

)


dose = (
    df
    .groupby(
        "ai_bin",
        observed=True
    )
    .agg(

        mean_ai=("ai_score", "mean"),

        n_papers=("focal_openalex_id", "size"),

        n_authors=("focal_first_author_id", "nunique"),

        mean_n_returning=("n_returning", "mean"),

        mean_returning_field_jaccard=(
            "returning_field_jaccard",
            "mean"
        ),

        mean_returning_subfield_jaccard=(
            "returning_subfield_jaccard",
            "mean"
        ),

        mean_returning_only_field_share=(
            "returning_only_field_share",
            "mean"
        ),

        mean_returning_only_subfield_share=(
            "returning_only_subfield_share",
            "mean"
        ),

        mean_n_new=("n_new", "mean"),

        mean_new_field_jaccard=(
            "new_field_jaccard",
            "mean"
        ),

        mean_new_subfield_jaccard=(
            "new_subfield_jaccard",
            "mean"
        ),

        mean_new_only_field_share=(
            "new_only_field_share",
            "mean"
        ),

        mean_new_only_subfield_share=(
            "new_only_subfield_share",
            "mean"
        )

    )
    .reset_index()
)


print(
    "\nDESCRIPTIVE AI GRADIENT"
)

display(
    dose
)


# =============================================================================
# 18. SAMPLE-SUPPORT SUMMARY
#
# This is important for the paper/SI because the conditional expertise
# outcomes do not use identical estimation samples.
# =============================================================================

support = []


for (

    outcome_name,

    outcome_label,

    interpretation_type

) in outcome_specs:

    if outcome_name.startswith(
        "returning_"
    ):

        condition = (
            df["n_returning"] > 0
        )

    else:

        condition = (
            df["n_new"] > 0
        )


    support.append({

        "outcome":
            outcome_name,

        "collaborator_type":
            (
                "returning"
                if outcome_name.startswith("returning_")
                else "new"
            ),

        "papers_with_type":
            int(
                condition.sum()
            ),

        "share_all_papers":
            float(
                condition.mean()
            ),

        "papers_with_nonmissing_outcome":
            int(
                (
                    condition
                    &
                    df[outcome_name].notna()
                ).sum()
            ),

    })


support_df = pd.DataFrame(
    support
)


print(
    "\nCONDITIONAL OUTCOME SUPPORT"
)

display(
    support_df
)


# =============================================================================
# 19. SAVE
# =============================================================================

results_df.to_csv(

    "fig4_within_type_expertise_results.csv",

    index=False

)


dose.to_csv(

    "fig4_within_type_expertise_dose_response.csv",

    index=False

)


support_df.to_csv(

    "fig4_within_type_expertise_support.csv",

    index=False

)


print(
    "\nSaved:"
)

print(
    "  fig4_within_type_expertise_results.csv"

)

print(
    "  fig4_within_type_expertise_dose_response.csv"

)

print(
    "  fig4_within_type_expertise_support.csv"

)


# =============================================================================
# 20. CLEANUP
# =============================================================================

del expertise
del paper_expertise
del R
del df

gc.collect()

### Pretreatment trajectory-moment construction (input to final EB)

In [ ]:
# =============================================================================
# FIGURE 4 — CONDITIONAL TRAJECTORY ENTROPY BALANCING
#
# FINAL DESIGN
#
# Goal:
#   Make the EB weighting estimand match the actual author-half regression
#   estimand.
#
# Pretreatment periods:
#   2021H1
#   2021H2
#   2022H1
#   2022H2
#
# For EACH pretreatment period we balance:
#
#   1. publication probability
#   2. published * mean new collaborators
#   3. published * mean returning collaborators
#
# Because (1) and (2)/(3) are both balanced:
#
#   weighted E[Y | published=1]
#
# is also balanced.
#
# IMPORTANT:
#   We DO NOT impute outcomes for non-publishing authors.
#
# Treatment:
#   eventual sustained post-2022 adopters
#
# Controls:
#   never-adopters
#
# AI score:
#   not used
#
# Sparse expertise:
#   not used
#
# No post-treatment information enters the weights.
# =============================================================================


import os
import gc
import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.special import logsumexp


# =============================================================================
# 1. FILES
# =============================================================================

PAPER_FILE = (
    "fig4_event_design_with_clean_status.csv"
)

BASE_WEIGHT_FILE = (
    "fig4_entropy_balanced_author_weights.csv"
)


if not os.path.exists(PAPER_FILE):

    raise FileNotFoundError(
        PAPER_FILE
    )


if not os.path.exists(BASE_WEIGHT_FILE):

    raise FileNotFoundError(
        BASE_WEIGHT_FILE
    )


# =============================================================================
# 2. LOAD BASE EB AUTHOR UNIVERSE
# =============================================================================

weights = pd.read_csv(

    BASE_WEIGHT_FILE,

    usecols=[

        "focal_first_author_id",
        "treated",
        "eb_weight",

    ]

)


weights["focal_first_author_id"] = (

    weights[
        "focal_first_author_id"
    ].astype(str)

)


weights["treated"] = pd.to_numeric(

    weights[
        "treated"
    ],

    errors="coerce"

)


weights["eb_weight"] = pd.to_numeric(

    weights[
        "eb_weight"
    ],

    errors="coerce"

)


weights = weights.loc[

    weights[
        "focal_first_author_id"
    ].notna()

    &

    weights[
        "treated"
    ].isin([0, 1])

    &

    np.isfinite(

        weights[
            "eb_weight"
        ].to_numpy(
            dtype=float
        )

    )

    &

    (
        weights[
            "eb_weight"
        ]
        >
        0
    )

].copy()


if weights[
    "focal_first_author_id"
].duplicated().any():

    raise ValueError(
        "Duplicate author IDs in base EB weight file."
    )


print(
    "BASE EB AUTHOR UNIVERSE"
)

print(
    "Authors:",
    f"{len(weights):,}"
)

print(
    "Treated:",
    f"{int((weights['treated'] == 1).sum()):,}"
)

print(
    "Controls:",
    f"{int((weights['treated'] == 0).sum()):,}"
)


# =============================================================================
# 3. DUCKDB
# =============================================================================

try:

    import duckdb

except ImportError:

    raise ImportError(
        "DuckDB required. Install with: %pip install duckdb"
    )


con = duckdb.connect()

con.register(
    "eb_weights",
    weights
)


# =============================================================================
# 4. BUILD AUTHOR × HALF-YEAR PRETREATMENT DATA
#
# ONLY 2021-2022.
#
# We calculate author-half outcomes only when the author actually publishes.
#
# There is NO zero-filling here.
# =============================================================================

print(
    "\nBUILDING PRETREATMENT AUTHOR-HALF DATA..."
)


safe_path = PAPER_FILE.replace(
    "'",
    "''"
)


sql = f"""
WITH papers AS (

    SELECT

        CAST(
            focal_first_author_id
            AS VARCHAR
        ) AS author_id,

        CAST(
            focal_openalex_id
            AS VARCHAR
        ) AS paper_id,

        TRY_CAST(
            accepted_date
            AS DATE
        ) AS accepted_date,

        COALESCE(

            TRY_CAST(
                n_new
                AS DOUBLE
            ),

            0.0

        ) AS n_new,

        COALESCE(

            TRY_CAST(
                n_returning
                AS DOUBLE
            ),

            0.0

        ) AS n_returning

    FROM read_csv_auto(

        '{safe_path}',

        header = true

    )

),

restricted AS (

    SELECT

        p.author_id,
        p.paper_id,
        p.accepted_date,
        p.n_new,
        p.n_returning

    FROM papers p

    INNER JOIN eb_weights w

        ON p.author_id =
           w.focal_first_author_id

    WHERE p.accepted_date

          BETWEEN DATE '2021-01-01'
              AND DATE '2022-12-31'

),

prepared AS (

    SELECT

        author_id,

        paper_id,

        n_new,

        n_returning,

        CASE

            WHEN MONTH(accepted_date) <= 6
            THEN

                CAST(
                    YEAR(accepted_date)
                    AS VARCHAR
                )
                || 'H1'

            ELSE

                CAST(
                    YEAR(accepted_date)
                    AS VARCHAR
                )
                || 'H2'

        END AS half_year

    FROM restricted

),

author_half AS (

    SELECT

        author_id,

        half_year,

        COUNT(
            DISTINCT paper_id
        ) AS n_papers,

        AVG(n_new)
            AS mean_n_new,

        AVG(n_returning)
            AS mean_n_returning

    FROM prepared

    GROUP BY

        author_id,
        half_year

)

SELECT *

FROM author_half

"""


half_observed = con.execute(
    sql
).fetchdf()


print(
    "Observed author-half rows:",
    f"{len(half_observed):,}"
)

print(
    "Authors with >=1 observed pre-period:",
    f"{half_observed['author_id'].nunique():,}"
)


# =============================================================================
# 5. RENAME COLUMNS
# =============================================================================

half_observed = half_observed.rename(

    columns={

        "author_id":
            "focal_first_author_id",

    }

)


half_observed[
    "focal_first_author_id"
] = (

    half_observed[
        "focal_first_author_id"
    ].astype(str)

)


# =============================================================================
# 6. COMPLETE AUTHOR × FOUR-PERIOD GRID
#
# We use the grid ONLY to create publication indicators and to define the
# author-level moment variables.
#
# Missing outcome values remain missing.
# =============================================================================

PRE_PERIODS = [

    "2021H1",
    "2021H2",
    "2022H1",
    "2022H2",

]


author_grid = pd.MultiIndex.from_product(

    [

        weights[
            "focal_first_author_id"
        ].tolist(),

        PRE_PERIODS,

    ],

    names=[

        "focal_first_author_id",
        "half_year",

    ]

).to_frame(
    index=False
)


author_grid = author_grid.merge(

    half_observed,

    on=[

        "focal_first_author_id",
        "half_year",

    ],

    how="left",

    validate="one_to_one"

)


del half_observed

gc.collect()


# =============================================================================
# 7. CONSTRUCT MOMENT VARIABLES
#
# For each period:
#
#   published = 1 if author has a paper
#
#   pub_x_new =
#       published * author-half mean new collaborators
#
#   pub_x_returning =
#       published * author-half mean returning collaborators
#
# Thus:
#
#   E(pub_x_new) / E(pub)
#
# equals the weighted mean of new collaborators among observed publishers.
# =============================================================================

for period in PRE_PERIODS:

    mask = (

        author_grid[
            "half_year"
        ]
        ==
        period

    )


    published_col = (
        f"published_{period}"
    )


    new_col = (
        f"pub_x_new_{period}"
    )


    returning_col = (
        f"pub_x_returning_{period}"
    )


    author_grid.loc[
        mask,
        published_col
    ] = (

        author_grid.loc[
            mask,
            "n_papers"
        ]

        .fillna(0)

        >

        0

    ).astype(
        float
    )


    # -------------------------------------------------------------------------
    # Non-publishers have contribution 0 to E[published * Y].
    #
    # This is NOT an imputed outcome.
    # It is the mathematical representation of:
    #
    #     Published * Y
    #
    # -------------------------------------------------------------------------

    author_grid.loc[
        mask,
        new_col
    ] = (

        author_grid.loc[
            mask,
            "mean_n_new"
        ]

        .fillna(0.0)

    )

    author_grid.loc[
        mask,
        new_col
    ] *= (

        author_grid.loc[
            mask,
            published_col
        ]

    )


    author_grid.loc[
        mask,
        returning_col
    ] = (

        author_grid.loc[
            mask,
            "mean_n_returning"
        ]

        .fillna(0.0)

    )

    author_grid.loc[
        mask,
        returning_col
    ] *= (

        author_grid.loc[
            mask,
            published_col
        ]

    )


# =============================================================================
# 8. RECENT PUBLICATION VOLUME
#
# This is a genuinely substantive productivity variable, but is redundant
# with the four publication indicators if defined only as "number of active
# half-years." We therefore use the actual pre-period paper counts.
# =============================================================================

paper_count_total = (

    author_grid[
        "n_papers"
    ]

    .fillna(0.0)

)


# Sum within author.
author_grid["preperiod_papers_piece"] = (

    paper_count_total

)


pre_volume = (

    author_grid

    .groupby(

        "focal_first_author_id",

        observed=True

    )[
        "preperiod_papers_piece"
    ]

    .sum()

    .rename(
        "preperiod_papers"
    )

    .reset_index()

)


pre_volume["log_preperiod_papers"] = np.log1p(

    pre_volume[
        "preperiod_papers"
    ]

)


# =============================================================================
# 9. WIDE MOMENT TABLE
# =============================================================================




# The previous construction already has period-specific column names.
# The pivot above can therefore generate awkward duplicated names. Instead,
# rebuild the wide table deterministically below.
# =============================================================================

moment_wide = (

    author_grid

    .set_index(

        [

            "focal_first_author_id",

            "half_year",

        ]

    )

)


wide_parts = []


for period in PRE_PERIODS:

    g = (

        moment_wide

        .xs(

            period,

            level="half_year"

        )

    )


    part = pd.DataFrame(

        index=g.index

    )


    part[
        f"published_{period}"
    ] = g[
        f"published_{period}"
    ]


    part[
        f"pub_x_new_{period}"
    ] = g[
        f"pub_x_new_{period}"
    ]


    part[
        f"pub_x_returning_{period}"
    ] = g[
        f"pub_x_returning_{period}"
    ]


    wide_parts.append(
        part
    )


moment_wide = pd.concat(

    wide_parts,

    axis=1

).reset_index()


del wide_parts
del author_grid

gc.collect()


# =============================================================================
# 10. ATTACH TREATMENT
# =============================================================================

moment_wide = moment_wide.merge(

    weights[

        [

            "focal_first_author_id",
            "treated",

        ]

    ],

    on="focal_first_author_id",

    how="inner",

    validate="one_to_one"

)


moment_wide = moment_wide.merge(

    pre_volume[

        [

            "focal_first_author_id",
            "log_preperiod_papers",

        ]

    ],

    on="focal_first_author_id",

    how="left",

    validate="one_to_one"

)


del pre_volume

gc.collect()


# =============================================================================
# 11. CHECK MOMENT VARIABLES
# =============================================================================

balance_vars = []


for period in PRE_PERIODS:

    balance_vars.append(

        f"published_{period}"

    )


for period in PRE_PERIODS:

    balance_vars.append(

        f"pub_x_new_{period}"

    )


for period in PRE_PERIODS:

    balance_vars.append(

        f"pub_x_returning_{period}"

    )


balance_vars.append(
    "log_preperiod_papers"
)


missing_balance = [

    c

    for c in balance_vars

    if c not in moment_wide.columns

]


if missing_balance:

    raise KeyError(

        "Missing trajectory-EB moment variables: "

        +

        ", ".join(missing_balance)

    )


print(
    "\nTRAJECTORY MOMENT CONSTRUCTION: PASS"
)

print(
    "Balance variables:",
    len(balance_vars)
)



# Save pretreatment author-level moments for the FINAL analysis-sample EB fit.
moment_wide.to_csv(
    "fig4_conditional_trajectory_eb_author_moments.csv",
    index=False
)
print("Saved pretreatment author-level moments")


### Final analysis-sample entropy balancing

In [ ]:
# =============================================================================
# FIGURE 4 — FINAL ANALYSIS-SAMPLE CONDITIONAL TRAJECTORY EB
#
# ESTIMAND POPULATION
# -------------------
# Authors with >=1 observed publication during:
#     2021H1, 2021H2, 2022H1, 2022H2
#
# This is EXACTLY the population eligible for the subsequent
# change-from-baseline analysis.
#
# IMPORTANT:
#   - No post-2022 information is used.
#   - No outcome-result-driven variable selection.
#   - Same 12 trajectory moments as before:
#
#       published_t
#       published_t * new_t
#       published_t * returning_t
#
#   - plus log pre-period paper count if available.
#
# Once these weights pass, FREEZE THEM.
# =============================================================================

import os
import gc
import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.special import logsumexp


# =============================================================================
# 1. INPUT
# =============================================================================

MOMENT_FILE = (
    "fig4_conditional_trajectory_eb_author_moments.csv"
)

if not os.path.exists(MOMENT_FILE):
    raise FileNotFoundError(MOMENT_FILE)


moments = pd.read_csv(
    MOMENT_FILE
)


moments["focal_first_author_id"] = (
    moments["focal_first_author_id"]
    .astype(str)
)


moments["treated"] = pd.to_numeric(
    moments["treated"],
    errors="coerce"
)


# =============================================================================
# 2. PREDETERMINED ANALYSIS POPULATION
# =============================================================================

PRE_PERIODS = [
    "2021H1",
    "2021H2",
    "2022H1",
    "2022H2",
]


publication_cols = [
    f"published_{p}"
    for p in PRE_PERIODS
]


required = (
    [
        "focal_first_author_id",
        "treated",
    ]
    +
    publication_cols
    +
    [
        f"pub_x_new_{p}"
        for p in PRE_PERIODS
    ]
    +
    [
        f"pub_x_returning_{p}"
        for p in PRE_PERIODS
    ]
)


missing = [
    c
    for c in required
    if c not in moments.columns
]


if missing:
    raise KeyError(
        "Missing required moment columns: "
        + ", ".join(missing)
    )


# At least one observed pre-2023 publication.
moments["n_active_preperiods"] = (
    moments[publication_cols]
    .fillna(0)
    .sum(axis=1)
)


analysis = moments.loc[
    moments["treated"].isin([0, 1])
    &
    (moments["n_active_preperiods"] > 0)
].copy()


print(
    "FINAL EB ESTIMAND POPULATION"
)

print(
    "Authors:",
    f"{len(analysis):,}"
)

print(
    "Treated:",
    f"{int((analysis['treated'] == 1).sum()):,}"
)

print(
    "Controls:",
    f"{int((analysis['treated'] == 0).sum()):,}"
)


# Expected from previous QC.
if int((analysis["treated"] == 1).sum()) != 17019:
    print(
        "WARNING: treated count differs from previous analysis sample."
    )

if int((analysis["treated"] == 0).sum()) != 295870:
    print(
        "WARNING: control count differs from previous analysis sample."
    )


# =============================================================================
# 3. BALANCE VARIABLES
# =============================================================================

balance_vars = []


for p in PRE_PERIODS:
    balance_vars.append(
        f"published_{p}"
    )


for p in PRE_PERIODS:
    balance_vars.append(
        f"pub_x_new_{p}"
    )


for p in PRE_PERIODS:
    balance_vars.append(
        f"pub_x_returning_{p}"
    )


# Keep the same productivity moment used previously, if present.
if "log_preperiod_papers" in analysis.columns:

    balance_vars.append(
        "log_preperiod_papers"
    )


print(
    "Balance moments:",
    len(balance_vars)
)

print(
    balance_vars
)


# =============================================================================
# 4. NUMERIC MATRIX
# =============================================================================

X_raw = (
    analysis[balance_vars]
    .apply(
        pd.to_numeric,
        errors="coerce"
    )
    .to_numpy(
        dtype=np.float64
    )
)


if not np.isfinite(X_raw).all():

    bad = np.where(
        ~np.isfinite(X_raw)
    )

    raise ValueError(
        f"Non-finite balance matrix entries: {len(bad[0]):,}"
    )


# =============================================================================
# 5. STANDARDIZE
# =============================================================================

mu = X_raw.mean(axis=0)

sd = X_raw.std(axis=0)


keep = (
    np.isfinite(sd)
    &
    (sd > 1e-12)
)


if not keep.all():

    dropped = [
        balance_vars[i]
        for i in np.where(~keep)[0]
    ]

    print(
        "Dropping zero-variance moments:",
        dropped
    )

    X_raw = X_raw[:, keep]

    mu = mu[keep]

    sd = sd[keep]

    balance_vars = [
        balance_vars[i]
        for i in np.where(keep)[0]
    ]


X = (
    X_raw - mu
) / sd


treated_mask = (
    analysis["treated"]
    .to_numpy(dtype=np.int8)
    == 1
)

control_mask = ~treated_mask


Xt = X[treated_mask]

Xc = X[control_mask]


target = Xt.mean(axis=0)


# =============================================================================
# 6. ENTROPY-BALANCING FUNCTIONS
# =============================================================================

def normalized_weights(lam):

    scores = Xc @ lam

    return np.exp(
        scores
        -
        logsumexp(scores)
    )


def objective(lam):

    scores = Xc @ lam

    return (
        logsumexp(scores)
        -
        target @ lam
    )


def gradient(lam):

    w = normalized_weights(lam)

    weighted_mean = (
        w[:, None]
        *
        Xc
    ).sum(axis=0)

    return (
        weighted_mean
        -
        target
    )


def hessian(lam):

    w = normalized_weights(lam)

    weighted_mean = (
        w[:, None]
        *
        Xc
    ).sum(axis=0)

    centered = (
        Xc
        -
        weighted_mean
    )

    return (
        centered.T
        @
        (
            centered
            *
            w[:, None]
        )
    )


# =============================================================================
# 7. L-BFGS
# =============================================================================

print(
    "\nSOLVING FINAL ANALYSIS-SAMPLE EB..."
)


fit = minimize(

    objective,

    np.zeros(
        Xc.shape[1],
        dtype=np.float64
    ),

    jac=gradient,

    method="L-BFGS-B",

    options={
        "maxiter": 5000,
        "ftol": 1e-14,
        "gtol": 1e-10,
        "maxls": 100,
    }

)


print(
    "L-BFGS success:",
    fit.success
)

print(
    "Iterations:",
    fit.nit
)

print(
    "Message:",
    fit.message
)


lam = fit.x.copy()


# =============================================================================
# 8. NEWTON REFINEMENT
# =============================================================================

NEWTON_TOL = 1e-11
NEWTON_MAX = 50

newton_converged = False


for newton_iter in range(
    1,
    NEWTON_MAX + 1
):

    g = gradient(lam)

    max_error = float(
        np.max(
            np.abs(g)
        )
    )


    if max_error < NEWTON_TOL:

        newton_converged = True
        break


    H = hessian(lam)


    ridge = (
        1e-12
        *
        max(
            1.0,
            float(np.trace(H))
        )
    )


    Hreg = (
        H
        +
        ridge
        *
        np.eye(H.shape[0])
    )


    try:

        step = np.linalg.solve(
            Hreg,
            g
        )

    except np.linalg.LinAlgError:

        step = (
            np.linalg.pinv(Hreg)
            @
            g
        )


    current_obj = objective(lam)

    alpha = 1.0
    accepted = False


    for _ in range(80):

        candidate = (
            lam
            -
            alpha * step
        )

        candidate_obj = objective(
            candidate
        )


        if (
            np.isfinite(candidate_obj)
            and
            candidate_obj
            <=
            current_obj + 1e-14
        ):

            accepted = True
            break


        alpha *= 0.5


    if not accepted:
        break


    lam = candidate


final_gradient = gradient(lam)


max_moment_error = float(
    np.max(
        np.abs(final_gradient)
    )
)


print(
    "Newton iterations:",
    newton_iter
)

print(
    "Newton converged:",
    newton_converged
)

print(
    "Maximum standardized moment error:",
    f"{max_moment_error:.12e}"
)


if max_moment_error > 1e-8:

    raise RuntimeError(
        "Final analysis-sample EB failed numerical balance."
    )


# =============================================================================
# 9. ATT WEIGHTS
# =============================================================================

raw_control_weights = normalized_weights(
    lam
)


n_treated = int(
    treated_mask.sum()
)


control_weights = (
    raw_control_weights
    *
    (
        n_treated
        /
        raw_control_weights.sum()
    )
)


analysis[
    "final_eb_weight"
] = np.nan


analysis.loc[
    treated_mask,
    "final_eb_weight"
] = 1.0


analysis.loc[
    control_mask,
    "final_eb_weight"
] = control_weights


# =============================================================================
# 10. BALANCE TABLE
# =============================================================================

treated = analysis.loc[
    analysis["treated"] == 1
]

controls = analysis.loc[
    analysis["treated"] == 0
]


def wmean(x, w):

    x = np.asarray(
        x,
        dtype=float
    )

    w = np.asarray(
        w,
        dtype=float
    )

    return (
        np.sum(x * w)
        /
        np.sum(w)
    )


rows = []


for var in balance_vars:

    t = treated[var].to_numpy(
        dtype=float
    )

    c = controls[var].to_numpy(
        dtype=float
    )


    mt = t.mean()

    mc0 = c.mean()

    mc1 = wmean(
        c,
        control_weights
    )


    psd = np.sqrt(
        0.5
        *
        (
            t.var(ddof=0)
            +
            c.var(ddof=0)
        )
    )


    smd0 = (
        (mt - mc0) / psd
        if psd > 1e-12
        else 0.0
    )

    smd1 = (
        (mt - mc1) / psd
        if psd > 1e-12
        else 0.0
    )


    rows.append({

        "variable":
            var,

        "treated_mean":
            mt,

        "control_mean_before":
            mc0,

        "control_mean_after":
            mc1,

        "difference_before":
            mt - mc0,

        "difference_after":
            mt - mc1,

        "SMD_before":
            smd0,

        "SMD_after":
            smd1,

    })


balance = pd.DataFrame(rows)


print(
    "\nFINAL ANALYSIS-SAMPLE BALANCE"
)

display(
    balance
)


max_smd = float(
    balance[
        "SMD_after"
    ].abs().max()
)


print(
    "Maximum |SMD|:",
    f"{max_smd:.12e}"
)


# =============================================================================
# 11. DIRECT CONDITIONAL MEAN CHECK
# =============================================================================

conditional_rows = []


for p in PRE_PERIODS:

    pub = f"published_{p}"

    new = f"pub_x_new_{p}"

    ret = f"pub_x_returning_{p}"


    tpub = treated[
        pub
    ].to_numpy(float)

    cpub = controls[
        pub
    ].to_numpy(float)


    tnew = treated[
        new
    ].to_numpy(float)

    cnew = controls[
        new
    ].to_numpy(float)


    tret = treated[
        ret
    ].to_numpy(float)

    cret = controls[
        ret
    ].to_numpy(float)


    t_pub_rate = tpub.mean()

    c_pub_rate = wmean(
        cpub,
        control_weights
    )


    t_new_cond = (
        tnew.sum()
        /
        tpub.sum()
    )

    c_new_cond = (
        np.sum(
            cnew * control_weights
        )
        /
        np.sum(
            cpub * control_weights
        )
    )


    t_ret_cond = (
        tret.sum()
        /
        tpub.sum()
    )

    c_ret_cond = (
        np.sum(
            cret * control_weights
        )
        /
        np.sum(
            cpub * control_weights
        )
    )


    conditional_rows.append({

        "half_year":
            p,

        "treated_publication_rate":
            t_pub_rate,

        "control_publication_rate_weighted":
            c_pub_rate,

        "publication_gap":
            t_pub_rate - c_pub_rate,

        "treated_conditional_new":
            t_new_cond,

        "control_conditional_new":
            c_new_cond,

        "conditional_new_gap":
            t_new_cond - c_new_cond,

        "treated_conditional_returning":
            t_ret_cond,

        "control_conditional_returning":
            c_ret_cond,

        "conditional_returning_gap":
            t_ret_cond - c_ret_cond,

    })


conditional_balance = pd.DataFrame(
    conditional_rows
)


print(
    "\nFINAL CONDITIONAL BALANCE"
)

display(
    conditional_balance
)


# =============================================================================
# 12. ESS
# =============================================================================

ess = (
    control_weights.sum() ** 2
    /
    np.sum(
        control_weights ** 2
    )
)


print(
    "\nFINAL WEIGHT DIAGNOSTICS"
)

print(
    "Treated:",
    f"{n_treated:,}"
)

print(
    "Controls:",
    f"{len(control_weights):,}"
)

print(
    "Control ESS:",
    f"{ess:,.1f}"
)

print(
    "ESS fraction:",
    f"{ess / len(control_weights):.4f}"
)

print(
    "Max control weight:",
    f"{control_weights.max():.6f}"
)


# =============================================================================
# 13. HARD FINAL QC
# =============================================================================

max_pub_gap = float(
    conditional_balance[
        "publication_gap"
    ].abs().max()
)

max_new_gap = float(
    conditional_balance[
        "conditional_new_gap"
    ].abs().max()
)

max_ret_gap = float(
    conditional_balance[
        "conditional_returning_gap"
    ].abs().max()
)


if (
    max_smd > 1e-5
    or
    max_pub_gap > 1e-7
    or
    max_new_gap > 1e-7
    or
    max_ret_gap > 1e-7
):

    raise RuntimeError(
        "Final analysis-sample EB did not achieve required balance."
    )


print(
    "\nFINAL ANALYSIS-SAMPLE CONDITIONAL EB: PASS"
)


# =============================================================================
# 14. SAVE
# =============================================================================

analysis[
    [
        "focal_first_author_id",
        "treated",
        "final_eb_weight",
    ]
].to_csv(

    "fig4_final_change_sample_eb_weights.csv",

    index=False
)


balance.to_csv(

    "fig4_final_change_sample_eb_balance.csv",

    index=False
)


conditional_balance.to_csv(

    "fig4_final_change_sample_conditional_balance.csv",

    index=False
)


pd.DataFrame([{

    "n_treated":
        n_treated,

    "n_controls":
        len(control_weights),

    "control_ESS":
        ess,

    "ESS_fraction":
        ess / len(control_weights),

    "max_control_weight":
        control_weights.max(),

    "max_abs_SMD":
        max_smd,

    "max_publication_gap":
        max_pub_gap,

    "max_conditional_new_gap":
        max_new_gap,

    "max_conditional_returning_gap":
        max_ret_gap,

    "max_standardized_moment_error":
        max_moment_error,

}]).to_csv(

    "fig4_final_change_sample_eb_diagnostics.csv",

    index=False
)


print(
    "\nSAVED:"
)

print(
    "  fig4_final_change_sample_eb_weights.csv"
)

print(
    "  fig4_final_change_sample_eb_balance.csv"
)

print(
    "  fig4_final_change_sample_conditional_balance.csv"
)

print(
    "  fig4_final_change_sample_eb_diagnostics.csv"
)


gc.collect()

### Final longitudinal collaboration and placebo (main figure 3 panels d–f)

In [ ]:
# =============================================================================
# FIGURE 4 — FINAL WEIGHTED CHANGE-FROM-PRETREATMENT ANALYSIS
#
# DESIGN
# ------
# Frozen weights:
#   fig4_conditional_trajectory_eb_author_weights.csv
#
# Paper-level source:
#   fig4_event_design_with_clean_status.csv
#
# Pre-treatment baseline:
#   2021H1, 2021H2, 2022H1, 2022H2
#
# Post-treatment / AI-era periods:
#   2023H1 ... 2025H2
#
# For each author:
#
#   baseline_i = mean observed outcome across pre-2023 author-half periods
#
#   change_it = outcome_it - baseline_i
#
# We compare weighted:
#
#   E(change_it | treated)
#       -
#   E(change_it | control)
#
# among authors who actually publish in period t.
#
# IMPORTANT:
#   Non-publishing periods are NOT coded as zero.
#
# PRIMARY OUTCOMES:
#   new collaborators
#   returning collaborators
#   total collaborators
#   new minus returning collaborators
#
# SECONDARY:
#   share of papers with any new collaborator
#
# PLACEBO:
#   2021H1-H2 baseline -> 2022H1 and 2022H2
#
# =============================================================================


import os
import gc
import numpy as np
import pandas as pd
import statsmodels.api as sm


# =============================================================================
# 1. FILES
# =============================================================================

WEIGHT_FILE = (
    "fig4_final_change_sample_eb_weights.csv"
)

PAPER_FILE = (
    "fig4_event_design_with_clean_status.csv"
)


if not os.path.exists(WEIGHT_FILE):
    raise FileNotFoundError(WEIGHT_FILE)


if not os.path.exists(PAPER_FILE):
    raise FileNotFoundError(PAPER_FILE)


# =============================================================================
# 2. LOAD FROZEN WEIGHTS
# =============================================================================

weights = pd.read_csv(

    WEIGHT_FILE,

    usecols=[

        "focal_first_author_id",
        "treated",
        "final_eb_weight",

    ]

)


weights["focal_first_author_id"] = (
    weights["focal_first_author_id"].astype(str)
)


weights["treated"] = pd.to_numeric(
    weights["treated"],
    errors="coerce",
)


weights["final_eb_weight"] = pd.to_numeric(
    weights["final_eb_weight"],
    errors="coerce",
)


weights = weights.loc[

    weights["focal_first_author_id"].notna()

    &

    weights["treated"].isin([0, 1])

    &

    np.isfinite(
        weights["final_eb_weight"].to_numpy(float)
    )

    &

    (weights["final_eb_weight"] > 0)

].copy()


if weights["focal_first_author_id"].duplicated().any():
    raise ValueError(
        "Duplicate author IDs in frozen trajectory-EB weights."
    )


print(
    "FROZEN CONDITIONAL TRAJECTORY-EB WEIGHTS"
)

print(
    "Authors:",
    f"{len(weights):,}"
)

print(
    "Treated:",
    f"{int((weights['treated'] == 1).sum()):,}"
)

print(
    "Controls:",
    f"{int((weights['treated'] == 0).sum()):,}"
)


# =============================================================================
# 3. BUILD COMPACT AUTHOR × HALF-YEAR PANEL WITH DUCKDB
# =============================================================================
#
# We deliberately read the large CSV with DuckDB rather than pandas.
# =============================================================================

try:
    import duckdb
except ImportError:
    raise ImportError(
        "DuckDB is required. Run: %pip install duckdb"
    )


con = duckdb.connect()

con.register(
    "author_weights",
    weights
)


safe_paper_file = PAPER_FILE.replace(
    "'",
    "''"
)


print(
    "\nBUILDING AUTHOR × HALF-YEAR PANEL..."
)


sql = f"""
WITH papers AS (

    SELECT

        CAST(
            focal_first_author_id
            AS VARCHAR
        ) AS author_id,

        CAST(
            focal_openalex_id
            AS VARCHAR
        ) AS paper_id,

        TRY_CAST(
            accepted_date
            AS DATE
        ) AS accepted_date,

        COALESCE(
            TRY_CAST(
                n_new
                AS DOUBLE
            ),
            0.0
        ) AS n_new,

        COALESCE(
            TRY_CAST(
                n_returning
                AS DOUBLE
            ),
            0.0
        ) AS n_returning

    FROM read_csv_auto(
        '{safe_paper_file}',
        header = true
    )

),

restricted AS (

    SELECT
        p.*

    FROM papers p

    INNER JOIN author_weights w

        ON p.author_id =
           w.focal_first_author_id

    WHERE p.accepted_date
          BETWEEN DATE '2021-01-01'
              AND DATE '2025-12-31'

),

prepared AS (

    SELECT

        author_id,

        paper_id,

        n_new,

        n_returning,

        n_new + n_returning
            AS total_collaborators,

        CASE
            WHEN n_new > 0
            THEN 1.0
            ELSE 0.0
        END
            AS has_new,

        CASE

            WHEN n_new + n_returning > 0

            THEN
                n_returning
                /
                (n_new + n_returning)

            ELSE NULL

        END
            AS returning_share,

        CASE

            WHEN MONTH(accepted_date) <= 6

            THEN
                CAST(
                    YEAR(accepted_date)
                    AS VARCHAR
                ) || 'H1'

            ELSE
                CAST(
                    YEAR(accepted_date)
                    AS VARCHAR
                ) || 'H2'

        END
            AS half_year

    FROM restricted

),

author_half AS (

    SELECT

        author_id,

        half_year,

        COUNT(
            DISTINCT paper_id
        )
            AS n_papers,

        AVG(n_new)
            AS mean_n_new,

        AVG(n_returning)
            AS mean_n_returning,

        AVG(total_collaborators)
            AS mean_total_collaborators,

        AVG(has_new)
            AS share_any_new,

        AVG(returning_share)
            AS mean_returning_share

    FROM prepared

    GROUP BY

        author_id,
        half_year

)

SELECT *

FROM author_half

"""


author_half = con.execute(
    sql
).fetchdf()


con.close()


print(
    "Author-half rows:",
    f"{len(author_half):,}"
)

print(
    "Authors:",
    f"{author_half['author_id'].nunique():,}"
)


# =============================================================================
# 4. STANDARDIZE ID
# =============================================================================

author_half = author_half.rename(

    columns={
        "author_id":
            "focal_first_author_id",
    }

)


author_half["focal_first_author_id"] = (

    author_half[
        "focal_first_author_id"
    ].astype(str)

)


# =============================================================================
# 5. ATTACH FROZEN WEIGHTS
# =============================================================================

author_half = author_half.merge(

    weights,

    on="focal_first_author_id",

    how="inner",

    validate="many_to_one"

)


# =============================================================================
# 6. PERIOD ORDER
# =============================================================================

PERIODS = [

    "2021H1",
    "2021H2",

    "2022H1",
    "2022H2",

    "2023H1",
    "2023H2",

    "2024H1",
    "2024H2",

    "2025H1",
    "2025H2",

]


period_index = {

    p: i

    for i, p in enumerate(
        PERIODS
    )

}


author_half["period_index"] = (

    author_half[
        "half_year"
    ].map(
        period_index
    )

)


# =============================================================================
# 7. RESTRICT TO EXPECTED PERIODS
# =============================================================================

author_half = author_half.loc[

    author_half[
        "period_index"
    ].notna()

].copy()


# =============================================================================
# 8. DEFINITIONS OF BASELINE / POST PERIODS
# =============================================================================

BASELINE_PERIODS = [

    "2021H1",
    "2021H2",
    "2022H1",
    "2022H2",

]


POST_PERIODS = [

    "2023H1",
    "2023H2",
    "2024H1",
    "2024H2",
    "2025H1",
    "2025H2",

]


PLACEBO_BASELINE_PERIODS = [

    "2021H1",
    "2021H2",

]


PLACEBO_PERIODS = [

    "2022H1",
    "2022H2",

]


# =============================================================================
# 9. COMPUTE AUTHOR-SPECIFIC PRETREATMENT BASELINE
#
# For each outcome:
#
#   baseline_i =
#       mean of that author's OBSERVED pre-treatment author-half outcomes
#
# Authors with no pre-treatment publication are excluded from the
# change-from-baseline analysis because their baseline is undefined.
# =============================================================================

BASELINE_OUTCOMES = [

    "mean_n_new",
    "mean_n_returning",
    "mean_total_collaborators",
    "share_any_new",

]


baseline = (

    author_half.loc[

        author_half[
            "half_year"
        ].isin(
            BASELINE_PERIODS
        )

    ]

    .groupby(

        "focal_first_author_id",

        observed=True

    )[

        BASELINE_OUTCOMES

    ]

    .mean()

    .reset_index()

)


baseline = baseline.rename(

    columns={

        col:
            f"baseline_{col}"

        for col in BASELINE_OUTCOMES

    }

)


baseline_n = (

    author_half.loc[

        author_half[
            "half_year"
        ].isin(
            BASELINE_PERIODS
        )

    ]

    .groupby(

        "focal_first_author_id",

        observed=True

    )

    .size()

    .rename(
        "n_pre_periods"
    )

    .reset_index()

)


baseline = baseline.merge(

    baseline_n,

    on="focal_first_author_id",

    how="left",

    validate="one_to_one"

)


# =============================================================================
# 10. MERGE BASELINE INTO AUTHOR-HALF PANEL
# =============================================================================

author_half = author_half.merge(

    baseline,

    on="focal_first_author_id",

    how="left",

    validate="many_to_one"

)


del baseline
del baseline_n

gc.collect()


# =============================================================================
# 11. KEEP AUTHORS WITH AT LEAST ONE PRE-PERIOD OBSERVATION
# =============================================================================

analysis_panel = author_half.loc[

    author_half[
        "n_pre_periods"
    ].fillna(0)

    >

    0

].copy()


print(
    "\nCHANGE-FROM-BASELINE PANEL"
)

print(
    "Rows:",
    f"{len(analysis_panel):,}"
)

print(
    "Authors:",
    f"{analysis_panel['focal_first_author_id'].nunique():,}"
)

print(
    "Treated authors:",
    f"{analysis_panel.loc[analysis_panel['treated'] == 1, 'focal_first_author_id'].nunique():,}"
)

print(
    "Control authors:",
    f"{analysis_panel.loc[analysis_panel['treated'] == 0, 'focal_first_author_id'].nunique():,}"
)


# =============================================================================
# 12. CONSTRUCT AUTHOR-LEVEL CHANGE
#
# Only actual observed publication periods are retained.
# =============================================================================

change_specs = [

    (
        "mean_n_new",
        "baseline_mean_n_new",
        "change_n_new",
    ),

    (
        "mean_n_returning",
        "baseline_mean_n_returning",
        "change_n_returning",
    ),

    (
        "mean_total_collaborators",
        "baseline_mean_total_collaborators",
        "change_total_collaborators",
    ),

    (
        "share_any_new",
        "baseline_share_any_new",
        "change_share_any_new",
    ),

]


for outcome_col, baseline_col, change_col in change_specs:

    analysis_panel[change_col] = (

        analysis_panel[
            outcome_col
        ]

        -

        analysis_panel[
            baseline_col
        ]

    )


# =============================================================================
# 13. DIRECT NEW-RETURNING CHANGE
# =============================================================================

analysis_panel["change_new_minus_returning"] = (

    analysis_panel[
        "change_n_new"
    ]

    -

    analysis_panel[
        "change_n_returning"
    ]

)


# =============================================================================
# 14. CHECK BASELINE OBSERVATION COUNTS
# =============================================================================

baseline_support = (

    author_half

    .loc[

        author_half[
            "n_pre_periods"
        ].fillna(0)
        >
        0

    ]

    .groupby(

        "treated",

        observed=True

    )

    .agg(

        n_authors=(

            "focal_first_author_id",

            "nunique"

        ),

        median_pre_periods=(

            "n_pre_periods",

            "median"

        ),

        mean_pre_periods=(

            "n_pre_periods",

            "mean"

        ),

    )

    .reset_index()

)


print(
    "\nBASELINE SUPPORT"
)

display(
    baseline_support
)


# =============================================================================
# 15. WEIGHTED CHANGE ESTIMATOR
#
# For every period:
#
#   weighted ATT-style difference =
#
#       mean_w(change | treated)
#       -
#       mean_w(change | control)
#
# This is transparent and does not impose a parallel-trends event-study
# structure on the changing author-period sample.
# =============================================================================

def weighted_mean_safe(
    x,
    w
):

    x = np.asarray(
        x,
        dtype=np.float64
    )

    w = np.asarray(
        w,
        dtype=np.float64
    )


    valid = (

        np.isfinite(x)
        &
        np.isfinite(w)
        &
        (w > 0)

    )


    if not valid.any():

        return np.nan


    return (

        np.sum(
            x[valid] * w[valid]
        )

        /

        np.sum(
            w[valid]
        )

    )


def run_change_descriptive(

    outcome_change,
    label,
    periods

):

    rows = []


    for period in periods:

        d = analysis_panel.loc[

            analysis_panel[
                "half_year"
            ]
            ==
            period

        ]


        d = d.loc[

            d[
                outcome_change
            ].notna()

        ]


        if len(d) == 0:

            continue


        treated = d.loc[
            d["treated"] == 1
        ]


        control = d.loc[
            d["treated"] == 0
        ]


        mt = weighted_mean_safe(

            treated[
                outcome_change
            ],

            treated[
                "final_eb_weight"
            ]

        )


        mc = weighted_mean_safe(

            control[
                outcome_change
            ],

            control[
                "final_eb_weight"
            ]

        )


        rows.append({

            "outcome":
                label,

            "half_year":
                period,

            "event_time":
                period_index[period]
                -
                period_index["2022H2"],

            "treated_n":
                treated[
                    "focal_first_author_id"
                ].nunique(),

            "control_n":
                control[
                    "focal_first_author_id"
                ].nunique(),

            "treated_weighted_change":
                mt,

            "control_weighted_change":
                mc,

            "weighted_gap":
                mt - mc,

        })


    return pd.DataFrame(
        rows
    )


# =============================================================================
# 16. DESCRIPTIVE CHANGE RESULTS
# =============================================================================

change_results = pd.concat(

    [

        run_change_descriptive(

            "change_n_new",

            "New collaborators",

            POST_PERIODS

        ),

        run_change_descriptive(

            "change_n_returning",

            "Returning collaborators",

            POST_PERIODS

        ),

        run_change_descriptive(

            "change_total_collaborators",

            "Total collaborators",

            POST_PERIODS

        ),

        run_change_descriptive(

            "change_new_minus_returning",

            "New minus returning collaborators",

            POST_PERIODS

        ),

    ],

    ignore_index=True

)


print(
    "\nPOST-2023 WEIGHTED CHANGE-FROM-BASELINE"
)

display(
    change_results
)


# =============================================================================
# 17. PRE-2023 PLACEBO
#
# Baseline = 2021H1 + 2021H2
#
# Then calculate changes in:
#   2022H1
#   2022H2
#
# This is NOT treatment.
# It is a diagnostic for whether the same weighted comparison shows
# systematic pre-period changes.
# =============================================================================

placebo_base = (

    author_half.loc[

        author_half[
            "half_year"
        ].isin(
            PLACEBO_BASELINE_PERIODS
        )

    ]

    .groupby(

        "focal_first_author_id",

        observed=True

    )[

        BASELINE_OUTCOMES

    ]

    .mean()

    .reset_index()

)


placebo_base = placebo_base.rename(

    columns={

        col:
            f"placebo_baseline_{col}"

        for col in BASELINE_OUTCOMES

    }

)


placebo_panel = (

    author_half.loc[

        author_half[
            "half_year"
        ].isin(
            PLACEBO_PERIODS
        )

    ]

    .merge(

        placebo_base,

        on="focal_first_author_id",

        how="inner",

        validate="many_to_one"

    )

    .merge(

        weights,

        on="focal_first_author_id",

        how="inner",

        validate="many_to_one"

    )

)


# Avoid duplicate columns if weights were already attached.
for col in [
    "treated",
    "final_eb_weight",
]:
    if f"{col}_x" in placebo_panel.columns:
        placebo_panel[col] = placebo_panel[f"{col}_x"]
        placebo_panel.drop(
            columns=[
                f"{col}_x",
                f"{col}_y",
            ],
            inplace=True,
            errors="ignore"
        )


for outcome_col in BASELINE_OUTCOMES:

    placebo_panel[
        f"placebo_change_{outcome_col}"
    ] = (

        placebo_panel[
            outcome_col
        ]

        -

        placebo_panel[
            f"placebo_baseline_{outcome_col}"
        ]

    )


placebo_panel[
    "placebo_change_new_minus_returning"
] = (

    placebo_panel[
        "placebo_change_mean_n_new"
    ]

    -

    placebo_panel[
        "placebo_change_mean_n_returning"
    ]

)


def run_placebo_descriptive(

    change_col,
    label

):

    rows = []


    for period in PLACEBO_PERIODS:

        d = placebo_panel.loc[

            placebo_panel[
                "half_year"
            ]
            ==
            period

        ]


        d = d.loc[
            d[change_col].notna()
        ]


        treated = d.loc[
            d["treated"] == 1
        ]


        control = d.loc[
            d["treated"] == 0
        ]


        mt = weighted_mean_safe(

            treated[
                change_col
            ],

            treated[
                "final_eb_weight"
            ]

        )


        mc = weighted_mean_safe(

            control[
                change_col
            ],

            control[
                "final_eb_weight"
            ]

        )


        rows.append({

            "outcome":
                label,

            "half_year":
                period,

            "placebo_gap":
                mt - mc,

            "treated_n":
                treated[
                    "focal_first_author_id"
                ].nunique(),

            "control_n":
                control[
                    "focal_first_author_id"
                ].nunique(),

        })


    return pd.DataFrame(
        rows
    )


placebo_results = pd.concat(

    [

        run_placebo_descriptive(

            "placebo_change_mean_n_new",

            "New collaborators"

        ),

        run_placebo_descriptive(

            "placebo_change_mean_n_returning",

            "Returning collaborators"

        ),

        run_placebo_descriptive(

            "placebo_change_mean_total_collaborators",

            "Total collaborators"

        ),

        run_placebo_descriptive(

            "placebo_change_new_minus_returning",

            "New minus returning collaborators"

        ),

    ],

    ignore_index=True

)


print(
    "\nPRE-2023 PLACEBO CHANGE"
)

display(
    placebo_results
)


# =============================================================================
# 18. AUTHOR-LEVEL REGRESSION FOR POST PERIODS
#
# We regress change from baseline on treated status separately by half-year:
#
#     change_it = alpha + beta * treated_i + epsilon_it
#
# Weighted by frozen trajectory EB weights.
#
# Cluster by author.
#
# This is NOT a two-way FE event study. The outcome has already been expressed
# as change from the author-specific pre-treatment baseline.
# =============================================================================

def run_change_regression(

    change_col,
    label,
    periods

):

    rows = []


    for period in periods:

        d = analysis_panel.loc[

            (

                analysis_panel[
                    "half_year"
                ]
                ==
                period

            )

            &

            analysis_panel[
                change_col
            ].notna()

        ].copy()


        if len(d) == 0:

            continue


        y = d[
            change_col
        ].to_numpy(
            dtype=float
        )


        x = d[
            "treated"
        ].to_numpy(
            dtype=float
        )


        w = d[
            "final_eb_weight"
        ].to_numpy(
            dtype=float
        )


        X = np.column_stack(

            [

                np.ones(
                    len(d)
                ),

                x,

            ]

        )


        cluster_codes, _ = pd.factorize(

            d[
                "focal_first_author_id"
            ],

            sort=False

        )


        cluster_codes = cluster_codes.astype(
            np.int64
        )


        model = sm.WLS(

            y,

            X,

            weights=w

        ).fit(

            cov_type="cluster",

            cov_kwds={

                "groups":
                    cluster_codes

            }

        )


        beta = float(
            model.params[1]
        )


        se = float(
            model.bse[1]
        )


        p = float(
            model.pvalues[1]
        )


        rows.append({

            "outcome":
                label,

            "half_year":
                period,

            "event_time":
                period_index[period]
                -
                period_index["2022H2"],

            "beta":
                beta,

            "se":
                se,

            "p":
                p,

            "ci_low":
                beta - 1.96 * se,

            "ci_high":
                beta + 1.96 * se,

            "n_obs":
                len(d),

            "n_authors":
                d[
                    "focal_first_author_id"
                ].nunique(),

        })


    return pd.DataFrame(
        rows
    )


# =============================================================================
# 19. PRIMARY CHANGE REGRESSIONS
# =============================================================================

change_regression_results = pd.concat(

    [

        run_change_regression(

            "change_n_new",

            "New collaborators",

            POST_PERIODS

        ),

        run_change_regression(

            "change_n_returning",

            "Returning collaborators",

            POST_PERIODS

        ),

        run_change_regression(

            "change_total_collaborators",

            "Total collaborators",

            POST_PERIODS

        ),

        run_change_regression(

            "change_new_minus_returning",

            "New minus returning collaborators",

            POST_PERIODS

        ),

    ],

    ignore_index=True

)


print(
    "\nCHANGE-FROM-BASELINE REGRESSION RESULTS"
)

display(
    change_regression_results
)


# =============================================================================
# 20. PLACEBO REGRESSIONS
# =============================================================================
#
# Same estimator, but using 2021H1-H2 as the baseline and 2022 as the
# pseudo-post period.
# =============================================================================

def run_placebo_regression(

    change_col,
    label

):

    rows = []


    for period in PLACEBO_PERIODS:

        d = placebo_panel.loc[

            (

                placebo_panel[
                    "half_year"
                ]
                ==
                period

            )

            &

            placebo_panel[
                change_col
            ].notna()

        ].copy()


        if len(d) == 0:
            continue


        y = d[
            change_col
        ].to_numpy(
            dtype=float
        )


        x = d[
            "treated"
        ].to_numpy(
            dtype=float
        )


        w = d[
            "final_eb_weight"
        ].to_numpy(
            dtype=float
        )


        X = np.column_stack(

            [

                np.ones(
                    len(d)
                ),

                x,

            ]

        )


        clusters, _ = pd.factorize(

            d[
                "focal_first_author_id"
            ],

            sort=False

        )


        clusters = clusters.astype(
            np.int64
        )


        model = sm.WLS(

            y,

            X,

            weights=w

        ).fit(

            cov_type="cluster",

            cov_kwds={

                "groups":
                    clusters

            }

        )


        beta = float(
            model.params[1]
        )


        se = float(
            model.bse[1]
        )


        rows.append({

            "outcome":
                label,

            "half_year":
                period,

            "event_time":
                period_index[period]
                -
                period_index["2022H2"],

            "beta":
                beta,

            "se":
                se,

            "p":
                float(
                    model.pvalues[1]
                ),

            "ci_low":
                beta - 1.96 * se,

            "ci_high":
                beta + 1.96 * se,

            "n_obs":
                len(d),

            "n_authors":
                d[
                    "focal_first_author_id"
                ].nunique(),

        })


    return pd.DataFrame(
        rows
    )


placebo_regression_results = pd.concat(

    [

        run_placebo_regression(

            "placebo_change_mean_n_new",

            "New collaborators"

        ),

        run_placebo_regression(

            "placebo_change_mean_n_returning",

            "Returning collaborators"

        ),

        run_placebo_regression(

            "placebo_change_mean_total_collaborators",

            "Total collaborators"

        ),

        run_placebo_regression(

            "placebo_change_new_minus_returning",

            "New minus returning collaborators"

        ),

    ],

    ignore_index=True

)


print(
    "\nPLACEBO REGRESSION RESULTS"
)

display(
    placebo_regression_results
)


# =============================================================================
# 21. SAVE ALL RESULTS
# =============================================================================

change_results.to_csv(

    "fig4_change_from_baseline_descriptive.csv",

    index=False

)


change_regression_results.to_csv(

    "fig4_change_from_baseline_regression.csv",

    index=False

)


placebo_results.to_csv(

    "fig4_change_from_baseline_placebo_descriptive.csv",

    index=False

)


placebo_regression_results.to_csv(

    "fig4_change_from_baseline_placebo_regression.csv",

    index=False

)


baseline_support.to_csv(

    "fig4_change_from_baseline_support.csv",

    index=False

)


analysis_panel.to_csv(

    "fig4_change_from_baseline_author_panel.csv",

    index=False

)


print(
    "\nSAVED:"
)

print(
    "  fig4_change_from_baseline_descriptive.csv"
)

print(
    "  fig4_change_from_baseline_regression.csv"
)

print(
    "  fig4_change_from_baseline_placebo_descriptive.csv"
)

print(
    "  fig4_change_from_baseline_placebo_regression.csv"
)

print(
    "  fig4_change_from_baseline_support.csv"
)

print(
    "  fig4_change_from_baseline_author_panel.csv"
)


# =============================================================================
# 22. FINAL QC
# =============================================================================

print(
    "\nFINAL CHANGE-FROM-BASELINE ANALYSIS COMPLETE"
)

print(
    "Frozen EB authors:",
    f"{len(weights):,}"
)

print(
    "Authors with >=1 pre-period observation:",
    f"{analysis_panel['focal_first_author_id'].nunique():,}"
)

print(
    "Post periods:",
    POST_PERIODS
)

print(
    "Placebo periods:",
    PLACEBO_PERIODS
)


# =============================================================================
# 23. CLEANUP
# =============================================================================

del author_half
del analysis_panel
del placebo_panel
del weights

gc.collect()

### Main figure 3 visualization

In [ ]:
# =============================================================================
# FIGURE 4 — REVISED 2 × 3 RENDERING, VERSION 2
#
# FINAL RESULTS ONLY
#
# Key changes from previous draft:
#   1. Panel a gets a complete legend.
#   2. Panel b becomes a 2 × 2 set of independent coefficient plots.
#      Each expertise outcome has its own x-axis.
#   3. New/Returning are directly paired within each mini-panel.
#   4. Panel c remains grouped bars.
#   5. Panel d keeps New/Returning horizontally dodged.
#   6. Panel e keeps a wider y-scale.
#   7. Panel f uses open vs filled markers for placebo periods.
#   8. Subtitles explicitly identify author-paper / author-collaborator panels.
#   9. Figure-wide note follows the compact Figure 3 grammar.
# =============================================================================


import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D


# =============================================================================
# 1. FILES
# =============================================================================

STATIC_FILE = "fig4_static_collaboration_results.csv"
EXPERTISE_FILE = "fig4_within_type_expertise_results.csv"
DOSE_FILE = "fig4_within_type_expertise_dose_response.csv"
CHANGE_FILE = "fig4_change_from_baseline_regression.csv"
PLACEBO_FILE = "fig4_change_from_baseline_placebo_regression.csv"


required_files = [

    STATIC_FILE,
    EXPERTISE_FILE,
    DOSE_FILE,
    CHANGE_FILE,
    PLACEBO_FILE,

]


missing_files = [

    f
    for f in required_files
    if not os.path.exists(f)

]


if missing_files:

    raise FileNotFoundError(
        "Missing required final Figure 4 files:\n"
        +
        "\n".join(missing_files)
    )


static = pd.read_csv(
    STATIC_FILE
)

expertise = pd.read_csv(
    EXPERTISE_FILE
)

dose = pd.read_csv(
    DOSE_FILE
)

change = pd.read_csv(
    CHANGE_FILE
)

placebo = pd.read_csv(
    PLACEBO_FILE
)


# =============================================================================
# 2. COLORS
# =============================================================================

NEW = "#F28E2B"
RETURNING = "#59A14F"
TOTAL = "#4E79A7"

TEXT = "#222222"
GRAY = "#777777"
GRID = "#D9D9D9"


# =============================================================================
# 3. GLOBAL STYLE
# =============================================================================

plt.rcParams.update({

    "font.family":
        "DejaVu Sans",

    "font.size":
        9,

    "axes.titlesize":
        10.5,

    "axes.labelsize":
        8.8,

    "xtick.labelsize":
        7.8,

    "ytick.labelsize":
        8.2,

    "legend.fontsize":
        8.0,

    "axes.spines.top":
        False,

    "axes.spines.right":
        False,

    "axes.linewidth":
        0.75,

    "xtick.major.width":
        0.65,

    "ytick.major.width":
        0.65,

    "figure.dpi":
        160,

})


# =============================================================================
# 4. HELPERS
# =============================================================================

def require_columns(
    df,
    cols,
    name,
):

    missing = [
        c
        for c in cols
        if c not in df.columns
    ]

    if missing:

        raise KeyError(
            f"{name} missing columns: {missing}"
        )


def clean_x_axis(
    ax,
):

    ax.tick_params(
        axis="both",
        length=2.8,
        width=0.6,
        color=TEXT,
        labelcolor=TEXT,
    )

    ax.spines["left"].set_color(TEXT)
    ax.spines["bottom"].set_color(TEXT)

    ax.grid(
        axis="x",
        color=GRID,
        linewidth=0.5,
        alpha=0.65,
    )

    ax.set_axisbelow(True)


def clean_y_axis(
    ax,
):

    ax.tick_params(
        axis="both",
        length=2.8,
        width=0.6,
        color=TEXT,
        labelcolor=TEXT,
    )

    ax.spines["left"].set_color(TEXT)
    ax.spines["bottom"].set_color(TEXT)

    ax.grid(
        axis="y",
        color=GRID,
        linewidth=0.5,
        alpha=0.65,
    )

    ax.set_axisbelow(True)


def zero_vline(
    ax,
):

    ax.axvline(
        0,
        color="#777777",
        linestyle=":",
        linewidth=0.75,
        zorder=1,
    )


def zero_hline(
    ax,
):

    ax.axhline(
        0,
        color="#777777",
        linestyle=":",
        linewidth=0.75,
        zorder=1,
    )


def add_panel_label(
    ax,
    letter,
):

    ax.text(
        -0.105,
        1.02,
        letter,
        transform=ax.transAxes,
        fontsize=17,
        fontweight="bold",
        color=TEXT,
        ha="left",
        va="bottom",
    )


def add_title(
    ax,
    title,
    subtitle=None,
):

    ax.text(
        0,
        1.045,
        title,
        transform=ax.transAxes,
        fontsize=11.0,
        fontweight="normal",
        color=TEXT,
        ha="left",
        va="bottom",
    )

    if subtitle is not None:

        ax.text(
            0,
            1.005,
            subtitle,
            transform=ax.transAxes,
            fontsize=8.0,
            color=GRAY,
            ha="left",
            va="bottom",
        )


def asymmetric_xerr(
    effect,
    low,
    high,
):

    left = effect - low
    right = high - effect

    if left < -1e-12 or right < -1e-12:

        raise ValueError(
            f"Invalid CI: low={low}, effect={effect}, high={high}"
        )

    return [

        [max(left, 0.0)],

        [max(right, 0.0)],

    ]


# =============================================================================
# 5. VERIFY / FILTER FINAL RESULTS
# =============================================================================

if "status" in static.columns:

    static = static.loc[
        static["status"] == "estimated"
    ].copy()


if "status" in expertise.columns:

    expertise = expertise.loc[
        expertise["status"] == "estimated"
    ].copy()


require_columns(

    static,

    [
        "outcome",
        "effect_per_1sd",
        "ci_low_per_1sd",
        "ci_high_per_1sd",
    ],

    "static results",

)


require_columns(

    expertise,

    [
        "outcome",
        "collaborator_type",
        "effect_per_1sd",
        "ci_low_per_1sd",
        "ci_high_per_1sd",
    ],

    "within-type expertise results",

)


require_columns(

    dose,

    [
        "ai_bin",
        "mean_new_only_field_share",
        "mean_returning_only_field_share",
    ],

    "dose-response results",

)


require_columns(

    change,

    [
        "outcome",
        "half_year",
        "beta",
        "ci_low",
        "ci_high",
    ],

    "change-from-baseline results",

)


require_columns(

    placebo,

    [
        "outcome",
        "half_year",
        "beta",
        "ci_low",
        "ci_high",
    ],

    "placebo results",

)


# =============================================================================
# 6. FIGURE
# =============================================================================

fig, axes = plt.subplots(

    3,
    2,

    figsize=(15.2, 16.4),

)


# =============================================================================
# PANEL A
# =============================================================================
#
# Static collaboration composition.
#
# Returning vs New are the only two primary plotted estimates.
# Returning-share is intentionally NOT plotted here.
# =============================================================================

ax = axes[0, 0]

clean_x_axis(ax)

zero_vline(ax)


a_specs = [

    (
        "n_returning_collaborators",
        "Returning collaborators",
        RETURNING,
        "o",
    ),

    (
        "n_new_collaborators",
        "New collaborators",
        NEW,
        "s",
    ),

]


a_y = [
    1,
    0,
]


for y, (
    outcome,
    label,
    color,
    marker,
) in zip(
    a_y,
    a_specs,
):

    row = static.loc[
        static["outcome"] == outcome
    ]


    if len(row) != 1:

        raise ValueError(
            f"{outcome}: expected one row, found {len(row)}"
        )


    row = row.iloc[0]


    effect = (
        float(row["effect_per_1sd"])
        * 100.0
    )

    low = (
        float(row["ci_low_per_1sd"])
        * 100.0
    )

    high = (
        float(row["ci_high_per_1sd"])
        * 100.0
    )


    ax.errorbar(

        effect,

        y,

        xerr=asymmetric_xerr(
            effect,
            low,
            high,
        ),

        fmt=marker,

        markersize=5.8,

        color=color,

        ecolor=color,

        elinewidth=1.35,

        capsize=2.8,

        zorder=3,

    )


    ax.text(

        effect,

        y + 0.16,

        f"{effect:+.2f}%",

        ha="center",

        va="bottom",

        fontsize=8.3,

        fontweight="bold",

        color=color,

    )


ax.set_yticks(
    a_y
)

ax.set_yticklabels([

    "Returning collaborators",
    "New collaborators",

])


ax.set_xlim(
    -2.25,
    0.30,
)

ax.set_ylim(
    -0.55,
    1.55,
)


ax.set_xlabel(
    "Association per 1 SD AI\n(percent change)"
)


ax.legend(

    handles=[

        Line2D(

            [0],
            [0],

            marker="o",

            linestyle="None",

            color=RETURNING,

            markersize=5.5,

            label="Returning",

        ),

        Line2D(

            [0],
            [0],

            marker="s",

            linestyle="None",

            color=NEW,

            markersize=5.5,

            label="New",

        ),

    ],

    frameon=False,

    loc="lower right",

)


add_panel_label(
    ax,
    "a"
)

add_title(

    ax,

    "AI intensity is more strongly associated\n"
    "with fewer returning collaborators",

    "Author-paper panel; within-author models",

)


# =============================================================================
# PANEL B — FOUR INDEPENDENT MINI-PLOTS
# =============================================================================
#
# Each expertise outcome gets its OWN x-axis.
#
# Within each mini-plot:
#     Returning = green circle
#     New       = orange square
#
# This makes the comparison within each concept the visual unit.
# =============================================================================

big_ax = axes[0, 1]

# Turn the original panel into a blank container.
big_ax.set_axis_off()


# Panel-b title and subtitle.
big_ax.text(

    0.0,
    1.045,

    "AI intensity is associated with different expertise profiles\n"
    "for new and returning collaborators",

    transform=big_ax.transAxes,

    fontsize=11.0,

    color=TEXT,

    ha="left",

    va="bottom",

)


big_ax.text(

    0.0,
    1.005,

    "Author-collaborator panel; within-author models",

    transform=big_ax.transAxes,

    fontsize=8.0,

    color=GRAY,

    ha="left",

    va="bottom",

)
big_ax.text(

    0.50,
    -0.055,

    "Association per 1 SD AI (percentage points)",

    transform=big_ax.transAxes,

    fontsize=8.8,

    color=TEXT,

    ha="center",

    va="top",

)

# Mini-panel definitions.
b_specs = [

    (
        "returning_field_jaccard",
        "new_field_jaccard",
        "Field overlap",
    ),

    (
        "returning_subfield_jaccard",
        "new_subfield_jaccard",
        "Subfield overlap",
    ),

    (
        "returning_only_field_share",
        "new_only_field_share",
        "Outside focal field",
    ),

    (
        "returning_only_subfield_share",
        "new_only_subfield_share",
        "Outside focal subfield",
    ),

]


# 2 × 2 placement inside panel b.
mini_positions = [

    [0.00, 0.51, 0.46, 0.35],
    [0.54, 0.51, 0.46, 0.35],
    [0.00, 0.06, 0.46, 0.35],
    [0.54, 0.06, 0.46, 0.35],

]


for (
    ret_name,
    new_name,
    mini_title,
), pos in zip(
    b_specs,
    mini_positions,
):

    mini = big_ax.inset_axes(
        pos
    )


    clean_x_axis(
        mini
    )

    zero_vline(
        mini
    )


    ret = expertise.loc[
        expertise["outcome"] == ret_name
    ]


    new = expertise.loc[
        expertise["outcome"] == new_name
    ]


    if len(ret) != 1:

        raise ValueError(
            f"{ret_name}: expected one row, found {len(ret)}"
        )


    if len(new) != 1:

        raise ValueError(
            f"{new_name}: expected one row, found {len(new)}"
        )


    ret = ret.iloc[0]
    new = new.iloc[0]


    ret_effect = (
        float(ret["effect_per_1sd"])
        * 100.0
    )

    ret_low = (
        float(ret["ci_low_per_1sd"])
        * 100.0
    )

    ret_high = (
        float(ret["ci_high_per_1sd"])
        * 100.0
    )


    new_effect = (
        float(new["effect_per_1sd"])
        * 100.0
    )

    new_low = (
        float(new["ci_low_per_1sd"])
        * 100.0
    )

    new_high = (
        float(new["ci_high_per_1sd"])
        * 100.0
    )


    # Returning above; New below.
    ret_y = 0.56
    new_y = -0.56


    mini.errorbar(

        ret_effect,
        ret_y,

        xerr=asymmetric_xerr(
            ret_effect,
            ret_low,
            ret_high,
        ),

        fmt="o",

        markersize=4.7,

        color=RETURNING,

        ecolor=RETURNING,

        elinewidth=1.25,

        capsize=2.4,

        zorder=3,

    )


    mini.errorbar(

        new_effect,
        new_y,

        xerr=asymmetric_xerr(
            new_effect,
            new_low,
            new_high,
        ),

        fmt="s",

        markersize=4.7,

        color=NEW,

        ecolor=NEW,

        elinewidth=1.25,

        capsize=2.4,

        zorder=3,

    )


    # Mini title.
    mini.set_title(

        mini_title,

        fontsize=8.2,

        loc="left",

        pad=2,

        color=TEXT,

    )


    # Row labels.
    mini.set_yticks([
        ret_y,
        new_y,
    ])

    mini.set_yticklabels([
        "Returning",
        "New",
    ])


    # Dynamically determine x-axis from the full CI range.
    all_low = min(
        ret_low,
        new_low,
    )

    all_high = max(
        ret_high,
        new_high,
    )


    span = all_high - all_low


    if span <= 0:
        span = 0.10


    margin = max(
        0.12 * span,
        0.012,
    )


    x_left = all_low - margin
    x_right = all_high + margin


    # Make sure zero is visible.
    x_left = min(
        x_left,
        -margin,
    )

    x_right = max(
        x_right,
        margin,
    )


    mini.set_xlim(
        x_left,
        x_right,
    )


    mini.set_ylim(
        -1.05,
        1.05,
    )


    # Numeric value labels, placed toward the outside of each CI.
    # These are intentionally smaller than panel-a labels.
    for (

        effect,
        low,
        high,
        yy,
        color,

    ) in [

        (
            ret_effect,
            ret_low,
            ret_high,
            ret_y,
            RETURNING,
        ),

        (
            new_effect,
            new_low,
            new_high,
            new_y,
            NEW,
        ),

    ]:


        if effect >= 0:

            label_x = high

            ha = "left"

        else:

            label_x = low

            ha = "right"


        mini.text(

            label_x,

            yy + (
                0.30
                if yy < 0
                else 0.30
            ),

            f"{effect:+.2f}",

            fontsize=6.7,

            color=color,

            ha=ha,

            va="center",

        )


# Add the panel label outside the mini-grid.
big_ax.text(

    -0.105,
    1.02,

    "b",

    transform=big_ax.transAxes,

    fontsize=17,

    fontweight="bold",

    color=TEXT,

    ha="left",

    va="bottom",

)


# One compact legend for the whole b panel.
big_ax.legend(

    handles=[

        Line2D(

            [0],
            [0],

            marker="o",

            linestyle="None",

            color=RETURNING,

            markersize=4.8,

            label="Returning",

        ),

        Line2D(

            [0],
            [0],

            marker="s",

            linestyle="None",

            color=NEW,

            markersize=4.8,

            label="New",

        ),

    ],

    frameon=False,

    loc="lower right",

    bbox_to_anchor=(
        1.00,
        0.915,
    ),

    ncol=2,

)


# =============================================================================
# PANEL C
# =============================================================================
#
# Grouped bars: new vs returning.
# =============================================================================

ax = axes[1, 0]

clean_y_axis(ax)


dose_plot = dose.copy()

dose_plot["x"] = np.arange(
    len(dose_plot)
)


dose_plot["new_pct"] = (

    dose_plot[
        "mean_new_only_field_share"
    ]
    * 100.0

)


dose_plot["returning_pct"] = (

    dose_plot[
        "mean_returning_only_field_share"
    ]
    * 100.0

)


bar_width = 0.34

x = dose_plot["x"].to_numpy(
    dtype=float
)


bars_new = ax.bar(

    x - bar_width / 2,

    dose_plot["new_pct"],

    width=bar_width,

    color=NEW,

    label="New collaborators",

)


bars_returning = ax.bar(

    x + bar_width / 2,

    dose_plot["returning_pct"],

    width=bar_width,

    color=RETURNING,

    label="Returning collaborators",

)


for bars in [
    bars_new,
    bars_returning,
]:

    for bar in bars:

        h = bar.get_height()

        ax.text(

            bar.get_x()
            +
            bar.get_width() / 2,

            h + 0.45,

            f"{h:.1f}",

            ha="center",

            va="bottom",

            fontsize=7.1,

            color=TEXT,

        )


ax.set_xticks(
    x
)

ax.set_xticklabels(
    dose_plot["ai_bin"]
)

ax.set_ylim(
    0,
    65,
)


ax.set_ylabel(
    "Expertise outside focal field (%)"
)

ax.set_xlabel(
    "AI-writing intensity"
)


ax.legend(
    frameon=False,
    loc="upper left",
)


add_panel_label(
    ax,
    "c"
)

add_title(

    ax,

    "Outside-field expertise rises with AI intensity\n"
    "among new collaborators",

    "Author-paper panel; descriptive paper-level means",

)


# =============================================================================
# PANEL D
# =============================================================================
#
# Longitudinal new / returning decomposition.
# =============================================================================

ax = axes[1, 1]

clean_y_axis(ax)


post_periods = [

    "2023H1",
    "2023H2",
    "2024H1",
    "2024H2",
    "2025H1",
    "2025H2",

]


d_plot = change.loc[

    change[
        "half_year"
    ].isin(
        post_periods
    )

].copy()


period_map = {

    p: i

    for i, p in enumerate(
        post_periods
    )

}


d_plot["x_center"] = (

    d_plot[
        "half_year"
    ]
    .map(
        period_map
    )

)


DODGE = 0.075


for outcome, color, marker, offset in [

    (
        "New collaborators",
        NEW,
        "s",
        -DODGE,
    ),

    (
        "Returning collaborators",
        RETURNING,
        "o",
        DODGE,
    ),

]:

    q = d_plot.loc[

        d_plot["outcome"]
        ==
        outcome

    ].sort_values(
        "x_center"
    )


    xvals = (

        q[
            "x_center"
        ].to_numpy(float)

        +
        offset

    )


    yvals = q[
        "beta"
    ].to_numpy(float)


    low = q[
        "ci_low"
    ].to_numpy(float)


    high = q[
        "ci_high"
    ].to_numpy(float)


    ax.errorbar(

        xvals,

        yvals,

        yerr=[

            yvals - low,

            high - yvals,

        ],

        fmt=marker,

        color=color,

        ecolor=color,

        markersize=5.0,

        linewidth=1.2,

        elinewidth=1.2,

        capsize=2.4,

        label=(
            "New collaborators"
            if outcome == "New collaborators"
            else
            "Returning collaborators"
        ),

        zorder=3,

    )


    ax.plot(

        xvals,

        yvals,

        color=color,

        linewidth=1.2,

        alpha=0.9,

        zorder=2,

    )


zero_hline(
    ax
)


ax.set_xticks(
    np.arange(len(post_periods))
)

ax.set_xticklabels([

    "23H1",
    "23H2",
    "24H1",
    "24H2",
    "25H1",
    "25H2",

])


ax.set_ylim(
    -0.82,
    0.12,
)


ax.set_ylabel(
    "Change from author's pre-2023 baseline"
)

ax.set_xlabel(
    "Post-2023 half-year"
)


ax.legend(
    frameon=False,
    loc="lower left",
)


add_panel_label(
    ax,
    "d"
)

add_title(

    ax,

    "Returning collaboration contracts earlier;\n"
    "the decline in new ties emerges later",

    "Author panel; change from pre-2023 baseline, EB-weighted",

)


# =============================================================================
# PANEL E
# =============================================================================
#
# Overall collaboration.
# =============================================================================

ax = axes[2, 0]

clean_y_axis(ax)


e = change.loc[

    (
        change["outcome"]
        ==
        "Total collaborators"
    )

    &

    change[
        "half_year"
    ].isin(
        post_periods
    )

].copy()


e["x"] = (

    e[
        "half_year"
    ].map(
        period_map
    )

)


ax.errorbar(

    e["x"],

    e["beta"],

    yerr=[

        e["beta"] - e["ci_low"],

        e["ci_high"] - e["beta"],

    ],

    fmt="o",

    color=TOTAL,

    ecolor=TOTAL,

    markersize=5.4,

    linewidth=1.5,

    elinewidth=1.25,

    capsize=2.8,

)


ax.plot(

    e["x"],

    e["beta"],

    color=TOTAL,

    linewidth=1.5,

)


zero_hline(
    ax
)


ax.set_xticks(
    np.arange(len(post_periods))
)

ax.set_xticklabels([

    "23H1",
    "23H2",
    "24H1",
    "24H2",
    "25H1",
    "25H2",

])


ax.set_ylim(
    -1.05,
    0.30,
)


ax.set_ylabel(
    "Change from author's pre-2023 baseline"
)

ax.set_xlabel(
    "Post-2023 half-year"
)


last = e.sort_values(
    "x"
).iloc[-1]


ax.text(

    last["x"] + 0.09,

    last["beta"],

    f"{last['beta']:.2f}",

    color=TOTAL,

    fontsize=8.2,

    va="center",

)


add_panel_label(
    ax,
    "e"
)

add_title(

    ax,

    "Overall collaboration also declines after 2023",

    "Author panel; change from pre-2023 baseline, EB-weighted",

)


# =============================================================================
# PANEL F
# =============================================================================
#
# Pre-2023 placebo.
#
# 2022H1 = open marker
# 2022H2 = filled marker
# =============================================================================

ax = axes[2, 1]

clean_x_axis(ax)

zero_vline(ax)


f_specs = [

    (
        "New collaborators",
        NEW,
        2,
    ),

    (
        "Returning collaborators",
        RETURNING,
        1,
    ),

    (
        "Total collaborators",
        TOTAL,
        0,
    ),

]


for outcome, color, y in f_specs:

    for period in [
        "2022H1",
        "2022H2",
    ]:

        row = placebo.loc[

            (
                placebo["outcome"]
                ==
                outcome
            )

            &

            (
                placebo["half_year"]
                ==
                period
            )

        ]


        if len(row) != 1:

            raise ValueError(

                f"Expected one placebo row for "
                f"{outcome} / {period}"

            )


        row = row.iloc[0]


        effect = float(
            row["beta"]
        )

        low = float(
            row["ci_low"]
        )

        high = float(
            row["ci_high"]
        )


        yy = (

            y - 0.09

            if period == "2022H1"

            else

            y + 0.09

        )


        facecolor = (

            "white"

            if period == "2022H1"

            else

            color

        )


        ax.errorbar(

            effect,

            yy,

            xerr=asymmetric_xerr(

                effect,
                low,
                high,

            ),

            fmt="o",

            markerfacecolor=facecolor,

            markeredgecolor=color,

            ecolor=color,

            color=color,

            markersize=5.0,

            capsize=2.6,

            elinewidth=1.2,

            zorder=3,

        )


ax.set_yticks([
    2,
    1,
    0,
])

ax.set_yticklabels([

    "New\ncollaborators",
    "Returning\ncollaborators",
    "Total\ncollaborators",

])


ax.set_xlim(
    -0.50,
    0.72,
)

ax.set_ylim(
    -0.62,
    2.62,
)


ax.set_xlabel(

    "Placebo change from 2021 baseline\n"
    "(2022 half-years)"

)


ax.legend(

    handles=[

        Line2D(

            [0],
            [0],

            marker="o",

            markerfacecolor="white",

            markeredgecolor=TEXT,

            linestyle="None",

            markersize=5,

            label="2022H1",

        ),

        Line2D(

            [0],
            [0],

            marker="o",

            markerfacecolor=TEXT,

            markeredgecolor=TEXT,

            linestyle="None",

            markersize=5,

            label="2022H2",

        ),

    ],

    frameon=False,

    loc="upper right",

)


add_panel_label(
    ax,
    "f"
)

add_title(

    ax,

    "No comparable collaboration shift appears\n"
    "in the pre-2023 placebo",

    "Author panel; change from 2021 baseline, EB-weighted",

)


# =============================================================================
# 7. FIGURE-WIDE TITLE
# =============================================================================

fig.suptitle(

    "AI-associated writing is linked to a reorganization of scientific collaboration",

    x=0.5,

    y=0.995,

    fontsize=15.5,

    fontweight="bold",

    color=TEXT,

)


# =============================================================================
# 8. FIGURE-WIDE NOTE
# =============================================================================

fig.text(

    0.5,

    0.018,

    "Notes: Panels a–b show adjusted estimates per 1 SD increase in residualized AI-writing intensity "
    "with 95% confidence intervals. Static models include focal-author, publication-month and subfield "
    "fixed effects. \n Panel c shows descriptive paper-level means by AI-writing-intensity bin. Panels d–e "
    "use entropy-balanced author weights and show changes relative to each author's observed pre-2023 "
    "baseline (2021H1–2022H2). \n Panel f reports a pre-2023 placebo using the 2021 baseline "
    "(2021H1–2021H2) and 2022 pseudo-post periods. Standard errors are clustered by focal author. "
    "See text for full model specifications.",

    ha="center",

    va="bottom",

    fontsize=7.25,

    color=GRAY,

)


# =============================================================================
# 9. LAYOUT
# =============================================================================

fig.subplots_adjust(
    left=0.16, right=0.975, top=0.91, bottom=0.095,
    wspace=0.40, hspace=0.50
)


# =============================================================================
# 10. SAVE
# =============================================================================

PNG_FILE = (
    "fig4_revised_rendering_v2.png"
)

SVG_FILE = (
    "fig4_revised_rendering_v2.svg"
)

PDF_FILE = (
    "fig4_revised_rendering_v2.pdf"
)


fig.savefig(

    PNG_FILE,

    dpi=400,

    bbox_inches="tight",

)


fig.savefig(

    SVG_FILE,

    bbox_inches="tight",

)


fig.savefig(

    PDF_FILE,

    bbox_inches="tight",

)


plt.show()


print(
    "\nSaved:"
)

print(
    f"  {PNG_FILE}"
)

print(
    f"  {SVG_FILE}"
)

print(
    f"  {PDF_FILE}"
)

## Manuscript Figure 4 — Selective knowledge transmission and recombination

Final loss composition, absolute retention, historical atypicality, and the model specifications directly displayed in the six-panel main figure.

### Loss composition (main figure 4 panels a–b)

In [ ]:
# =============================================================================
# FIGURE 5 — LOSS COMPOSITION ANALYSIS
#
# PRIMARY QUESTION
# ----------------
# Among the predecessor knowledge embedded in a focal paper, which kinds of
# knowledge are disproportionately represented among the knowledge that is
# NOT retained by future external papers?
#
#
# EXISTING BIGQUERY TABLES
# ------------------------
#
#   fig5_loss_composition_boundary_12m
#   fig5_loss_composition_boundary_24m
#   fig5_loss_composition_distinctive_12m
#   fig5_loss_composition_distinctive_24m
#
#
# IMPORTANT
# ---------
# These BigQuery tables are category-level:
#
#   focal paper × knowledge category
#
# The notebook loads ONE TABLE AT A TIME and immediately converts it to a
# focal-paper-level dataset.
#
#
# PRIMARY ESTIMAND
# ----------------
#
# loss_overrepresentation =
#
#       share of all lost predecessor opportunities belonging to category
#
#       MINUS
#
#       share of all predecessor opportunities belonging to category
#
#
# Thus:
#
#   > 0 = category disproportionately represented among lost knowledge
#   < 0 = category underrepresented among lost knowledge
#
#
# REGRESSION
# ----------
#
# loss_overrepresentation_p
#
#   = beta * AI_p
#     + author × subfield FE
#     + publication-month FE
#     + log(reference count)
#     + error
#
#
# Standard errors:
#   clustered by focal author
#
#
# PRIMARY:
#   12m
#
# ROBUSTNESS:
#   24m
#
# =============================================================================


# =============================================================================
# 0. IMPORTS
# =============================================================================

import gc
import numpy as np
import pandas as pd
import statsmodels.api as sm

from google.cloud import bigquery


# =============================================================================
# 1. BIGQUERY CLIENT
# =============================================================================

try:
    client
except NameError:

    client = bigquery.Client(
        project="aiwriting-488923"
    )


PROJECT = "aiwriting-488923"
DATASET = "redo"


# =============================================================================
# 2. TABLE NAMES
# =============================================================================

TABLES = {

    "boundary_12m":
        f"{PROJECT}.{DATASET}.fig5_loss_composition_boundary_12m",

    "boundary_24m":
        f"{PROJECT}.{DATASET}.fig5_loss_composition_boundary_24m",

    "distinctive_12m":
        f"{PROJECT}.{DATASET}.fig5_loss_composition_distinctive_12m",

    "distinctive_24m":
        f"{PROJECT}.{DATASET}.fig5_loss_composition_distinctive_24m",

}


# =============================================================================
# 3. LOAD ONLY REQUIRED COLUMNS
#
# IMPORTANT:
#   boundary_group and topic_redundancy_group MUST be included.
# =============================================================================

BOUNDARY_COLUMNS = [

    "focal_openalex_id",

    "focal_first_author_id",

    "ai_score",

    "accepted_month",

    "accepted_year",

    "focal_subfield_id",

    "focal_reference_count",

    "boundary_group",

    "reference_opportunities",

    "loss_events",

    "total_opportunities",

    "total_losses",

]


DISTINCTIVE_COLUMNS = [

    "focal_openalex_id",

    "focal_first_author_id",

    "ai_score",

    "accepted_month",

    "accepted_year",

    "focal_subfield_id",

    "focal_reference_count",

    "topic_redundancy_group",

    "reference_opportunities",

    "loss_events",

    "total_opportunities",

    "total_losses",

]


# =============================================================================
# 4. SCHEMA CHECK
# =============================================================================

def check_schema(
    table_name,
    required_columns
):

    table = client.get_table(
        table_name
    )

    actual = {

        field.name

        for field in table.schema

    }


    missing = [

        c

        for c in required_columns

        if c not in actual

    ]


    if missing:

        raise KeyError(

            f"\n{table_name} is missing:\n"
            +
            "\n".join(
                missing
            )

        )


    print(
        f"Schema PASS: {table_name}"
    )


check_schema(
    TABLES["boundary_12m"],
    BOUNDARY_COLUMNS
)

check_schema(
    TABLES["boundary_24m"],
    BOUNDARY_COLUMNS
)

check_schema(
    TABLES["distinctive_12m"],
    DISTINCTIVE_COLUMNS
)

check_schema(
    TABLES["distinctive_24m"],
    DISTINCTIVE_COLUMNS
)


# =============================================================================
# 5. LOAD FUNCTION
# =============================================================================

def load_loss_table(
    table_name,
    columns
):

    sql = f"""
    SELECT
        {", ".join(columns)}
    FROM `{table_name}`
    """


    print(
        "\nLOADING:"
    )

    print(
        table_name
    )


    df = client.query(
        sql
    ).to_dataframe()


    print(
        "Rows:",
        f"{len(df):,}"
    )

    print(
        "Focal papers:",
        f"{df['focal_openalex_id'].nunique():,}"
    )

    print(
        "Focal authors:",
        f"{df['focal_first_author_id'].nunique():,}"
    )


    return df


# =============================================================================
# 6. PREPARE CATEGORY TABLE
# =============================================================================

def prepare_loss_table(

    df,
    category_column

):

    d = df.copy()


    # -------------------------------------------------------------------------
    # Numeric.
    # -------------------------------------------------------------------------

    for col in [

        "ai_score",
        "focal_reference_count",
        "reference_opportunities",
        "loss_events",
        "total_opportunities",
        "total_losses",

    ]:

        d[col] = pd.to_numeric(

            d[col],

            errors="coerce"

        )


    # -------------------------------------------------------------------------
    # Date.
    # -------------------------------------------------------------------------

    d["accepted_month"] = pd.to_datetime(

        d["accepted_month"],

        errors="coerce"

    )


    # -------------------------------------------------------------------------
    # Reconstruct loss share.
    # -------------------------------------------------------------------------

    d["loss_share"] = (

        d["loss_events"]

        /

        d["total_losses"]

    )


    # -------------------------------------------------------------------------
    # Reconstruct initial reference/opportunity share.
    # -------------------------------------------------------------------------

    d["reference_share"] = (

        d["reference_opportunities"]

        /

        d["total_opportunities"]

    )


    # -------------------------------------------------------------------------
    # Reconstruct the loss-overrepresentation outcome.
    #
    # This avoids relying on a pre-computed value if the BigQuery table was
    # created with an earlier version.
    # -------------------------------------------------------------------------

    d["loss_overrepresentation"] = (

        d["loss_share"]

        -

        d["reference_share"]

    )


    # -------------------------------------------------------------------------
    # Validity.
    # -------------------------------------------------------------------------

    valid = (

        np.isfinite(
            d["ai_score"].to_numpy(
                dtype=float
            )
        )

        &

        np.isfinite(
            d["loss_share"].to_numpy(
                dtype=float
            )
        )

        &

        np.isfinite(
            d["reference_share"].to_numpy(
                dtype=float
            )
        )

        &

        np.isfinite(
            d["loss_overrepresentation"].to_numpy(
                dtype=float
            )
        )

        &

        (
            d["total_losses"]
            >
            0
        )

        &

        (
            d["total_opportunities"]
            >
            0
        )

        &

        d[
            category_column
        ].notna()

        &

        d[
            "focal_first_author_id"
        ].notna()

        &

        d[
            "focal_subfield_id"
        ].notna()

        &

        d[
            "accepted_month"
        ].notna()

    )


    d = d.loc[
        valid
    ].copy()


    # -------------------------------------------------------------------------
    # Author × subfield identifier.
    # -------------------------------------------------------------------------

    d["author_subfield"] = (

        d[
            "focal_first_author_id"
        ].astype(str)

        +

        "::"

        +

        d[
            "focal_subfield_id"
        ].astype(str)

    )


    # -------------------------------------------------------------------------
    # Compact dtypes.
    # -------------------------------------------------------------------------

    d["ai_score"] = d[
        "ai_score"
    ].astype(
        "float32"
    )


    d["loss_overrepresentation"] = d[
        "loss_overrepresentation"
    ].astype(
        "float32"
    )


    d["total_losses"] = d[
        "total_losses"
    ].astype(
        "float64"
    )


    d["total_opportunities"] = d[
        "total_opportunities"
    ].astype(
        "float64"
    )


    return d


# =============================================================================
# 7. COLLAPSE TO ONE OBSERVATION PER FOCAL PAPER
#
# This is the key step.
#
# We use one focal-paper row containing:
#
#   - total losses
#   - category loss composition
#
# No reference-level data remain after this function.
# =============================================================================

def make_paper_loss_composition(

    d,
    category_column

):

    # -------------------------------------------------------------------------
    # Confirm there is at most one row per focal paper × category.
    # -------------------------------------------------------------------------

    dup = (

        d

        .groupby(

            [
                "focal_openalex_id",
                category_column,
            ],

            observed=True

        )

        .size()

    )


    if (
        dup > 1
    ).any():

        raise ValueError(

            "Duplicate focal-paper × category rows detected."

        )


    # -------------------------------------------------------------------------
    # Pivot category loss-overrepresentation.
    # -------------------------------------------------------------------------

    outcome_wide = (

        d

        .pivot(

            index="focal_openalex_id",

            columns=category_column,

            values="loss_overrepresentation"

        )

        .reset_index()

    )


    # -------------------------------------------------------------------------
    # Pivot category loss shares.
    # -------------------------------------------------------------------------

    loss_share_wide = (

        d

        .pivot(

            index="focal_openalex_id",

            columns=category_column,

            values="loss_share"

        )

        .reset_index()

    )


    # -------------------------------------------------------------------------
    # Pivot category reference shares.
    # -------------------------------------------------------------------------

    reference_share_wide = (

        d

        .pivot(

            index="focal_openalex_id",

            columns=category_column,

            values="reference_share"

        )

        .reset_index()

    )


    # -------------------------------------------------------------------------
    # Paper characteristics.
    # -------------------------------------------------------------------------

    paper_info = (

        d

        .sort_values(
            "focal_openalex_id"
        )

        .drop_duplicates(

            "focal_openalex_id"

        )

        [

            [

                "focal_openalex_id",

                "focal_first_author_id",

                "ai_score",

                "accepted_month",

                "accepted_year",

                "focal_subfield_id",

                "focal_reference_count",

                "author_subfield",

            ]

        ]

    )


    # -------------------------------------------------------------------------
    # Merge the three small paper-level pieces.
    # -------------------------------------------------------------------------

    out = paper_info.merge(

        outcome_wide,

        on="focal_openalex_id",

        how="left",

        validate="one_to_one"

    )


    # This merge is also one-to-one because outcome_wide has one row per paper.
    #
    # Rename category columns later; first preserve them in a separate frame.
    # -------------------------------------------------------------------------

    loss_share_wide = loss_share_wide.rename(

        columns={

            c:
                f"loss_share__{c}"

            for c in loss_share_wide.columns

            if c != "focal_openalex_id"

        }

    )


    reference_share_wide = reference_share_wide.rename(

        columns={

            c:
                f"reference_share__{c}"

            for c in reference_share_wide.columns

            if c != "focal_openalex_id"

        }

    )


    out = out.merge(

        loss_share_wide,

        on="focal_openalex_id",

        how="left",

        validate="one_to_one"

    )


    out = out.merge(

        reference_share_wide,

        on="focal_openalex_id",

        how="left",

        validate="one_to_one"

    )


    return out


# =============================================================================
# 8. TWO-WAY FWL
#
# Absorb:
#
#   author × subfield FE
#   publication month FE
#
# without building a massive dummy matrix.
# =============================================================================

def weighted_group_demean(

    x,
    groups,
    weights

):

    x = np.asarray(
        x,
        dtype=np.float64
    )

    groups = np.asarray(
        groups,
        dtype=np.int64
    )

    weights = np.asarray(
        weights,
        dtype=np.float64
    )


    numerator = np.bincount(

        groups,

        weights=(
            weights
            *
            x
        )

    )


    denominator = np.bincount(

        groups,

        weights=weights

    )


    means = np.divide(

        numerator,

        denominator,

        out=np.zeros_like(
            numerator
        ),

        where=denominator > 0

    )


    return (

        x

        -

        means[
            groups
        ]

    )


def two_way_fwl(

    y,
    X,
    group_a,
    group_b,
    weights,

    tolerance=1e-10,
    max_iterations=500

):

    y = np.asarray(
        y,
        dtype=np.float64
    )


    X = np.asarray(
        X,
        dtype=np.float64
    )


    if X.ndim == 1:

        X = X.reshape(
            -1,
            1
        )


    M = np.column_stack(

        [
            y,
            X,
        ]

    )


    for iteration in range(

        1,
        max_iterations + 1

    ):

        old = M.copy()


        # First FE.
        for j in range(
            M.shape[1]
        ):

            M[:, j] = weighted_group_demean(

                M[:, j],

                group_a,

                weights

            )


        # Second FE.
        for j in range(
            M.shape[1]
        ):

            M[:, j] = weighted_group_demean(

                M[:, j],

                group_b,

                weights

            )


        change = float(

            np.max(

                np.abs(
                    M - old
                )

            )

        )


        if change < tolerance:

            break


    return (

        M[:, 0],

        M[:, 1:],

        iteration,

        change

    )


# =============================================================================
# 9. PAPER-LEVEL MODEL
# =============================================================================

def estimate_paper_model(

    d,
    outcome_column,
    label,

):

    x = d.copy()


    # -------------------------------------------------------------------------
    # Drop missing outcome.
    # -------------------------------------------------------------------------

    x = x.loc[

        x[
            outcome_column
        ].notna()

    ].copy()


    # -------------------------------------------------------------------------
    # Factor FE.
    # -------------------------------------------------------------------------

    author_fe, _ = pd.factorize(

        x[
            "author_subfield"
        ],

        sort=False

    )


    month_fe, _ = pd.factorize(

        x[
            "accepted_month"
        ]
        .dt
        .to_period("M")
        .astype(str),

        sort=False

    )


    # -------------------------------------------------------------------------
    # Outcome.
    # -------------------------------------------------------------------------

    y = x[
        outcome_column
    ].to_numpy(
        dtype=float
    )


    # -------------------------------------------------------------------------
    # Treatment.
    # -------------------------------------------------------------------------

    ai = x[
        "ai_score"
    ].to_numpy(
        dtype=float
    )


    # -------------------------------------------------------------------------
    # Reference-list size control.
    # -------------------------------------------------------------------------

    log_refs = np.log1p(

        x[
            "focal_reference_count"
        ]
        .clip(
            lower=0
        )
        .to_numpy(
            dtype=float
        )

    )


    X = np.column_stack(

        [

            ai,

            log_refs,

        ]

    )


    # -------------------------------------------------------------------------
    # Primary model uses equal focal-paper weight.
    #
    # Every focal paper contributes one observation.
    # -------------------------------------------------------------------------

    weights = np.ones(

        len(x),

        dtype=float

    )


    y_r, X_r, fwl_iterations, fwl_change = two_way_fwl(

        y,

        X,

        author_fe,

        month_fe,

        weights,

    )


    # -------------------------------------------------------------------------
    # Standardize residualized AI.
    # -------------------------------------------------------------------------

    ai_r = X_r[:, 0]


    ai_mean = ai_r.mean()


    ai_sd = np.sqrt(

        np.mean(

            (
                ai_r
                -
                ai_mean
            ) ** 2

        )

    )


    if (
        not np.isfinite(ai_sd)
        or
        ai_sd <= 0
    ):

        raise RuntimeError(

            f"{label}: residualized AI has invalid SD."

        )


    X_r[:, 0] /= ai_sd


    # -------------------------------------------------------------------------
    # WLS = OLS because primary weight is one.
    # -------------------------------------------------------------------------

    model = sm.WLS(

        y_r,

        X_r,

        weights=weights

    ).fit(

        cov_type="cluster",

        cov_kwds={

            "groups":
                author_fe

        }

    )


    beta = float(
        model.params[0]
    )


    se = float(
        model.bse[0]
    )


    p = float(
        model.pvalues[0]
    )


    ci_low = (

        beta

        -

        1.96
        *
        se

    )


    ci_high = (

        beta

        +

        1.96
        *
        se

    )


    result = pd.DataFrame([{

        "analysis":
            label,

        "effect":
            beta,

        "se":
            se,

        "p":
            p,

        "ci_low":
            ci_low,

        "ci_high":
            ci_high,

        "effect_pp":
            beta * 100.0,

        "ci_low_pp":
            ci_low * 100.0,

        "ci_high_pp":
            ci_high * 100.0,

    }])


    diagnostics = pd.DataFrame([{

        "analysis":
            label,

        "n_focal_papers":
            x[
                "focal_openalex_id"
            ].nunique(),

        "n_focal_authors":
            x[
                "focal_first_author_id"
            ].nunique(),

        "n_author_subfield_FE":
            x[
                "author_subfield"
            ].nunique(),

        "n_month_FE":
            x[
                "accepted_month"
            ].nunique(),

        "mean_outcome":
            x[
                outcome_column
            ].mean(),

        "sd_outcome":
            x[
                outcome_column
            ].std(),

        "residualized_ai_sd":
            ai_sd,

        "fwl_iterations":
            fwl_iterations,

        "fwl_final_change":
            fwl_change,

    }])


    return (
        result,
        diagnostics
    )


# =============================================================================
# 10. BOUNDARY 12m
# =============================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "BOUNDARY-SPANNING LOSS COMPOSITION — 12m PRIMARY"
)

print(
    "=" * 80
)


boundary12 = load_loss_table(

    TABLES[
        "boundary_12m"
    ],

    BOUNDARY_COLUMNS

)


boundary12 = prepare_loss_table(

    boundary12,

    "boundary_group"

)


boundary12_paper = make_paper_loss_composition(

    boundary12,

    "boundary_group"

)


# Require all three boundary categories.
boundary12_paper = boundary12_paper.loc[

    boundary12_paper[

        [

            "Cross-field",
            "Cross-subfield; same field",
            "Same subfield",

        ]

    ]

    .notna()
    .all(
        axis=1
    )

].copy()


# -------------------------------------------------------------------------
# Main outcomes.
# -------------------------------------------------------------------------

boundary12_crossfield_result, boundary12_crossfield_diag = (

    estimate_paper_model(

        boundary12_paper,

        "Cross-field",

        "Boundary loss composition: cross-field — 12m"

    )

)


boundary12_crosssub_result, boundary12_crosssub_diag = (

    estimate_paper_model(

        boundary12_paper,

        "Cross-subfield; same field",

        "Boundary loss composition: cross-subfield — 12m"

    )

)


print(
    "\nCross-field"
)

display(
    boundary12_crossfield_result
)


print(
    "\nCross-subfield"
)

display(
    boundary12_crosssub_result
)


# -------------------------------------------------------------------------
# Save.
# -------------------------------------------------------------------------

boundary12_results = pd.concat(

    [

        boundary12_crossfield_result,

        boundary12_crosssub_result,

    ],

    ignore_index=True

)


boundary12_diagnostics = pd.concat(

    [

        boundary12_crossfield_diag,

        boundary12_crosssub_diag,

    ],

    ignore_index=True

)


boundary12_results.to_csv(

    "fig5_loss_boundary_12m_results.csv",

    index=False

)


boundary12_diagnostics.to_csv(

    "fig5_loss_boundary_12m_diagnostics.csv",

    index=False

)


del boundary12
del boundary12_paper

gc.collect()


# =============================================================================
# 11. BOUNDARY 24m
# =============================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "BOUNDARY-SPANNING LOSS COMPOSITION — 24m"
)

print(
    "=" * 80
)


boundary24 = load_loss_table(

    TABLES[
        "boundary_24m"
    ],

    BOUNDARY_COLUMNS

)


boundary24 = prepare_loss_table(

    boundary24,

    "boundary_group"

)


boundary24_paper = make_paper_loss_composition(

    boundary24,

    "boundary_group"

)


boundary24_paper = boundary24_paper.loc[

    boundary24_paper[

        [

            "Cross-field",
            "Cross-subfield; same field",
            "Same subfield",

        ]

    ]

    .notna()
    .all(
        axis=1
    )

].copy()


boundary24_crossfield_result, boundary24_crossfield_diag = (

    estimate_paper_model(

        boundary24_paper,

        "Cross-field",

        "Boundary loss composition: cross-field — 24m"

    )

)


boundary24_crosssub_result, boundary24_crosssub_diag = (

    estimate_paper_model(

        boundary24_paper,

        "Cross-subfield; same field",

        "Boundary loss composition: cross-subfield — 24m"

    )

)


print(
    "\nCross-field"
)

display(
    boundary24_crossfield_result
)


print(
    "\nCross-subfield"
)

display(
    boundary24_crosssub_result
)


boundary24_results = pd.concat(

    [

        boundary24_crossfield_result,

        boundary24_crosssub_result,

    ],

    ignore_index=True

)


boundary24_diagnostics = pd.concat(

    [

        boundary24_crossfield_diag,

        boundary24_crosssub_diag,

    ],

    ignore_index=True

)


boundary24_results.to_csv(

    "fig5_loss_boundary_24m_results.csv",

    index=False

)


boundary24_diagnostics.to_csv(

    "fig5_loss_boundary_24m_diagnostics.csv",

    index=False

)


del boundary24
del boundary24_paper

gc.collect()


# =============================================================================
# 12. DISTINCTIVENESS 12m
# =============================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "DISTINCTIVE VS REDUNDANT LOSS COMPOSITION — 12m PRIMARY"
)

print(
    "=" * 80
)


distinctive12 = load_loss_table(

    TABLES[
        "distinctive_12m"
    ],

    DISTINCTIVE_COLUMNS

)


distinctive12 = prepare_loss_table(

    distinctive12,

    "topic_redundancy_group"

)


distinctive12_paper = make_paper_loss_composition(

    distinctive12,

    "topic_redundancy_group"

)


distinctive12_paper = distinctive12_paper.loc[

    distinctive12_paper[

        [

            "High redundancy (4+)",
            "Low redundancy (2-3)",
            "Unique topic",

        ]

    ]

    .notna()
    .all(
        axis=1
    )

].copy()


unique12_result, unique12_diag = (

    estimate_paper_model(

        distinctive12_paper,

        "Unique topic",

        "Loss composition: unique topic — 12m"

    )

)


low12_result, low12_diag = (

    estimate_paper_model(

        distinctive12_paper,

        "Low redundancy (2-3)",

        "Loss composition: low redundancy — 12m"

    )

)


print(
    "\nUnique topic"
)

display(
    unique12_result
)


print(
    "\nLow redundancy"
)

display(
    low12_result
)


distinctive12_results = pd.concat(

    [

        unique12_result,

        low12_result,

    ],

    ignore_index=True

)


distinctive12_diagnostics = pd.concat(

    [

        unique12_diag,

        low12_diag,

    ],

    ignore_index=True

)


distinctive12_results.to_csv(

    "fig5_loss_distinctive_12m_results.csv",

    index=False

)


distinctive12_diagnostics.to_csv(

    "fig5_loss_distinctive_12m_diagnostics.csv",

    index=False

)


del distinctive12
del distinctive12_paper

gc.collect()


# =============================================================================
# 13. DISTINCTIVENESS 24m
# =============================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "DISTINCTIVE VS REDUNDANT LOSS COMPOSITION — 24m"
)

print(
    "=" * 80
)


distinctive24 = load_loss_table(

    TABLES[
        "distinctive_24m"
    ],

    DISTINCTIVE_COLUMNS

)


distinctive24 = prepare_loss_table(

    distinctive24,

    "topic_redundancy_group"

)


distinctive24_paper = make_paper_loss_composition(

    distinctive24,

    "topic_redundancy_group"

)


distinctive24_paper = distinctive24_paper.loc[

    distinctive24_paper[

        [

            "High redundancy (4+)",
            "Low redundancy (2-3)",
            "Unique topic",

        ]

    ]

    .notna()
    .all(
        axis=1
    )

].copy()


unique24_result, unique24_diag = (

    estimate_paper_model(

        distinctive24_paper,

        "Unique topic",

        "Loss composition: unique topic — 24m"

    )

)


low24_result, low24_diag = (

    estimate_paper_model(

        distinctive24_paper,

        "Low redundancy (2-3)",

        "Loss composition: low redundancy — 24m"

    )

)


print(
    "\nUnique topic"
)

display(
    unique24_result
)


print(
    "\nLow redundancy"
)

display(
    low24_result
)


distinctive24_results = pd.concat(

    [

        unique24_result,

        low24_result,

    ],

    ignore_index=True

)


distinctive24_diagnostics = pd.concat(

    [

        unique24_diag,

        low24_diag,

    ],

    ignore_index=True

)


distinctive24_results.to_csv(

    "fig5_loss_distinctive_24m_results.csv",

    index=False

)


distinctive24_diagnostics.to_csv(

    "fig5_loss_distinctive_24m_diagnostics.csv",

    index=False

)


del distinctive24
del distinctive24_paper

gc.collect()


# =============================================================================
# 14. COMBINED RESULTS
# =============================================================================

final_results = pd.concat(

    [

        boundary12_results,
        boundary24_results,

        distinctive12_results,
        distinctive24_results,

    ],

    ignore_index=True

)


final_diagnostics = pd.concat(

    [

        boundary12_diagnostics,
        boundary24_diagnostics,

        distinctive12_diagnostics,
        distinctive24_diagnostics,

    ],

    ignore_index=True

)


# =============================================================================
# 15. SAVE FINAL FILES
# =============================================================================

final_results.to_csv(

    "fig5_loss_composition_results.csv",

    index=False

)


final_diagnostics.to_csv(

    "fig5_loss_composition_diagnostics.csv",

    index=False

)


# =============================================================================
# 16. FINAL DISPLAY
# =============================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "FIGURE 5 — LOSS COMPOSITION RESULTS"
)

print(
    "=" * 80
)


display(

    final_results[

        [

            "analysis",
            "weighted",
            "effect_pp",
            "ci_low_pp",
            "ci_high_pp",
            "p",

        ]

    ]

)


print(
    "\nDIAGNOSTICS"
)

display(
    final_diagnostics
)


# =============================================================================
# 17. FINAL CLEANUP
# =============================================================================

del final_results
del final_diagnostics

gc.collect()


print(
    "\nFIGURE 5 LOSS-COMPOSITION ANALYSIS COMPLETE."
)

### Absolute retention (main figure 4 panel c)

In [ ]:
# =============================================================================
# FIGURE 5 — FINAL VALIDATION
#
# ABSOLUTE RETENTION + LOSS COMPOSITION
#
# PRIMARY:
#     12m
#
# ROBUSTNESS:
#     24m
#
# =============================================================================


import gc
import numpy as np
import pandas as pd
import statsmodels.api as sm

from google.cloud import bigquery


# =============================================================================
# 1. CLIENT
# =============================================================================

try:
    client

except NameError:

    client = bigquery.Client(
        project="aiwriting-488923"
    )


PROJECT = "aiwriting-488923"
DATASET = "redo"


TABLES = {

    "boundary_12m":
        f"{PROJECT}.{DATASET}.fig5_final_boundary_12m",

    "boundary_24m":
        f"{PROJECT}.{DATASET}.fig5_final_boundary_24m",

    "distinctive_12m":
        f"{PROJECT}.{DATASET}.fig5_final_distinctive_12m",

    "distinctive_24m":
        f"{PROJECT}.{DATASET}.fig5_final_distinctive_24m",

}


# =============================================================================
# 2. LOADER
# =============================================================================

COMMON_COLS = [

    "focal_openalex_id",

    "focal_first_author_id",

    "ai_score",

    "accepted_month",

    "accepted_year",

    "focal_subfield_id",

    "focal_reference_count",

    "n_references",

    "external_citing_opportunities",

    "reuse_events",

    "reference_opportunities",

    "retention_rate",

    "loss_events",

]


BOUNDARY_COLS = (

    COMMON_COLS

    +

    [

        "boundary_group",

        "total_reference_opportunities_boundary",

        "total_loss_events_boundary",

    ]

)


DISTINCTIVE_COLS = (

    COMMON_COLS

    +

    [

        "topic_redundancy_group",

        "total_reference_opportunities_distinctive",

        "total_loss_events_distinctive",

    ]

)


def load_table(
    table_name,
    columns
):

    sql = f"""
    SELECT
        {", ".join(columns)}
    FROM `{table_name}`
    """


    print(
        "\nLOADING:"
    )

    print(
        table_name
    )


    d = client.query(
        sql
    ).to_dataframe()


    print(
        "Rows:",
        f"{len(d):,}"
    )


    print(
        "Focal papers:",
        f"{d['focal_openalex_id'].nunique():,}"
    )


    return d


# =============================================================================
# 3. PREPARE
# =============================================================================

def prepare(
    d
):

    x = d.copy()


    for col in [

        "ai_score",
        "focal_reference_count",
        "n_references",
        "external_citing_opportunities",
        "reuse_events",
        "reference_opportunities",
        "retention_rate",
        "loss_events",
        "total_reference_opportunities_boundary",
        "total_reference_opportunities_distinctive",
        "total_loss_events_boundary",
        "total_loss_events_distinctive",

    ]:

        if col in x.columns:

            x[col] = pd.to_numeric(

                x[col],

                errors="coerce"

            )


    x["accepted_month"] = pd.to_datetime(

        x["accepted_month"],

        errors="coerce"

    )


    # -------------------------------------------------------------------------
    # Verify the aggregate rate.
    # -------------------------------------------------------------------------

    reconstructed_rate = (

        x[
            "reuse_events"
        ]

        /

        x[
            "reference_opportunities"
        ]

    )


    if "retention_rate" in x.columns:

        stored_rate = x[
            "retention_rate"
        ]


        max_error = np.nanmax(

            np.abs(

                reconstructed_rate
                -
                stored_rate

            )

        )


        print(
            "Maximum retention-rate reconstruction error:",
            f"{max_error:.12e}"
        )


        if max_error > 1e-10:

            raise ValueError(
                "Retention-rate reconstruction failed."
            )


    x["retention_rate"] = reconstructed_rate


    # -------------------------------------------------------------------------
    # Author × subfield FE.
    # -------------------------------------------------------------------------

    x["author_subfield"] = (

        x[
            "focal_first_author_id"
        ].astype(str)

        +

        "::"

        +

        x[
            "focal_subfield_id"
        ].astype(str)

    )


    x["author_subfield_code"], _ = pd.factorize(

        x[
            "author_subfield"
        ],

        sort=False

    )


    x["month_code"], _ = pd.factorize(

        x[
            "accepted_month"
        ]
        .dt
        .to_period("M")
        .astype(str),

        sort=False

    )


    x["log_reference_count"] = np.log1p(

        x[
            "focal_reference_count"
        ].clip(
            lower=0
        )

    )


    return x


# =============================================================================
# 4. WEIGHTED GROUP DEMEAN
# =============================================================================

def weighted_group_demean(

    x,
    groups,
    weights

):

    x = np.asarray(
        x,
        dtype=np.float64
    )

    groups = np.asarray(
        groups,
        dtype=np.int64
    )

    weights = np.asarray(
        weights,
        dtype=np.float64
    )


    numerator = np.bincount(

        groups,

        weights=(
            weights * x
        )

    )


    denominator = np.bincount(

        groups,

        weights=weights

    )


    means = np.divide(

        numerator,

        denominator,

        out=np.zeros_like(
            numerator
        ),

        where=denominator > 0

    )


    return (
        x
        -
        means[groups]
    )


# =============================================================================
# 5. TWO-WAY FWL
# =============================================================================

def two_way_fwl(

    y,
    X,
    group_a,
    group_b,

    tolerance=1e-10,
    max_iterations=500

):

    y = np.asarray(
        y,
        dtype=np.float64
    )


    X = np.asarray(
        X,
        dtype=np.float64
    )


    if X.ndim == 1:

        X = X.reshape(
            -1,
            1
        )


    M = np.column_stack(

        [
            y,
            X,
        ]

    )


    for iteration in range(

        1,
        max_iterations + 1

    ):

        old = M.copy()


        for j in range(
            M.shape[1]
        ):

            M[:, j] = weighted_group_demean(

                M[:, j],

                group_a,

                np.ones(len(M))

            )


        for j in range(
            M.shape[1]
        ):

            M[:, j] = weighted_group_demean(

                M[:, j],

                group_b,

                np.ones(len(M))

            )


        change = float(

            np.max(

                np.abs(
                    M - old
                )

            )

        )


        if change < tolerance:

            break


    return (

        M[:, 0],

        M[:, 1:],

        iteration,

        change

    )


# =============================================================================
# 6. ABSOLUTE RETENTION MODEL
#
# IMPORTANT:
#
# For this validation we use one focal-paper × knowledge-category observation.
#
# The outcome is the absolute downstream retention rate.
#
# We use equal observation weight here because this is a PAPER-LEVEL
# validation of the substantive pattern.
#
# The existing opportunity counts are retained for descriptive support.
# =============================================================================

def retention_model(

    d,
    category_col,
    categories,
    label

):

    x = d.loc[

        d[
            category_col
        ].isin(categories)

    ].copy()


    # -------------------------------------------------------------------------
    # FE.
    # -------------------------------------------------------------------------

    author_fe = x[
        "author_subfield_code"
    ].to_numpy(
        dtype=np.int64
    )


    month_fe = x[
        "month_code"
    ].to_numpy(
        dtype=np.int64
    )


    # -------------------------------------------------------------------------
    # Outcome.
    # -------------------------------------------------------------------------

    y = x[
        "retention_rate"
    ].to_numpy(
        dtype=float
    )


    # -------------------------------------------------------------------------
    # Continuous AI.
    # -------------------------------------------------------------------------

    ai = x[
        "ai_score"
    ].to_numpy(
        dtype=float
    )


    # -------------------------------------------------------------------------
    # Reference-list size.
    # -------------------------------------------------------------------------

    log_refs = x[
        "log_reference_count"
    ].to_numpy(
        dtype=float
    )


    # -------------------------------------------------------------------------
    # Category indicators.
    # -------------------------------------------------------------------------

    category_values = x[
        category_col
    ].to_numpy(
        dtype=object
    )


    X_parts = [
        ai
    ]


    for category in categories[1:]:

        X_parts.append(

            (
                category_values == category
            ).astype(float)

        )


    for category in categories[1:]:

        X_parts.append(

            ai

            *

            (
                category_values == category
            ).astype(float)

        )


    X_parts.append(
        log_refs
    )


    X = np.column_stack(
        X_parts
    )


    # -------------------------------------------------------------------------
    # Residualize.
    # -------------------------------------------------------------------------

    y_r, X_r, iterations, final_change = two_way_fwl(

        y,

        X,

        author_fe,

        month_fe

    )


    # -------------------------------------------------------------------------
    # Standardize AI.
    # -------------------------------------------------------------------------

    ai_r = X_r[
        :,
        0
    ]


    ai_sd = np.std(
        ai_r
    )


    if (
        not np.isfinite(ai_sd)
        or
        ai_sd <= 0
    ):

        raise RuntimeError(

            f"{label}: invalid residualized AI SD."

        )


    X_r[
        :,
        0
    ] /= ai_sd


    n_interactions = (
        len(categories) - 1
    )


    interaction_start = (
        1 + n_interactions
    )


    interaction_end = (

        interaction_start
        +
        n_interactions

    )


    X_r[

        :,
        interaction_start:
        interaction_end

    ] /= ai_sd


    # -------------------------------------------------------------------------
    # Equal-weight WLS = OLS.
    # -------------------------------------------------------------------------

    weights = np.ones(
        len(x),
        dtype=float
    )


    model = sm.WLS(

        y_r,

        X_r,

        weights=weights

    ).fit(

        cov_type="cluster",

        cov_kwds={
            "groups": author_fe
        }

    )


    covariance = model.cov_params()


    results = []


    # -------------------------------------------------------------------------
    # Category-specific AI slopes.
    # -------------------------------------------------------------------------

    beta0 = float(
        model.params[0]
    )


    se0 = float(
        model.bse[0]
    )


    z0 = (

        beta0 / se0

        if se0 > 0

        else np.nan

    )


    results.append({

        "analysis":
            label,

        "category":
            categories[0],

        "effect":
            beta0,

        "se":
            se0,

        "p":
            (
                2 * norm.sf(
                    abs(z0)
                )
                if np.isfinite(z0)
                else np.nan
            ),

    })


    for j, category in enumerate(

        categories[1:]

    ):

        idx = (

            interaction_start
            +
            j

        )


        beta = (

            model.params[0]

            +

            model.params[idx]

        )


        variance = (

            covariance[0, 0]

            +

            covariance[idx, idx]

            +

            2
            *
            covariance[0, idx]

        )


        se = np.sqrt(

            max(
                variance,
                0
            )

        )


        z = (

            beta / se

            if se > 0

            else np.nan

        )


        results.append({

            "analysis":
                label,

            "category":
                category,

            "effect":
                beta,

            "se":
                se,

            "p":
                (
                    2 * norm.sf(
                        abs(z)
                    )
                    if np.isfinite(z)
                    else np.nan
                ),

        })


    results = pd.DataFrame(
        results
    )


    results["ci_low"] = (

        results["effect"]

        -

        1.96
        *
        results["se"]

    )


    results["ci_high"] = (

        results["effect"]

        +

        1.96
        *
        results["se"]

    )


    results["effect_pp"] = (

        results["effect"]
        *
        100.0

    )


    results["ci_low_pp"] = (

        results["ci_low"]
        *
        100.0

    )


    results["ci_high_pp"] = (

        results["ci_high"]
        *
        100.0

    )


    diagnostic = pd.DataFrame([{

        "analysis":
            label,

        "n_rows":
            len(x),

        "focal_papers":
            x[
                "focal_openalex_id"
            ].nunique(),

        "focal_authors":
            x[
                "focal_first_author_id"
            ].nunique(),

        "author_subfield_FE":
            x[
                "author_subfield"
            ].nunique(),

        "month_FE":
            x[
                "accepted_month"
            ].nunique(),

        "mean_retention":
            x[
                "retention_rate"
            ].mean(),

        "residualized_ai_sd":
            ai_sd,

        "fwl_iterations":
            iterations,

        "fwl_final_change":
            final_change,

    }])


    return (
        results,
        diagnostic
    )


# =============================================================================
# 7. BOUNDARY — 12m PRIMARY
# =============================================================================

boundary12 = load_table(

    TABLES["boundary_12m"],

    BOUNDARY_COLS

)


boundary12 = prepare(
    boundary12
)


print(
    "\nBOUNDARY 12m RAW RETENTION"
)

display(

    boundary12

    .groupby(
        "boundary_group",
        observed=True
    )

    .agg(

        focal_papers=(
            "focal_openalex_id",
            "nunique"
        ),

        references=(
            "n_references",
            "sum"
        ),

        opportunities=(
            "reference_opportunities",
            "sum"
        ),

        reused=(
            "reuse_events",
            "sum"
        ),

    )

    .assign(

        retention=lambda z:
            z["reused"]
            /
            z["opportunities"]

    )

)


boundary12_results, boundary12_diag = retention_model(

    boundary12,

    "boundary_group",

    [

        "Same subfield",
        "Cross-subfield; same field",
        "Cross-field",

    ],

    "Absolute retention — boundary — 12m"

)


print(
    "\nBOUNDARY 12m ABSOLUTE RETENTION"
)

display(
    boundary12_results
)

display(
    boundary12_diag
)


boundary12_results.to_csv(

    "fig5_absolute_retention_boundary_12m.csv",

    index=False

)


del boundary12

gc.collect()


# =============================================================================
# 8. BOUNDARY — 24m
# =============================================================================

boundary24 = load_table(

    TABLES["boundary_24m"],

    BOUNDARY_COLS

)


boundary24 = prepare(
    boundary24
)


boundary24_results, boundary24_diag = retention_model(

    boundary24,

    "boundary_group",

    [

        "Same subfield",
        "Cross-subfield; same field",
        "Cross-field",

    ],

    "Absolute retention — boundary — 24m"

)


print(
    "\nBOUNDARY 24m ABSOLUTE RETENTION"
)

display(
    boundary24_results
)

display(
    boundary24_diag
)


boundary24_results.to_csv(

    "fig5_absolute_retention_boundary_24m.csv",

    index=False

)


del boundary24

gc.collect()


# =============================================================================
# 9. DISTINCTIVE — 12m
# =============================================================================

distinctive12 = load_table(

    TABLES["distinctive_12m"],

    DISTINCTIVE_COLS

)


distinctive12 = prepare(
    distinctive12
)


print(
    "\nDISTINCTIVE 12m RAW RETENTION"
)

display(

    distinctive12

    .groupby(
        "topic_redundancy_group",
        observed=True
    )

    .agg(

        focal_papers=(
            "focal_openalex_id",
            "nunique"
        ),

        references=(
            "n_references",
            "sum"
        ),

        opportunities=(
            "reference_opportunities",
            "sum"
        ),

        reused=(
            "reuse_events",
            "sum"
        ),

    )

    .assign(

        retention=lambda z:
            z["reused"]
            /
            z["opportunities"]

    )

)


distinctive12_results, distinctive12_diag = retention_model(

    distinctive12,

    "topic_redundancy_group",

    [

        "High redundancy (4+)",
        "Low redundancy (2-3)",
        "Unique topic",

    ],

    "Absolute retention — distinctiveness — 12m"

)


print(
    "\nDISTINCTIVENESS 12m ABSOLUTE RETENTION"
)

display(
    distinctive12_results
)

display(
    distinctive12_diag
)


distinctive12_results.to_csv(

    "fig5_absolute_retention_distinctive_12m.csv",

    index=False

)


del distinctive12

gc.collect()


# =============================================================================
# 10. DISTINCTIVE — 24m
# =============================================================================

distinctive24 = load_table(

    TABLES["distinctive_24m"],

    DISTINCTIVE_COLS

)


distinctive24 = prepare(
    distinctive24
)


distinctive24_results, distinctive24_diag = retention_model(

    distinctive24,

    "topic_redundancy_group",

    [

        "High redundancy (4+)",
        "Low redundancy (2-3)",
        "Unique topic",

    ],

    "Absolute retention — distinctiveness — 24m"

)


print(
    "\nDISTINCTIVENESS 24m ABSOLUTE RETENTION"
)

display(
    distinctive24_results
)

display(
    distinctive24_diag
)


distinctive24_results.to_csv(

    "fig5_absolute_retention_distinctive_24m.csv",

    index=False

)


del distinctive24

gc.collect()


# =============================================================================
# 11. COMBINE ABSOLUTE RETENTION RESULTS
# =============================================================================

absolute_retention_results = pd.concat(

    [

        boundary12_results,
        boundary24_results,

        distinctive12_results,
        distinctive24_results,

    ],

    ignore_index=True

)


absolute_retention_diagnostics = pd.concat(

    [

        boundary12_diag,
        boundary24_diag,

        distinctive12_diag,
        distinctive24_diag,

    ],

    ignore_index=True

)


absolute_retention_results.to_csv(

    "fig5_absolute_retention_results.csv",

    index=False

)


absolute_retention_diagnostics.to_csv(

    "fig5_absolute_retention_diagnostics.csv",

    index=False

)


# =============================================================================
# 12. FINAL SUMMARY
# =============================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "FIGURE 5 — ABSOLUTE RETENTION VALIDATION"
)

print(
    "=" * 80
)


display(

    absolute_retention_results[

        [

            "analysis",
            "category",
            "effect_pp",
            "ci_low_pp",
            "ci_high_pp",
            "p",

        ]

    ]

)


print(
    "\nDiagnostics:"
)

display(
    absolute_retention_diagnostics
)


# =============================================================================
# 13. CLEANUP
# =============================================================================

del absolute_retention_results
del absolute_retention_diagnostics

gc.collect()


print(
    "\nFINAL ABSOLUTE-RETENTION VALIDATION COMPLETE."
)

### Historical atypicality and unseen pairs (main figure 4 panels d–e)

In [ ]:
# =============================================================================
# FIGURE 5 — FINAL D ANALYSIS
#
# RECOMBINATIONAL NOVELTY OF KNOWLEDGE TRANSMITTED THROUGH FOCAL PAPER F
#
# PRIMARY SPECIFICATION
# ---------------------
#
#   Y_F = beta * AI_F
#         + focal-subfield FE
#         + acceptance-month FE
#         + log(1 + focal references)
#         + log(1 + focal authors)
#         + error
#
# SE clustered by focal first author.
#
#
# UNIT OF ANALYSIS
# ----------------
#
# ONE ROW = ONE FOCAL PAPER F
#
#
# CORE PATHWAY
# ------------
#
# predecessor_i
#       \
#        -> F -> future citer C
#       /
# predecessor_j
#
# D asks:
#
#   Among predecessor pairs that are actually jointly transmitted through F,
#   are the transmitted combinations more historically atypical when F has
#   a higher AI-writing score?
#
#
# PRIMARY HISTORICAL WINDOW
# -------------------------
#
#   5 years before F
#
# ROBUSTNESS
# ----------
#
#   3-year atypicality
#   10-year atypicality
#   never-before-seen pair
#
#
# FOLLOW-UP
# ---------
#
# 12m and 24m are estimated separately.
#
#   12m -> eligible_12m == True
#   24m -> eligible_24m == True
#
#
# FOCAL UNIVERSE
# --------------
#
# accepted_date >= 2023-07-01
#
#
# AI SCALE
# --------
#
# Raw 0-1 score.
#
# Main coefficient = effect of a 0.10 increase in AI score.
#
#
# PRIMARY FE
# ----------
#
#   focal subfield
#   acceptance month
#
#
# STRINGENT ROBUSTNESS
# --------------------
#
#   first-author × focal-subfield FE
#   acceptance month FE
#
# This is intentionally NOT the primary specification because it discards
# most of the post-2023 focal-paper observations.
#
# =============================================================================


# =============================================================================
# 0. IMPORTS
# =============================================================================

import gc
import numpy as np
import pandas as pd
import statsmodels.api as sm

from google.cloud import bigquery


# =============================================================================
# 1. CLIENT
# =============================================================================

try:
    client

except NameError:

    client = bigquery.Client(
        project="aiwriting-488923"
    )


SUMMARY_TABLE = (
    "aiwriting-488923.redo."
    "fig5_recombination_focal_summary_final"
)


# =============================================================================
# 2. LOAD ONLY THE COLUMNS WE NEED
# =============================================================================

sql = f"""
SELECT

    focal_openalex_id,

    focal_first_author_id,

    ai_score,

    accepted_date,

    accepted_year,

    eligible_12m,

    eligible_24m,

    focal_reference_count,

    focal_authors_count,

    focal_subfield_id,

    mean_atypicality_3y_12m,

    mean_atypicality_5y_12m,

    mean_atypicality_10y_12m,

    share_never_seen_5y_12m,

    mean_atypicality_3y_24m,

    mean_atypicality_5y_24m,

    mean_atypicality_10y_24m,

    share_never_seen_5y_24m,

    n_transmitted_pairs_12m,

    n_transmitted_pairs_24m

FROM `{SUMMARY_TABLE}`
"""


print(
    "=" * 80
)

print(
    "LOADING FINAL D-ANALYSIS DATA"
)

print(
    "=" * 80
)


df = client.query(
    sql
).to_dataframe()


print(
    "Loaded rows:",
    f"{len(df):,}"
)


# =============================================================================
# 3. TYPE CLEANUP
# =============================================================================

df["accepted_date"] = pd.to_datetime(
    df["accepted_date"],
    errors="coerce"
)


numeric_cols = [

    "ai_score",

    "focal_reference_count",

    "focal_authors_count",

    "mean_atypicality_3y_12m",
    "mean_atypicality_5y_12m",
    "mean_atypicality_10y_12m",

    "share_never_seen_5y_12m",

    "mean_atypicality_3y_24m",
    "mean_atypicality_5y_24m",
    "mean_atypicality_10y_24m",

    "share_never_seen_5y_24m",

    "n_transmitted_pairs_12m",
    "n_transmitted_pairs_24m",

]


for col in numeric_cols:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )


for col in [
    "eligible_12m",
    "eligible_24m"
]:

    df[col] = (

        df[col]
        .fillna(False)
        .astype(bool)

    )


# =============================================================================
# 4. FINAL POST-DIFFUSION FOCAL UNIVERSE
#
# Remove the single missing-subfield observation as it cannot belong to the
# intended scientific-location FE.
# =============================================================================

before = len(df)


df = df.loc[

    df["accepted_date"].notna()

    &

    (
        df["accepted_date"]
        >=
        pd.Timestamp("2023-07-01")
    )

    &

    df["ai_score"].notna()

    &

    df["focal_first_author_id"].notna()

    &

    df["focal_subfield_id"].notna()

].copy()


print(
    "\nFINAL FOCAL UNIVERSE"
)

print(
    "Before filtering:",
    f"{before:,}"
)

print(
    "After filtering:",
    f"{len(df):,}"
)

print(
    "Removed:",
    f"{before - len(df):,}"
)


# =============================================================================
# 5. GLOBAL ASSERTIONS
# =============================================================================

assert not df[
    "focal_openalex_id"
].duplicated().any()


assert df[
    "ai_score"
].between(
    0,
    1
).all()


assert (
    df["accepted_date"].min()
    >=
    pd.Timestamp("2023-07-01")
)


# =============================================================================
# 6. CONTROL VARIABLES
#
# These are fixed characteristics of F and are included because the
# recombination opportunity structure depends strongly on paper size.
# =============================================================================

df["log_reference_count"] = np.log1p(

    df[
        "focal_reference_count"
    ].clip(
        lower=0
    )

)


df["log_author_count"] = np.log1p(

    df[
        "focal_authors_count"
    ].clip(
        lower=0
    )

)


# Main treatment scaling.
#
# AI score is 0-1.
#
# Divide by 0.10 so the coefficient is:
#
#   change in D associated with a 0.10 increase in AI score.
#
df["ai_0p10"] = (

    df[
        "ai_score"
    ]
    /
    0.10

)


# =============================================================================
# 7. FIXED EFFECT CODES
# =============================================================================

df["subfield_code"], _ = pd.factorize(

    df[
        "focal_subfield_id"
    ],

    sort=False

)


df["accepted_month"] = (

    df[
        "accepted_date"
    ]
    .dt
    .to_period("M")
    .astype(str)

)


df["month_code"], _ = pd.factorize(

    df[
        "accepted_month"
    ],

    sort=False

)


# =============================================================================
# 8. SUPPORT CHECK
# =============================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "D SUPPORT"
)

print(
    "=" * 80
)


for window in [12, 24]:

    eligible = (

        df[
            f"eligible_{window}m"
        ]

    )

    has_pairs = (

        df[
            f"n_transmitted_pairs_{window}m"
        ]
        >
        0

    )


    print(
        f"\n{window}m"
    )

    print(
        "  complete follow-up:",
        f"{eligible.sum():,}"
    )

    print(
        "  complete + transmitted pairs:",
        f"{(eligible & has_pairs).sum():,}"
    )


# =============================================================================
# 9. FOCAL-PAPER SAMPLE BUILDER
# =============================================================================

def build_sample(

    data,

    outcome,

    window,

):

    eligible_col = (
        f"eligible_{window}m"
    )

    pair_col = (
        f"n_transmitted_pairs_{window}m"
    )


    columns = [

        "focal_openalex_id",

        "focal_first_author_id",

        "ai_0p10",

        "ai_score",

        "accepted_date",

        "accepted_year",

        "subfield_code",

        "month_code",

        "focal_reference_count",

        "focal_authors_count",

        "log_reference_count",

        "log_author_count",

        eligible_col,

        pair_col,

        outcome,

    ]


    missing = [

        c
        for c in columns
        if c not in data.columns

    ]


    if missing:

        raise KeyError(

            "Missing columns: "
            +
            ", ".join(missing)

        )


    x = data[
        columns
    ].copy()


    # Complete follow-up.
    x = x.loc[
        x[
            eligible_col
        ]
    ].copy()


    # At least one transmitted predecessor pair.
    x = x.loc[

        x[
            pair_col
        ]
        >
        0

    ].copy()


    # Finite model variables.
    needed = [

        "ai_0p10",

        "log_reference_count",

        "log_author_count",

        outcome,

    ]


    finite = np.ones(
        len(x),
        dtype=bool
    )


    for col in needed:

        finite &= np.isfinite(

            x[
                col
            ]
            .to_numpy(
                dtype=float
            )

        )


    x = x.loc[
        finite
    ].copy()


    if len(x) == 0:

        raise RuntimeError(

            f"No observations remain for {outcome}."

        )


    # Never-seen shares must be in [0,1].
    if outcome.startswith(
        "share_"
    ):

        valid = x[
            outcome
        ].between(
            0,
            1
        )


        if not valid.all():

            raise ValueError(

                f"{outcome} has values outside [0,1]."

            )


    return x


# =============================================================================
# 10. GROUP DEMEAN
# =============================================================================

def group_demean(

    x,

    groups

):

    x = np.asarray(
        x,
        dtype=np.float64
    )

    groups = np.asarray(
        groups,
        dtype=np.int64
    )


    sums = np.bincount(

        groups,

        weights=x

    )


    counts = np.bincount(
        groups
    )


    means = np.divide(

        sums,

        counts,

        out=np.zeros_like(
            sums,
            dtype=float
        ),

        where=counts > 0

    )


    return (

        x
        -
        means[
            groups
        ]

    )


# =============================================================================
# 11. FWL WITH TWO FIXED EFFECTS
#
# FE:
#
#     subfield
#     accepted month
#
# X:
#
#     AI
#     log reference count
#     log author count
#
# =============================================================================

def fwl_two_fe(

    y,

    X,

    fe1,

    fe2,

    tolerance=1e-10,

    max_iterations=1000

):

    y = np.asarray(
        y,
        dtype=np.float64
    )


    X = np.asarray(
        X,
        dtype=np.float64
    )


    if X.ndim == 1:

        X = X.reshape(
            -1,
            1
        )


    M = np.column_stack(

        [
            y,
            X
        ]

    )


    for iteration in range(

        1,
        max_iterations + 1

    ):

        old = M.copy()


        # -------------------------------------------------------------
        # Remove FE 1
        # -------------------------------------------------------------

        for j in range(
            M.shape[1]
        ):

            M[:, j] = group_demean(

                M[:, j],

                fe1

            )


        # -------------------------------------------------------------
        # Remove FE 2
        # -------------------------------------------------------------

        for j in range(
            M.shape[1]
        ):

            M[:, j] = group_demean(

                M[:, j],

                fe2

            )


        change = float(

            np.max(

                np.abs(
                    M - old
                )

            )

        )


        if change < tolerance:

            return (

                M[:, 0],

                M[:, 1:],

                iteration,

                change

            )


    raise RuntimeError(

        "FWL failed to converge after "
        f"{max_iterations} iterations."

    )


# =============================================================================
# 12. CLUSTER-ROBUST REGRESSION
#
# IMPORTANT:
#
# We residualize both outcome and ALL regressors against FE.
#
# The regression is then:
#
#     residualized outcome
#        ~
#     residualized AI
#     residualized log references
#     residualized log authors
#
# SE clustered by first author.
# =============================================================================

def estimate_primary(

    data,

    outcome,

    window,

    label

):

    x = build_sample(

        data,

        outcome,

        window

    )


    # -------------------------------------------------------------------------
    # Arrays
    # -------------------------------------------------------------------------

    y = x[
        outcome
    ].to_numpy(
        dtype=float
    )


    X = x[

        [

            "ai_0p10",

            "log_reference_count",

            "log_author_count"

        ]

    ].to_numpy(
        dtype=float
    )


    fe_subfield = x[
        "subfield_code"
    ].to_numpy(
        dtype=np.int64
    )


    fe_month = x[
        "month_code"
    ].to_numpy(
        dtype=np.int64
    )


    cluster = x[
        "focal_first_author_id"
    ].astype(str).to_numpy()


    # -------------------------------------------------------------------------
    # FWL
    # -------------------------------------------------------------------------

    y_r, X_r, iterations, change = fwl_two_fe(

        y,

        X,

        fe_subfield,

        fe_month

    )


    # -------------------------------------------------------------------------
    # Identification diagnostic
    # -------------------------------------------------------------------------

    ai_r = X_r[
        :,
        0
    ]


    ai_resid_sd = float(
        np.std(
            ai_r,
            ddof=1
        )
    )


    if (

        not np.isfinite(
            ai_resid_sd
        )

        or

        ai_resid_sd <= 0

    ):

        raise RuntimeError(

            f"{label}: no residualized AI variation."

        )


    # -------------------------------------------------------------------------
    # Regression
    # -------------------------------------------------------------------------

    model = sm.OLS(

        y_r,

        X_r

    ).fit(

        cov_type="cluster",

        cov_kwds={

            "groups":
                cluster,

            "use_correction":
                True

        }

    )


    # -------------------------------------------------------------------------
    # AI coefficient
    # -------------------------------------------------------------------------

    beta = float(
        model.params[0]
    )


    se = float(
        model.bse[0]
    )


    p = float(
        model.pvalues[0]
    )


    ci_low = (

        beta
        -
        1.96
        *
        se

    )


    ci_high = (

        beta
        +
        1.96
        *
        se

    )


    # -------------------------------------------------------------------------
    # Diagnostic output
    # -------------------------------------------------------------------------

    result = {

        "analysis":
            label,

        "outcome":
            outcome,

        "window":
            window,

        "n_obs":
            len(x),

        "n_focal_papers":
            x[
                "focal_openalex_id"
            ].nunique(),

        "n_focal_authors":
            x[
                "focal_first_author_id"
            ].nunique(),

        "n_subfield_FE":
            x[
                "subfield_code"
            ].nunique(),

        "n_month_FE":
            x[
                "month_code"
            ].nunique(),

        "beta_per_0p10_ai":
            beta,

        "se":
            se,

        "p":
            p,

        "ci_low":
            ci_low,

        "ci_high":
            ci_high,

        "ai_residualized_sd":
            ai_resid_sd,

        "fwl_iterations":
            iterations,

        "fwl_final_change":
            change,

        "r_squared_within":
            model.rsquared,

    }


    return (

        pd.DataFrame([
            result
        ]),

        x

    )


# =============================================================================
# 13. RUN PRIMARY D MODELS
# =============================================================================
#
# Primary:
#     5-year historical atypicality
#
# Estimated separately for:
#     12m
#     24m
# =============================================================================

primary_results = []
primary_samples = {}


for window in [12, 24]:

    outcome = (
        f"mean_atypicality_5y_{window}m"
    )


    label = (
        f"D PRIMARY — 5y atypicality — {window}m"
    )


    r, s = estimate_primary(

        df,

        outcome,

        window,

        label

    )


    primary_results.append(
        r
    )


    primary_samples[
        label
    ] = s


# =============================================================================
# 14. RUN HISTORICAL-WINDOW ROBUSTNESS
# =============================================================================

robustness_results = []


for window in [12, 24]:

    for history in [3, 10]:

        outcome = (
            f"mean_atypicality_{history}y_{window}m"
        )


        label = (
            f"D ROBUSTNESS — {history}y atypicality — {window}m"
        )


        r, _ = estimate_primary(

            df,

            outcome,

            window,

            label

        )


        robustness_results.append(
            r
        )


# =============================================================================
# 15. NEVER-SEEN ROBUSTNESS
# =============================================================================

for window in [12, 24]:

    outcome = (
        f"share_never_seen_5y_{window}m"
    )


    label = (
        f"D ROBUSTNESS — never-seen 5y — {window}m"
    )


    r, _ = estimate_primary(

        df,

        outcome,

        window,

        label

    )


    robustness_results.append(
        r
    )


# =============================================================================
# 16. PRIMARY RESULTS
# =============================================================================

primary_results = pd.concat(

    primary_results,

    ignore_index=True

)


print(
    "\n"
    + "=" * 80
)

print(
    "D — PRIMARY RESULTS"
)

print(
    "=" * 80
)


display(

    primary_results[

        [

            "analysis",

            "n_obs",

            "n_focal_papers",

            "n_focal_authors",

            "beta_per_0p10_ai",

            "se",

            "p",

            "ci_low",

            "ci_high",

            "n_subfield_FE",

            "n_month_FE",

            "ai_residualized_sd",

        ]

    ]

)


# =============================================================================
# 17. ROBUSTNESS RESULTS
# =============================================================================

robustness_results = pd.concat(

    robustness_results,

    ignore_index=True

)


print(
    "\n"
    + "=" * 80
)

print(
    "D — ROBUSTNESS RESULTS"
)

print(
    "=" * 80
)


display(

    robustness_results[

        [

            "analysis",

            "n_obs",

            "beta_per_0p10_ai",

            "se",

            "p",

            "ci_low",

            "ci_high",

        ]

    ]

)


# =============================================================================
# 18. STRINGENT FIRST-AUTHOR × SUBFIELD FE ROBUSTNESS
#
# This deliberately reproduces the more restrictive logic that generated the
# earlier 33k / 18k samples.
#
# It is NOT the primary model.
# =============================================================================

df["author_subfield"] = (

    df[
        "focal_first_author_id"
    ].astype(str)

    +

    "::"

    +

    df[
        "focal_subfield_id"
    ].astype(str)

)


df["author_subfield_code"], _ = pd.factorize(

    df[
        "author_subfield"
    ],

    sort=False

)


def estimate_strict(

    data,

    outcome,

    window,

    label

):

    x = build_sample(

        data,

        outcome,

        window

    )


    # -------------------------------------------------------------------------
    # Replace ordinary subfield FE with author × subfield FE.
    # -------------------------------------------------------------------------

    x = x.copy()


    x["strict_fe"], _ = pd.factorize(

        x[
            "focal_first_author_id"
        ].astype(str)

        +

        "::"

        +

        x[
            "subfield_code"
        ].astype(str),

        sort=False

    )


    group_sizes = (

        x
        .groupby(
            "strict_fe",
            observed=True
        )
        .size()

    )


    supported = group_sizes[
        group_sizes >= 2
    ].index


    x = x.loc[

        x[
            "strict_fe"
        ]
        .isin(
            supported
        )

    ].copy()


    if len(x) == 0:

        raise RuntimeError(
            f"{label}: no strict-FE observations."
        )


    y = x[
        outcome
    ].to_numpy(
        dtype=float
    )


    X = x[

        [

            "ai_0p10",

            "log_reference_count",

            "log_author_count"

        ]

    ].to_numpy(
        dtype=float
    )


    fe_author_subfield = x[
        "strict_fe"
    ].to_numpy(
        dtype=np.int64
    )


    fe_month = x[
        "month_code"
    ].to_numpy(
        dtype=np.int64
    )


    cluster = x[
        "focal_first_author_id"
    ].astype(str).to_numpy()


    y_r, X_r, iterations, change = fwl_two_fe(

        y,

        X,

        fe_author_subfield,

        fe_month

    )


    model = sm.OLS(

        y_r,

        X_r

    ).fit(

        cov_type="cluster",

        cov_kwds={

            "groups":
                cluster,

            "use_correction":
                True

        }

    )


    beta = float(
        model.params[0]
    )


    se = float(
        model.bse[0]
    )


    return pd.DataFrame([{

        "analysis":
            label,

        "n_obs":
            len(x),

        "n_focal_authors":
            x[
                "focal_first_author_id"
            ].nunique(),

        "n_author_subfield_FE":
            x[
                "strict_fe"
            ].nunique(),

        "beta_per_0p10_ai":
            beta,

        "se":
            se,

        "p":
            float(
                model.pvalues[0]
            ),

        "ci_low":
            beta - 1.96 * se,

        "ci_high":
            beta + 1.96 * se,

        "fwl_iterations":
            iterations,

        "fwl_final_change":
            change,

    }])


strict_results = []


for window in [12, 24]:

    outcome = (
        f"mean_atypicality_5y_{window}m"
    )


    label = (
        f"D STRINGENT — author×subfield FE — {window}m"
    )


    strict_results.append(

        estimate_strict(

            df,

            outcome,

            window,

            label

        )

    )


strict_results = pd.concat(

    strict_results,

    ignore_index=True

)


print(
    "\n"
    + "=" * 80
)

print(
    "D — STRINGENT AUTHOR×SUBFIELD ROBUSTNESS"
)

print(
    "=" * 80
)


display(
    strict_results
)


# =============================================================================
# 21. FINAL COMBINED RESULT TABLE
# =============================================================================

all_results = pd.concat(

    [

        primary_results,

        robustness_results,

        strict_results,



    ],

    ignore_index=True

)


print(
    "\n"
    + "=" * 80
)

print(
    "FINAL D RESULTS — ALL SPECIFICATIONS"
)

print(
    "=" * 80
)


display(

    all_results[

        [

            "analysis",

            "n_obs",

            "beta_per_0p10_ai",

            "se",

            "p",

            "ci_low",

            "ci_high",

        ]

    ]

)


# =============================================================================
# 22. SAVE
# =============================================================================

primary_results.to_csv(

    "fig5_D_primary_results.csv",

    index=False

)


robustness_results.to_csv(

    "fig5_D_historical_robustness.csv",

    index=False

)


strict_results.to_csv(

    "fig5_D_author_subfield_robustness.csv",

    index=False

)






all_results.to_csv(

    "fig5_D_all_results.csv",

    index=False

)


# =============================================================================
# 23. FINAL DIAGNOSTIC SUMMARY
# =============================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "FINAL D ANALYSIS COMPLETE"
)

print(
    "=" * 80
)

print(
    "\nPrimary interpretation:"
)

print(
    "beta = change in transmitted-pair historical atypicality "
    "associated with a 0.10 increase in focal-paper AI score."
)

print(
    "\nPrimary FE:"
)

print(
    "focal subfield + acceptance month"
)

print(
    "\nControls:"
)

print(
    "log(1 + focal reference count) + "
    "log(1 + focal author count)"
)

print(
    "\nSE:"
)

print(
    "clustered by focal first author"
)

print(
    "\nStringent robustness:"
)

print(
    "first-author x subfield FE + acceptance month FE"
)


# =============================================================================
# CLEANUP
# =============================================================================

del primary_samples
del df

gc.collect()

### No-controls specification (main figure 4 panel f)

In [ ]:
# =============================================================================
# FIGURE 5 — FINAL D ROBUSTNESS
#
# PURPOSE
# -------
# Test whether the main D result survives WITHOUT:
#
#     log(1 + focal reference count)
#     log(1 + focal author count)
#
#
# PRIMARY MODEL:
#
#   Y_F =
#       beta * AI_F
#       + focal-subfield FE
#       + acceptance-month FE
#       + error_F
#
#
# SAME AS PRIMARY MODEL IN EVERY OTHER RESPECT:
#
#   * post-diffusion focal universe
#   * 2023-07-01 cutoff
#   * complete 12m / 24m follow-up
#   * >=1 transmitted predecessor pair
#   * focal-paper unit
#   * raw AI score
#   * coefficient per 0.10 AI increase
#   * SE clustered by focal first author
#
#
# This is intentionally a SINGLE specification test.
# =============================================================================


import gc

import numpy as np
import pandas as pd
import statsmodels.api as sm


# =============================================================================
# 1. LOAD DATA
# =============================================================================

SUMMARY_TABLE = (
    "aiwriting-488923.redo."
    "fig5_recombination_focal_summary_final"
)


sql = f"""
SELECT

    focal_openalex_id,

    focal_first_author_id,

    ai_score,

    accepted_date,

    accepted_year,

    eligible_12m,

    eligible_24m,

    focal_subfield_id,

    mean_atypicality_5y_12m,

    mean_atypicality_5y_24m,

    n_transmitted_pairs_12m,

    n_transmitted_pairs_24m

FROM `{SUMMARY_TABLE}`
"""


print(
    "=" * 80
)

print(
    "D — NO-CONTROL ROBUSTNESS"
)

print(
    "=" * 80
)


try:
    client

except NameError:

    from google.cloud import bigquery

    client = bigquery.Client(
        project="aiwriting-488923"
    )


d = client.query(
    sql
).to_dataframe()


print(
    "Loaded rows:",
    f"{len(d):,}"
)


# =============================================================================
# 2. TYPE CLEANUP
# =============================================================================

d["accepted_date"] = pd.to_datetime(

    d["accepted_date"],

    errors="coerce"

)


numeric_cols = [

    "ai_score",

    "mean_atypicality_5y_12m",

    "mean_atypicality_5y_24m",

    "n_transmitted_pairs_12m",

    "n_transmitted_pairs_24m",

]


for col in numeric_cols:

    d[col] = pd.to_numeric(

        d[col],

        errors="coerce"

    )


for col in [

    "eligible_12m",

    "eligible_24m"

]:

    d[col] = (

        d[col]
        .fillna(False)
        .astype(bool)

    )


# =============================================================================
# 3. FINAL FOCAL UNIVERSE
# =============================================================================

d = d.loc[

    d["accepted_date"].notna()

    &

    (
        d["accepted_date"]
        >=
        pd.Timestamp("2023-07-01")
    )

    &

    d["ai_score"].notna()

    &

    d["focal_first_author_id"].notna()

    &

    d["focal_subfield_id"].notna()

].copy()


assert not d[
    "focal_openalex_id"
].duplicated().any()


assert d[
    "ai_score"
].between(
    0,
    1
).all()


print(
    "\nFinal focal papers:",
    f"{len(d):,}"
)


# =============================================================================
# 4. FIXED-EFFECT IDS
# =============================================================================

d["subfield_code"], _ = pd.factorize(

    d[
        "focal_subfield_id"
    ],

    sort=False

)


d["accepted_month"] = (

    d[
        "accepted_date"
    ]
    .dt
    .to_period("M")
    .astype(str)

)


d["month_code"], _ = pd.factorize(

    d[
        "accepted_month"
    ],

    sort=False

)


# =============================================================================
# 5. AI SCALE
#
# beta = change in outcome per 0.10 AI-score increase.
# =============================================================================

d["ai_0p10"] = (

    d[
        "ai_score"
    ]
    /
    0.10

)


# =============================================================================
# 6. SIMPLE TWO-WAY FWL
# =============================================================================

def group_demean(

    x,

    groups

):

    x = np.asarray(
        x,
        dtype=float
    )

    groups = np.asarray(
        groups,
        dtype=np.int64
    )


    sums = np.bincount(

        groups,

        weights=x

    )


    counts = np.bincount(
        groups
    )


    means = np.divide(

        sums,

        counts,

        out=np.zeros_like(
            sums,
            dtype=float
        ),

        where=counts > 0

    )


    return (

        x
        -
        means[
            groups
        ]

    )


def fwl_two_fe(

    y,

    X,

    fe1,

    fe2,

    tolerance=1e-10,

    max_iterations=1000

):

    y = np.asarray(
        y,
        dtype=float
    )


    X = np.asarray(
        X,
        dtype=float
    )


    if X.ndim == 1:

        X = X.reshape(
            -1,
            1
        )


    M = np.column_stack(

        [
            y,
            X
        ]

    )


    for iteration in range(

        1,
        max_iterations + 1

    ):

        old = M.copy()


        for j in range(
            M.shape[1]
        ):

            M[:, j] = group_demean(

                M[:, j],

                fe1

            )


        for j in range(
            M.shape[1]
        ):

            M[:, j] = group_demean(

                M[:, j],

                fe2

            )


        change = float(

            np.max(

                np.abs(
                    M - old
                )

            )

        )


        if change < tolerance:

            return (

                M[:, 0],

                M[:, 1:],

                iteration,

                change

            )


    raise RuntimeError(

        "FWL failed to converge."

    )


# =============================================================================
# 7. NO-CONTROL ESTIMATOR
# =============================================================================

def estimate_no_controls(

    data,

    outcome,

    window,

    label

):

    eligible_col = (
        f"eligible_{window}m"
    )

    pair_col = (
        f"n_transmitted_pairs_{window}m"
    )


    # -------------------------------------------------------------------------
    # Model sample
    # -------------------------------------------------------------------------

    x = data.loc[

        data[
            eligible_col
        ]

        &

        (
            data[
                pair_col
            ]
            >
            0
        )

        &

        np.isfinite(
            data[
                outcome
            ]
            .to_numpy(
                dtype=float
            )
        )

    ].copy()


    if len(x) == 0:

        raise RuntimeError(
            f"{label}: no observations."
        )


    # -------------------------------------------------------------------------
    # Arrays
    # -------------------------------------------------------------------------

    y = x[
        outcome
    ].to_numpy(
        dtype=float
    )


    X = x[
        "ai_0p10"
    ].to_numpy(
        dtype=float
    ).reshape(
        -1,
        1
    )


    fe_subfield = x[
        "subfield_code"
    ].to_numpy(
        dtype=np.int64
    )


    fe_month = x[
        "month_code"
    ].to_numpy(
        dtype=np.int64
    )


    cluster = x[
        "focal_first_author_id"
    ].astype(str).to_numpy()


    # -------------------------------------------------------------------------
    # FWL
    # -------------------------------------------------------------------------

    y_r, X_r, iterations, change = fwl_two_fe(

        y,

        X,

        fe_subfield,

        fe_month

    )


    # -------------------------------------------------------------------------
    # Identification
    # -------------------------------------------------------------------------

    ai_resid_sd = float(

        np.std(
            X_r[:, 0],
            ddof=1
        )

    )


    if (

        not np.isfinite(
            ai_resid_sd
        )

        or

        ai_resid_sd <= 0

    ):

        raise RuntimeError(

            f"{label}: no residualized AI variation."

        )


    # -------------------------------------------------------------------------
    # Regression
    # -------------------------------------------------------------------------

    model = sm.OLS(

        y_r,

        X_r

    ).fit(

        cov_type="cluster",

        cov_kwds={

            "groups":
                cluster,

            "use_correction":
                True

        }

    )


    beta = float(
        model.params[0]
    )


    se = float(
        model.bse[0]
    )


    p = float(
        model.pvalues[0]
    )


    ci_low = (
        beta
        -
        1.96
        *
        se
    )


    ci_high = (
        beta
        +
        1.96
        *
        se
    )


    return pd.DataFrame([{

        "analysis":
            label,

        "window":
            window,

        "outcome":
            outcome,

        "n_obs":
            len(x),

        "n_focal_papers":
            x[
                "focal_openalex_id"
            ].nunique(),

        "n_focal_authors":
            x[
                "focal_first_author_id"
            ].nunique(),

        "n_subfield_FE":
            x[
                "subfield_code"
            ].nunique(),

        "n_month_FE":
            x[
                "month_code"
            ].nunique(),

        "beta_per_0p10_ai":
            beta,

        "se":
            se,

        "p":
            p,

        "ci_low":
            ci_low,

        "ci_high":
            ci_high,

        "ai_residualized_sd":
            ai_resid_sd,

        "fwl_iterations":
            iterations,

        "fwl_final_change":
            change,

        "controls":
            "none",

    }])


# =============================================================================
# 8. RUN 12m / 24m
# =============================================================================

no_control_results = []


for window in [12, 24]:

    outcome = (

        f"mean_atypicality_5y_{window}m"

    )


    label = (

        f"D NO-CONTROL — 5y atypicality — {window}m"

    )


    no_control_results.append(

        estimate_no_controls(

            d,

            outcome,

            window,

            label

        )

    )


no_control_results = pd.concat(

    no_control_results,

    ignore_index=True

)


# =============================================================================
# 9. DISPLAY
# =============================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "NO-CONTROL RESULTS"
)

print(
    "=" * 80
)


display(

    no_control_results[

        [

            "analysis",

            "n_obs",

            "n_focal_papers",

            "n_focal_authors",

            "beta_per_0p10_ai",

            "se",

            "p",

            "ci_low",

            "ci_high",

            "n_subfield_FE",

            "n_month_FE",

            "ai_residualized_sd",

        ]

    ]

)


# =============================================================================
# 10. COMPARE WITH THE PRIMARY SPECIFICATION
#
# These are the previously estimated primary coefficients:
#
#     12m = 0.045232
#     24m = 0.040402
#
# We do NOT hard-code their SEs into the analysis; this table is only a
# descriptive comparison.
# =============================================================================

previous_primary = pd.DataFrame({

    "window": [
        12,
        24
    ],

    "primary_beta_with_controls": [
        0.045232,
        0.040402
    ]

})


comparison = (

    no_control_results[

        [

            "window",

            "beta_per_0p10_ai"

        ]

    ]

    .merge(

        previous_primary,

        on="window",

        how="left"

    )

)


comparison["difference"] = (

    comparison[
        "beta_per_0p10_ai"
    ]

    -

    comparison[
        "primary_beta_with_controls"
    ]

)


comparison["percent_difference"] = (

    100
    *

    comparison[
        "difference"
    ]

    /

    comparison[
        "primary_beta_with_controls"
    ]

)


print(
    "\n"
    + "=" * 80
)

print(
    "PRIMARY VS NO-CONTROL COMPARISON"
)

print(
    "=" * 80
)


display(
    comparison
)


# =============================================================================
# 11. SAVE
# =============================================================================

no_control_results.to_csv(

    "fig5_D_no_control_robustness.csv",

    index=False

)


comparison.to_csv(

    "fig5_D_primary_vs_no_control.csv",

    index=False

)


# =============================================================================
# 12. DONE
# =============================================================================

del d

gc.collect()


print(
    "\n"
    + "=" * 80
)

print(
    "D NO-CONTROL ROBUSTNESS COMPLETE"
)

print(
    "=" * 80
)

### Main figure 4 visualization

In [ ]:
# =============================================================================
# FIGURE 5 — FINAL DATA-RICH RENDERING
#
# FINAL RESULTS ONLY
#
# A  Boundary-spanning loss composition
# B  Distinctive-loss composition
# C  Absolute retention across boundary + distinctiveness
# D  Recombinational novelty across 3y / 5y / 10y historical baselines
# E  Previously unseen predecessor combinations
# F  Specification robustness of recombinational novelty
#
# VISUAL GRAMMAR
# --------------
#
# 12m = BLUE filled circle
# 24m = ORANGE open circle
#
# Within each category:
#     12m and 24m are vertically offset
#
# Categories are compactly grouped.
#
# =============================================================================


# =============================================================================
# 0. IMPORTS
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D


# =============================================================================
# 1. FILES
# =============================================================================

LOSS_FILE = "fig5_loss_composition_results.csv"

ABS_FILE = "fig5_absolute_retention_results.csv"

D_PRIMARY_FILE = "fig5_D_primary_results.csv"

D_HIST_FILE = "fig5_D_historical_robustness.csv"

D_NOCONTROL_FILE = "fig5_D_no_control_robustness.csv"

D_STRICT_FILE = "fig5_D_author_subfield_robustness.csv"


required_files = [

    LOSS_FILE,
    ABS_FILE,
    D_PRIMARY_FILE,
    D_HIST_FILE,
    D_NOCONTROL_FILE,
    D_STRICT_FILE,

]


missing = [

    f
    for f in required_files
    if not os.path.exists(f)

]


if missing:

    raise FileNotFoundError(

        "Missing required final Figure 5 files:\n"
        +
        "\n".join(missing)

    )


# =============================================================================
# 2. LOAD FINAL RESULTS
# =============================================================================

loss = pd.read_csv(
    LOSS_FILE
)

absolute = pd.read_csv(
    ABS_FILE
)

d_primary = pd.read_csv(
    D_PRIMARY_FILE
)

d_hist = pd.read_csv(
    D_HIST_FILE
)

d_nocontrol = pd.read_csv(
    D_NOCONTROL_FILE
)

d_strict = pd.read_csv(
    D_STRICT_FILE
)


# =============================================================================
# 3. SCHEMA CHECK
# =============================================================================

def require_columns(

    df,
    cols,
    name

):

    missing_cols = [

        c
        for c in cols
        if c not in df.columns

    ]


    if missing_cols:

        raise KeyError(

            f"{name} missing columns: "
            +
            ", ".join(missing_cols)

        )


require_columns(

    loss,

    [

        "analysis",
        "effect_pp",
        "ci_low_pp",
        "ci_high_pp",

    ],

    "Loss composition"

)


require_columns(

    absolute,

    [

        "analysis",
        "category",
        "effect_pp",
        "ci_low_pp",
        "ci_high_pp",

    ],

    "Absolute retention"

)


for x, name in [

    (
        d_primary,
        "D primary"
    ),

    (
        d_hist,
        "D historical robustness"
    ),

    (
        d_nocontrol,
        "D no-control"
    ),

    (
        d_strict,
        "D author-subfield robustness"
    ),

]:

    require_columns(

        x,

        [

            "analysis",
            "beta_per_0p10_ai",
            "ci_low",
            "ci_high",

        ],

        name

    )


# =============================================================================
# 4. NUMERIC CLEANUP
# =============================================================================

for col in [

    "effect_pp",
    "ci_low_pp",
    "ci_high_pp",

]:

    loss[col] = pd.to_numeric(
        loss[col],
        errors="coerce"
    )

    absolute[col] = pd.to_numeric(
        absolute[col],
        errors="coerce"
    )


for x in [

    d_primary,
    d_hist,
    d_nocontrol,
    d_strict,

]:

    for col in [

        "beta_per_0p10_ai",
        "ci_low",
        "ci_high",

    ]:

        x[col] = pd.to_numeric(
            x[col],
            errors="coerce"
        )


# =============================================================================
# 5. PARSE LOSS CATEGORIES
# =============================================================================

def parse_loss_category(
    s
):

    s = str(
        s
    ).lower()


    # Check cross-subfield first because it contains "cross-field".
    if "cross-subfield" in s:

        return (
            "Cross-subfield,\nsame field"
        )


    if "cross-field" in s:

        return "Cross-field"


    if "unique topic" in s:

        return "Unique topic"


    if "low redundancy" in s:

        return "Low redundancy"


    raise ValueError(

        f"Unrecognized loss category: {s}"

    )


loss["plot_category"] = (

    loss["analysis"]
    .apply(
        parse_loss_category
    )

)


# =============================================================================
# 6. COLORS
# =============================================================================
#
# Deliberately distinct colors for the observation windows.
#
# These are used consistently in ALL six panels.
# =============================================================================

COLOR_12M = "#4E79A7"     # blue

COLOR_24M = "#F28E2B"     # orange

TEXT = "#222222"

GRAY = "#777777"

GRID = "#D8D8D8"


# =============================================================================
# 7. GLOBAL STYLE
# =============================================================================

plt.rcParams.update({

    "font.family":
        "DejaVu Sans",

    "font.size":
        9,

    "axes.titlesize":
        10.8,

    "axes.labelsize":
        8.8,

    "xtick.labelsize":
        7.8,

    "ytick.labelsize":
        7.8,

    "legend.fontsize":
        8.0,

    "axes.spines.top":
        False,

    "axes.spines.right":
        False,

    "axes.linewidth":
        0.75,

})


# =============================================================================
# 8. HELPERS
# =============================================================================

def get_one(

    df,
    mask,
    label

):

    x = df.loc[
        mask
    ].copy()


    if len(x) != 1:

        raise ValueError(

            f"{label}: expected exactly one row; "
            f"found {len(x)}."

        )


    return x.iloc[0]


def setup_axis(

    ax,
    grid_axis="x"

):

    ax.tick_params(

        axis="both",

        length=2.8,

        width=0.6,

        color=TEXT,

        labelcolor=TEXT

    )


    ax.spines[
        "left"
    ].set_color(
        TEXT
    )

    ax.spines[
        "bottom"
    ].set_color(
        TEXT
    )


    ax.grid(

        axis=grid_axis,

        color=GRID,

        linewidth=0.5,

        alpha=0.70

    )


    ax.set_axisbelow(
        True
    )


def add_zero(
    ax
):

    ax.axvline(

        0,

        color=GRAY,

        linestyle=":",

        linewidth=0.75,

        zorder=1

    )


def add_panel_letter(

    ax,
    letter

):

    ax.text(

        -0.105,

        1.025,

        letter,

        transform=ax.transAxes,

        fontsize=17,

        fontweight="bold",

        color=TEXT,

        ha="left",

        va="bottom"

    )


def add_title(

    ax,
    main,
    subtitle

):

    ax.text(

        0,

        1.045,

        main,

        transform=ax.transAxes,

        fontsize=10.8,

        color=TEXT,

        ha="left",

        va="bottom"

    )


    ax.text(

        0,

        1.005,

        subtitle,

        transform=ax.transAxes,

        fontsize=7.8,

        color=GRAY,

        ha="left",

        va="bottom"

    )


def ci_widths(

    effect,
    low,
    high

):

    left = effect - low

    right = high - effect


    if (

        not np.isfinite(left)
        or
        not np.isfinite(right)
        or
        left < -1e-12
        or
        right < -1e-12

    ):

        raise ValueError(

            "Invalid confidence interval: "
            f"effect={effect}, "
            f"low={low}, "
            f"high={high}"

        )


    return (

        max(left, 0.0),

        max(right, 0.0)

    )


def draw_point(

    ax,

    effect,

    low,

    high,

    y,

    window,

    decimals=3,

    label_offset=0.01

):

    left, right = ci_widths(

        effect,

        low,

        high

    )


    if window == "12m":

        color = COLOR_12M

        filled = True

    elif window == "24m":

        color = COLOR_24M

        filled = False

    else:

        raise ValueError(
            f"Unknown window: {window}"
        )


    facecolor = (

        color
        if filled
        else "white"

    )


    # -------------------------------------------------------------------------
    # IMPORTANT:
    # xerr is explicitly calculated from the point estimate, so CI bars are
    # centered exactly on the reported estimate.
    # -------------------------------------------------------------------------

    ax.errorbar(

        effect,

        y,

        xerr=np.array([

            [left],

            [right]

        ]),

        fmt="o",

        markersize=5.5,

        markerfacecolor=facecolor,

        markeredgecolor=color,

        markeredgewidth=1.0,

        color=color,

        ecolor=color,

        elinewidth=1.25,

        capsize=2.5,

        zorder=4

    )


    # -------------------------------------------------------------------------
    # Put label just beyond the CI.
    # -------------------------------------------------------------------------

    if effect >= 0:

        tx = (
            high
            +
            label_offset
        )

        ha = "left"

    else:

        tx = (
            low
            -
            label_offset
        )

        ha = "right"


    ax.text(

        tx,

        y,

        f"{effect:+.{decimals}f}",

        fontsize=7.1,

        color=color,

        ha=ha,

        va="center"

    )


def auto_xlim(

    ax,

    values,

    minimum_span

):

    values = np.asarray(

        values,

        dtype=float

    )


    values = values[
        np.isfinite(values)
    ]


    if len(values) == 0:

        return


    lo = values.min()

    hi = values.max()


    span = max(

        hi - lo,

        minimum_span

    )


    margin = max(

        span * 0.14,

        minimum_span * 0.035

    )


    left = min(

        lo - margin,

        -margin

    )


    right = max(

        hi + margin,

        margin

    )


    ax.set_xlim(

        left,

        right

    )


def add_window_legend(

    ax,
    loc="lower right"

):

    ax.legend(

        handles=[

            Line2D(

                [0],
                [0],

                marker="o",

                linestyle="None",

                color=COLOR_12M,

                markerfacecolor=COLOR_12M,

                markeredgecolor=COLOR_12M,

                markersize=5.4,

                label="12m"

            ),

            Line2D(

                [0],
                [0],

                marker="o",

                linestyle="None",

                color=COLOR_24M,

                markerfacecolor="white",

                markeredgecolor=COLOR_24M,

                markersize=5.4,

                label="24m"

            ),

        ],

        frameon=False,

        loc=loc

    )


# =============================================================================
# 9. FIGURE
# =============================================================================

fig, axs = plt.subplots(

    3,

    2,

    figsize=(15.2, 16.4)

)


# =============================================================================
# PANEL A
#
# BOUNDARY-SPANNING LOSS COMPOSITION
#
# IMPORTANT:
# Use FOUR decimal places because the effects are approximately -0.001 to
# -0.003 percentage points. Two decimals incorrectly displayed them as -0.00.
# =============================================================================

ax = axs[0, 0]

setup_axis(
    ax,
    "x"
)

add_zero(
    ax
)


cats_A = [

    "Cross-field",

    "Cross-subfield,\nsame field",

]


y_A = {

    "Cross-field":
        0.00,

    "Cross-subfield,\nsame field":
        0.72,

}


all_vals = []


for cat in cats_A:

    y0 = y_A[cat]


    # 12m ABOVE category center
    row = get_one(

        loss,

        (
            loss[
                "plot_category"
            ]
            ==
            cat
        )

        &

        loss[
            "analysis"
        ]
        .str.contains(
            "12m",
            regex=False
        ),

        f"Panel A — {cat} — 12m"

    )


    effect = float(
        row["effect_pp"]
    )

    low = float(
        row["ci_low_pp"]
    )

    high = float(
        row["ci_high_pp"]
    )


    draw_point(

        ax,

        effect,

        low,

        high,

        y0 + 0.085,

        "12m",

        decimals=4,

        label_offset=0.00055

    )


    all_vals.extend([
        effect,
        low,
        high
    ])


    # 24m BELOW category center
    row = get_one(

        loss,

        (
            loss[
                "plot_category"
            ]
            ==
            cat
        )

        &

        loss[
            "analysis"
        ]
        .str.contains(
            "24m",
            regex=False
        ),

        f"Panel A — {cat} — 24m"

    )


    effect = float(
        row["effect_pp"]
    )

    low = float(
        row["ci_low_pp"]
    )

    high = float(
        row["ci_high_pp"]
    )


    draw_point(

        ax,

        effect,

        low,

        high,

        y0 - 0.085,

        "24m",

        decimals=4,

        label_offset=0.00055

    )


    all_vals.extend([
        effect,
        low,
        high
    ])


ax.set_yticks(
    list(y_A.values())
)

ax.set_yticklabels([

    "Cross-field",

    "Cross-subfield,\nsame field",

])


ax.set_ylim(
    -0.20,
    0.92
)


ax.set_xlabel(
    "Association with AI intensity\n"
    "(percentage points)"
)


auto_xlim(

    ax,

    all_vals,

    minimum_span=0.012

)


add_panel_letter(
    ax,
    "a"
)


add_title(

    ax,

    "Cross-boundary knowledge is\n"
    "underrepresented among losses",

    "Focal-paper panel; loss composition; 12m and 24m",

)


add_window_legend(
    ax,
    loc="lower right"
)


# =============================================================================
# PANEL B
#
# DISTINCTIVE LOSS COMPOSITION
#
# Same high-precision labeling.
# Explicit legend ADDED.
# =============================================================================

ax = axs[0, 1]

setup_axis(
    ax,
    "x"
)

add_zero(
    ax
)


cats_B = [

    "Unique topic",

    "Low redundancy",

]


y_B = {

    "Unique topic":
        0.00,

    "Low redundancy":
        0.72,

}


all_vals = []


for cat in cats_B:

    y0 = y_B[cat]


    for window, dy in [

        ("12m", +0.085),

        ("24m", -0.085),

    ]:

        row = get_one(

            loss,

            (
                loss[
                    "plot_category"
                ]
                ==
                cat
            )

            &

            loss[
                "analysis"
            ]
            .str.contains(
                window,
                regex=False
            ),

            f"Panel B — {cat} — {window}"

        )


        effect = float(
            row["effect_pp"]
        )

        low = float(
            row["ci_low_pp"]
        )

        high = float(
            row["ci_high_pp"]
        )


        draw_point(

            ax,

            effect,

            low,

            high,

            y0 + dy,

            window,

            decimals=4,

            label_offset=0.00055

        )


        all_vals.extend([
            effect,
            low,
            high
        ])


ax.set_yticks(
    list(y_B.values())
)

ax.set_yticklabels([

    "Unique topic",

    "Low redundancy",

])


ax.set_ylim(
    -0.20,
    0.92
)


ax.set_xlabel(
    "Association with AI intensity\n"
    "(percentage points)"
)


auto_xlim(

    ax,

    all_vals,

    minimum_span=0.010

)


add_panel_letter(
    ax,
    "b"
)


add_title(

    ax,

    "Distinctive knowledge is\n"
    "underrepresented among losses",

    "Focal-paper panel; loss composition; 12m and 24m",

)


# -------------------------------------------------------------------------
# Explicit legend — this fixes the missing Panel B legend.
# -------------------------------------------------------------------------

add_window_legend(

    ax,

    loc="lower right"

)


# =============================================================================
# PANEL C
#
# ABSOLUTE RETENTION
#
# Six category estimates.
#
# 12m / 24m are vertically separated within every category.
# =============================================================================

ax = axs[1, 0]

setup_axis(
    ax,
    "x"
)

add_zero(
    ax
)


cats_C = [

    (
        "Cross-field",
        "Cross-field"
    ),

    (
        "Cross-subfield; same field",
        "Cross-subfield,\nsame field"
    ),

    (
        "Same subfield",
        "Same subfield"
    ),

    (
        "Unique topic",
        "Unique topic"
    ),

    (
        "Low redundancy (2-3)",
        "Low redundancy"
    ),

    (
        "High redundancy (4+)",
        "High redundancy"
    ),

]


# Compact grouping with slightly larger gap between conceptual families.
y_C = {

    "Cross-field":
        0.00,

    "Cross-subfield; same field":
        0.40,

    "Same subfield":
        0.80,

    "Unique topic":
        1.48,

    "Low redundancy (2-3)":
        1.88,

    "High redundancy (4+)":
        2.28,

}


all_vals = []


for cat, label in cats_C:

    y0 = y_C[cat]


    for window, dy in [

        ("12m", +0.065),

        ("24m", -0.065),

    ]:

        row = get_one(

            absolute,

            (
                absolute[
                    "category"
                ]
                ==
                cat
            )

            &

            absolute[
                "analysis"
            ]
            .str.contains(
                window,
                regex=False
            ),

            f"Panel C — {cat} — {window}"

        )


        effect = float(
            row["effect_pp"]
        )

        low = float(
            row["ci_low_pp"]
        )

        high = float(
            row["ci_high_pp"]
        )


        draw_point(

            ax,

            effect,

            low,

            high,

            y0 + dy,

            window,

            decimals=3,

            label_offset=0.004

        )


        all_vals.extend([
            effect,
            low,
            high
        ])


# Conceptual family labels.
ax.text(

    -0.35,

    0.78,

    "BOUNDARY",

    transform=ax.get_yaxis_transform(),

    fontsize=7.0,

    color=GRAY,

    ha="right",

    va="center"

)


ax.text(

    -0.35,

    1.74,

    "DISTINCTIVENESS",

    transform=ax.get_yaxis_transform(),

    fontsize=7.0,

    color=GRAY,

    ha="right",

    va="center"

)


ax.set_yticks([

    y_C[c]

    for c, _ in cats_C

])


ax.set_yticklabels([

    label

    for _, label in cats_C

])


ax.set_ylim(
    -0.16,
    2.48
)
ax.set_xlim(
    -0.16,
    0.12
)


ax.set_xlabel(
    "Association with AI intensity\n"
    "(percentage points)"
)





add_panel_letter(
    ax,
    "c"
)


add_title(

    ax,

    "Absolute retention shows the same\n"
    "selective-transmission pattern",

    "Focal-paper panel; boundary and distinctiveness categories",

)


add_window_legend(
    ax,
    loc="lower right"
)


# =============================================================================
# PANEL D
#
# CONTINUOUS RECOMBINATIONAL NOVELTY
#
# Six final estimates:
#
#     3y × 12m / 24m
#     5y × 12m / 24m
#     10y × 12m / 24m
#
# 5y comes from d_primary.
# 3y / 10y come from d_hist.
# =============================================================================

ax = axs[1, 1]

setup_axis(
    ax,
    "x"
)

add_zero(
    ax
)


history_windows = [

    "3y",
    "5y",
    "10y",

]


y_D = {

    "3y":
        0.00,

    "5y":
        0.60,

    "10y":
        1.20,

}


all_vals = []


for history in history_windows:

    y0 = y_D[history]


    for window, dy in [

        ("12m", +0.08),

        ("24m", -0.08),

    ]:

        if history == "5y":

            source = d_primary

            pattern = "5y atypicality"

        else:

            source = d_hist

            pattern = f"{history} atypicality"


        row = get_one(

            source,

            source[
                "analysis"
            ]
            .str.contains(
                pattern,
                regex=False
            )

            &

            source[
                "analysis"
            ]
            .str.contains(
                window,
                regex=False
            ),

            f"Panel D — {history} — {window}"

        )


        effect = float(
            row["beta_per_0p10_ai"]
        )

        low = float(
            row["ci_low"]
        )

        high = float(
            row["ci_high"]
        )


        draw_point(

            ax,

            effect,

            low,

            high,

            y0 + dy,

            window,

            decimals=3,

            label_offset=0.0025

        )


        all_vals.extend([
            effect,
            low,
            high
        ])


ax.set_yticks(
    list(y_D.values())
)

ax.set_yticklabels([

    "3-year",
    "5-year",
    "10-year",

])


ax.set_ylim(
    -0.16,
    1.36
)
ax.set_xlim(
    0,
    0.08
)

ax.set_xlabel(
    "Change in historical atypicality\n"
    "per 0.10 AI-score increase"
)





add_panel_letter(
    ax,
    "d"
)


add_title(

    ax,

    "AI intensity is associated with more\n"
    "atypical transmitted combinations",

    "Pairs jointly transmitted through F; robustness across pre-F historical windows",

)


add_window_legend(
    ax
)


# =============================================================================
# PANEL E
#
# NEVER-BEFORE-SEEN COMBINATIONS
# =============================================================================

ax = axs[2, 0]

setup_axis(
    ax,
    "x"
)

add_zero(
    ax
)


y_E = {

    "12m":
        0.15,

    "24m":
        0.85,

}


all_vals = []


for window in [

    "12m",
    "24m"

]:

    row = get_one(

        d_hist,

        (
            d_hist[
                "analysis"
            ]
            .str.contains(
                "never-seen 5y",
                regex=False
            )

            &

            d_hist[
                "analysis"
            ]
            .str.contains(
                window,
                regex=False
            )
        ),

        f"Panel E — {window}"

    )


    effect = (

        float(
            row[
                "beta_per_0p10_ai"
            ]
        )
        *
        100.0

    )


    low = (

        float(
            row[
                "ci_low"
            ]
        )
        *
        100.0

    )


    high = (

        float(
            row[
                "ci_high"
            ]
        )
        *
        100.0

    )


    draw_point(

        ax,

        effect,

        low,

        high,

        y_E[window],

        window,

        decimals=2,

        label_offset=0.09

    )


    all_vals.extend([
        effect,
        low,
        high
    ])


ax.set_yticks([

    y_E["12m"],

    y_E["24m"]

])

ax.set_yticklabels([

    "12m",

    "24m"

])


ax.set_ylim(
    -0.10,
    1.10
)

ax.set_xlim(
    0,
    2.5
)
ax.set_xlabel(
    "Change in never-seen share\n"
    "per 0.10 AI-score increase (percentage points)"
)





add_panel_letter(
    ax,
    "e"
)


add_title(

    ax,

    "More transmitted combinations are\n"
    "previously unseen",

    "Share of transmitted predecessor pairs absent during the preceding 5 years",

)


add_window_legend(
    ax
)


# =============================================================================
# PANEL F
#
# SPECIFICATION ROBUSTNESS
#
# Primary
# No controls
# Author × subfield FE
#
# 12m / 24m within each specification.
# =============================================================================

ax = axs[2, 1]

setup_axis(
    ax,
    "x"
)

add_zero(
    ax
)


specs = [

    (
        "Primary",
        d_primary
    ),

    (
        "No controls",
        d_nocontrol
    ),

    (
        "Author × subfield FE",
        d_strict
    ),

]


y_F = {

    "Primary":
        0.00,

    "No controls":
        0.68,

    "Author × subfield FE":
        1.36,

}


all_vals = []


for name, source in specs:

    y0 = y_F[name]


    for window, dy in [

        ("12m", +0.08),

        ("24m", -0.08),

    ]:

        row = get_one(

            source,

            source[
                "analysis"
            ]
            .str.contains(
                window,
                regex=False
            ),

            f"Panel F — {name} — {window}"

        )


        effect = float(
            row[
                "beta_per_0p10_ai"
            ]
        )

        low = float(
            row[
                "ci_low"
            ]
        )

        high = float(
            row[
                "ci_high"
            ]
        )


        draw_point(

            ax,

            effect,

            low,

            high,

            y0 + dy,

            window,

            decimals=3,

            label_offset=0.0025

        )


        all_vals.extend([
            effect,
            low,
            high
        ])


ax.set_yticks([

    y_F["Primary"],

    y_F["No controls"],

    y_F["Author × subfield FE"],

])


ax.set_yticklabels([

    "Primary",

    "No controls",

    "Author × subfield FE",

])


ax.set_ylim(
    -0.17,
    1.53
)
ax.set_xlim(
    0,
    0.08
)

ax.set_xlabel(
    "Change in historical atypicality\n"
    "per 0.10 AI-score increase"
)





add_panel_letter(
    ax,
    "f"
)


add_title(

    ax,

    "The recombinational-novelty association\n"
    "is stable across specifications",

    "Five-year pre-F historical baseline",

)


add_window_legend(
    ax
)


# =============================================================================
# 10. GLOBAL TITLE
# =============================================================================

fig.suptitle(

    "AI-associated writing is linked to selective loss and more novel "
    "downstream recombination of scientific knowledge",

    fontsize=15.0,

    y=0.988,

    color=TEXT

)


# =============================================================================
# 11. FIGURE-WIDE NOTE
# =============================================================================

note = (

    "Panels a–c summarize downstream "
    "loss composition and absolute retention at the focal-paper level. "
    "Panel d measures the historical atypicality of predecessor pairs "
    "jointly transmitted through focal paper F to future external citing "
    "papers C; \n the 5-year pre-F history is the primary measure, with "
    "3-year and 10-year histories shown as robustness. Panel e reports "
    "the corresponding share of transmitted predecessor pairs not observed "
    "during the preceding five years. \n Panel f compares the primary "
    "specification with models omitting focal-paper size controls and "
    "models with first-author × focal-subfield fixed effects. "
)


fig.text(

    0.02,

    0.018,

    note,

    ha="left",

    va="bottom",

    fontsize=6.7,

    color=GRAY,

    wrap=True

)


# =============================================================================
# 12. LAYOUT
# =============================================================================

fig.subplots_adjust(
    left=0.16, right=0.975, top=0.91, bottom=0.095,
    wspace=0.40, hspace=0.50
)


# =============================================================================
# 13. SAVE
# =============================================================================

outputs = [

    "fig5_final_rich.png",

    "fig5_final_rich.pdf",

    "fig5_final_rich.svg",

]


for filename in outputs:

    fig.savefig(

        filename,

        dpi=600,

        bbox_inches="tight"

    )


plt.show()


print(
    "\nSaved:"
)

for filename in outputs:

    print(
        f"  {filename}"
    )